# 35 — Controlled-300 Dashboard and Deployment Validation

**Presentation:** All eight principal rooms, the Focused Portrait Review, and the museum passport/free-exploration experience are user-approved. This is not deployment certification.
**Controlled-300 validation:** In progress. **N35 completion gate: No.**
**Input:** the immutable `dashboard_package.v2` N34 release.
**Outputs:** four canonical files, written only by the final consolidation batch.

The app presents existing evidence; it performs no restoration inference or scientific recomputation. The approved layouts and artwork remain frozen. These replacement cells reconcile validation with the completed application, not rebuild its rooms.

The ten-batch plan remains: (1) contract/preflight, (2) Foyer/shared shell, (3) Study Design, (4) Metric Framework, (5) Model Gallery, (6) Stability Lab, (7) Trustworthiness/D02, (8) Case Explorer, (9) Research Archive, (10) integration/deployment/readiness/persistence.

## Execution boundary

Replace the first 12 cells with this pack, preserving their Markdown/code types. Restart the kernel and run in order, stopping at the end of Batch 2. Do not use Run All: later cells/results still include historical pilot logic and will be reconciled separately. Historical “N35 complete” output is not a current result. Preserve the old notebook in Git history and clear stale saved outputs in your working notebook before this rerun.

Batch 1 checks the existing project inventory's freshness. After saving your replacement cells, refresh the inventory from the project root with `.\.venv\Scripts\python.exe tools/build_project_inventory.py --root .` before running Batch 1. This is a separate user-run inventory operation, not scientific recomputation. Do not bypass inventory errors or change failed checks to pass.

Installed-version mismatches remain explicit warnings at this local gate. Missing packages or inconsistent requirements/config pins remain blocking. Deployment alignment, browser behavior, resource budgets, and remote asset availability are not certified by this pack.

Canonical outputs: `validation/dashboard_checks.csv`, `reports/deployment_readiness.md`, `manifests/run_manifest.json`, and `manifests/artifacts.csv`, all under `outputs/35_dashboard_and_deployment_validation/`.


## Batch 1 — Controlled-300 contract and runtime trust-boundary preflight

Validate the exact N34 release, package checksums, population, room order, schemas, indexed paths, governance records, source syntax, and focused helper tests. Network access is blocked during the package-read audit. This is a local preflight, not a clean deployment simulation.

The approved population is 300 paintings, 3,425 registered cases, 2,620 four-model-eligible cases, 10,480 primary four-model candidates, 13,879 indexed candidates, 24 bounded-SDXL candidates, 1,025 repeated-seed groups, and 337 indexed reports. Values are checked against the versioned contract, not rewritten.

The default deterministic LaMa candidate must never borrow another candidate's uncertainty evidence. Room-routing prose is not a numerical table. Existing typed opening records are checked here; all-selection deployed numerical parity remains a later integration gate.

These cells may create empty N35 output directories, but must not create canonical N35 files. Stale files block the run; nothing is deleted automatically. Rerunning a validation cell removes its old checks and downstream Batch 1/2 checks so an old success or failure cannot survive unnoticed. Restart from the beginning after any source/config change.


In [1]:
from __future__ import annotations

import ast
import json
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
from unittest import mock

import pandas as pd
from IPython.display import display


CURRENT_LOCATION = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (
            CURRENT_LOCATION,
            *CURRENT_LOCATION.parents,
        )
        if (
            candidate
            / "config"
            / "evaluation"
            / "dashboard_validation.yaml"
        ).is_file()
        and (
            candidate
            / "outputs"
            / "34_final_streamlit_dashboard_assets"
            / "manifests"
            / "dashboard_runtime_manifest.json"
        ).is_file()
        and (candidate / "src" / "restoration_eval").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the painting-restoration-eval repository root."
    )

SOURCE_ROOT = PROJECT_ROOT / "src"

if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))


from restoration_eval.dashboard_application import (
    ALL_ROOM_IDS,
    CHILD_ROOM_ID,
    DASHBOARD_APPLICATION_VERSION,
    DASHBOARD_PACKAGE_SCHEMA_VERSION,
    DASHBOARD_RUNTIME_MANIFEST_SCHEMA_VERSION,
    DASHBOARD_VALIDATION_CONFIG_SCHEMA_VERSION,
    PRINCIPAL_ROOM_IDS,
    audit_dashboard_package,
    audit_indexed_paths,
    configuration_checksum,
    load_dashboard_validation_config,
    open_dashboard_package,
    required_input_paths,
    truthy,
    validate_input_path_templates,
)
from restoration_eval.paths import (
    PATHS_MODULE_VERSION,
    PROJECT_PATHS_SCHEMA_VERSION,
    ensure_output_subdirs,
    notebook_output_root,
    validate_project_paths_registry,
)
from restoration_eval.validation import (
    VALIDATION_MODULE_VERSION,
    ValidationCollector,
)


NOTEBOOK_ID = "35"
NOTEBOOK_STEM = "35_dashboard_and_deployment_validation"

RUN_STARTED_AT_UTC = (
    datetime.now(timezone.utc)
    .isoformat(timespec="seconds")
    .replace("+00:00", "Z")
)

VALIDATION = ValidationCollector()

print("Notebook 35 Controlled-300 environment initialized.")
print("Project root:", PROJECT_ROOT)
print("Run started:", RUN_STARTED_AT_UTC)
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print(
    "Helper versions:",
    {
        "dashboard_application": DASHBOARD_APPLICATION_VERSION,
        "validation": VALIDATION_MODULE_VERSION,
        "paths": PATHS_MODULE_VERSION,
    },
)

# Only validation state is reset; no scientific or application file is changed.
import hashlib
import re

def drop_validation_stages(prefixes):
    global VALIDATION
    refreshed = ValidationCollector()
    refreshed.extend(c for c in VALIDATION.checks
                     if not c.validation_stage.startswith(tuple(prefixes)))
    VALIDATION = refreshed


Notebook 35 Controlled-300 environment initialized.
Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Run started: 2026-10-01T06:50:36Z
Python: 3.12.6
Platform: Windows-11-10.0.26200-SP0
Helper versions: {'dashboard_application': '2.1.0', 'validation': '1.0.0', 'paths': '1.1.1'}


In [2]:
drop_validation_stages(('batch_1_', 'batch_2_'))

CONFIG = load_dashboard_validation_config(
    PROJECT_ROOT
)

APPLICATION = CONFIG["application"]
DATASET = CONFIG["dataset"]
N34_RELEASE = CONFIG["n34_release"]
EXPECTED_POPULATION = CONFIG["expected_population"]
EXPECTED_PACKAGE = CONFIG["expected_package"]
RUNTIME = CONFIG["runtime"]
BOUNDARIES = CONFIG["scientific_boundaries"]
OUTPUT_SPEC = CONFIG["outputs"]

REQUIRED_INPUT_PATHS = required_input_paths(
    CONFIG,
    PROJECT_ROOT,
)

INPUT_PATH_TEMPLATES = (
    validate_input_path_templates(
        CONFIG,
        PROJECT_ROOT,
    )
)

OUTPUT_ROOT = notebook_output_root(
    NOTEBOOK_STEM,
    PROJECT_ROOT,
)

PREEXISTING_OUTPUT_FILES = (
    sorted(
        path.relative_to(
            OUTPUT_ROOT
        ).as_posix()
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    )
    if OUTPUT_ROOT.exists()
    else []
)

OUTPUT_PATHS = {
    key: PROJECT_ROOT / contract["path"]
    for key, contract in OUTPUT_SPEC.items()
}

EXPECTED_OUTPUT_PATHS = {
    "dashboard_checks": (
        "outputs/35_dashboard_and_deployment_validation/"
        "validation/dashboard_checks.csv"
    ),
    "deployment_readiness": (
        "outputs/35_dashboard_and_deployment_validation/"
        "reports/deployment_readiness.md"
    ),
    "run_manifest": (
        "outputs/35_dashboard_and_deployment_validation/"
        "manifests/run_manifest.json"
    ),
    "artifacts": (
        "outputs/35_dashboard_and_deployment_validation/"
        "manifests/artifacts.csv"
    ),
}

OBSERVED_OUTPUT_PATHS = {
    key: path.relative_to(
        PROJECT_ROOT
    ).as_posix()
    for key, path in OUTPUT_PATHS.items()
}

OBSERVED_CHILD_ROUTES = [
    (
        route["room_id"],
        route["parent_room_id"],
    )
    for route in APPLICATION[
        "child_routes"
    ]
]

IMPLEMENTATION_BATCH_IDS = [
    int(record["batch"])
    for record in CONFIG[
        "implementation_batches"
    ]
]

CONTRACT_CHECKS = (
    (
        "configuration_schema",
        "Notebook 35 uses the Controlled-300 validation schema",
        DASHBOARD_VALIDATION_CONFIG_SCHEMA_VERSION,
        CONFIG["schema_version"],
        (
            CONFIG["schema_version"]
            == DASHBOARD_VALIDATION_CONFIG_SCHEMA_VERSION
        ),
    ),
    (
        "notebook_identity",
        "Notebook identity is fixed",
        {
            "id": NOTEBOOK_ID,
            "stem": NOTEBOOK_STEM,
        },
        {
            "id": CONFIG["notebook"]["id"],
            "stem": CONFIG["notebook"]["stem"],
        },
        (
            CONFIG["notebook"]["id"]
            == NOTEBOOK_ID
            and CONFIG["notebook"]["stem"]
            == NOTEBOOK_STEM
        ),
    ),
    (
        "dataset_scope",
        "The application targets the Controlled-300 dataset",
        "controlled_300",
        DATASET["dataset_scope"],
        DATASET["dataset_scope"]
        == "controlled_300",
    ),
    (
        "package_schema",
        "The application accepts only dashboard_package.v2",
        DASHBOARD_PACKAGE_SCHEMA_VERSION,
        APPLICATION[
            "package_schema_version"
        ],
        (
            APPLICATION[
                "package_schema_version"
            ]
            == DASHBOARD_PACKAGE_SCHEMA_VERSION
        ),
    ),
    (
        "runtime_manifest_schema",
        "The N34 runtime manifest schema is fixed",
        DASHBOARD_RUNTIME_MANIFEST_SCHEMA_VERSION,
        N34_RELEASE[
            "runtime_manifest_schema_version"
        ],
        (
            N34_RELEASE[
                "runtime_manifest_schema_version"
            ]
            == DASHBOARD_RUNTIME_MANIFEST_SCHEMA_VERSION
        ),
    ),
    (
        "principal_room_order",
        "The approved eight-room order is unchanged",
        list(PRINCIPAL_ROOM_IDS),
        APPLICATION[
            "principal_room_order"
        ],
        (
            list(PRINCIPAL_ROOM_IDS)
            == APPLICATION[
                "principal_room_order"
            ]
        ),
    ),
    (
        "d02_child_route",
        "D02 remains a child of Trustworthiness",
        [
            (
                CHILD_ROOM_ID,
                "trustworthiness",
            )
        ],
        OBSERVED_CHILD_ROUTES,
        (
            OBSERVED_CHILD_ROUTES
            == [
                (
                    CHILD_ROOM_ID,
                    "trustworthiness",
                )
            ]
        ),
    ),
    (
        "implementation_batches",
        "The room-by-room implementation plan has ten batches",
        list(range(1, 11)),
        IMPLEMENTATION_BATCH_IDS,
        (
            IMPLEMENTATION_BATCH_IDS
            == list(range(1, 11))
        ),
    ),
    (
        "required_input_contracts",
        (
            "All required input contracts, resolved locations, "
            "and the painting-shard template are declared"
        ),
        {
            "declared_contracts": 30,
            "resolved_locations": 30,
            "path_templates": 1,
        },
        {
            "declared_contracts": len(
                CONFIG["required_inputs"]
            ),
            "resolved_locations": len(
                REQUIRED_INPUT_PATHS
            ),
            "path_templates": len(
                INPUT_PATH_TEMPLATES
            ),
        },
        (
            len(
                CONFIG["required_inputs"]
            )
            == 30
            and len(
                REQUIRED_INPUT_PATHS
            )
            == 30
            and len(
                INPUT_PATH_TEMPLATES
            )
            == 1
        ),
    ),
    (
        "canonical_output_count",
        "Exactly four canonical N35 outputs are declared",
        4,
        len(OUTPUT_PATHS),
        len(OUTPUT_PATHS) == 4,
    ),
    (
        "canonical_output_paths",
        "Canonical output paths match the approved contract",
        EXPECTED_OUTPUT_PATHS,
        OBSERVED_OUTPUT_PATHS,
        (
            OBSERVED_OUTPUT_PATHS
            == EXPECTED_OUTPUT_PATHS
        ),
    ),
    (
        "clean_output_root",
        "No stale Controlled-50 N35 files remain",
        [],
        PREEXISTING_OUTPUT_FILES,
        not PREEXISTING_OUTPUT_FILES,
    ),
    (
        "zero_network_startup",
        "Application startup makes no external request",
        0,
        int(
            N34_RELEASE[
                "startup_network_requests"
            ]
        ),
        (
            int(
                N34_RELEASE[
                    "startup_network_requests"
                ]
            )
            == 0
        ),
    ),
    (
        "no_scientific_recomputation",
        "The application cannot recompute scientific metrics",
        True,
        bool(
            BOUNDARIES[
                "no_scientific_recomputation"
            ]
        ),
        bool(
            BOUNDARIES[
                "no_scientific_recomputation"
            ]
        ),
    ),
    (
        "no_restoration_inference",
        "The application cannot run restoration inference",
        True,
        bool(
            BOUNDARIES[
                "no_restoration_inference"
            ]
        ),
        bool(
            BOUNDARIES[
                "no_restoration_inference"
            ]
        ),
    ),
)

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in CONTRACT_CHECKS:
    VALIDATION.add(
        validation_stage=(
            "batch_1_contract"
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else (
                "The Controlled-300 N35 "
                "contract is incomplete."
            )
        ),
    )

VALIDATION.raise_for_blocking()

OUTPUT_DIRS = ensure_output_subdirs(
    NOTEBOOK_STEM,
    (
        "validation",
        "reports",
        "manifests",
    ),
    PROJECT_ROOT,
)

ROOM_CONTRACT_VIEW = pd.DataFrame(
    [
        {
            "display_order": index,
            "room_id": room_id,
            "display_name": (
                APPLICATION[
                    "display_names"
                ][room_id]
            ),
            "route_kind": "principal",
        }
        for index, room_id in enumerate(
            PRINCIPAL_ROOM_IDS,
            start=1,
        )
    ]
    + [
        {
            "display_order": "",
            "room_id": CHILD_ROOM_ID,
            "display_name": (
                APPLICATION[
                    "display_names"
                ][CHILD_ROOM_ID]
            ),
            "route_kind": (
                "Trustworthiness child"
            ),
        }
    ]
)

BATCH_PLAN_VIEW = pd.DataFrame(
    CONFIG["implementation_batches"]
)

display(ROOM_CONTRACT_VIEW)
display(BATCH_PLAN_VIEW)

print("Controlled-300 N35 contract passed.")
print(
    "Concrete input paths:",
    len(REQUIRED_INPUT_PATHS),
)
print(
    "Input templates:",
    INPUT_PATH_TEMPLATES,
)
print(
    "Pre-existing output files:",
    len(PREEXISTING_OUTPUT_FILES),
)
print(
    "Canonical files persisted:",
    0,
)

,display_order,room_id,display_name,route_kind
0,1,exhibition_foyer,Exhibition Foyer,principal
1,2,study_design,Study Design,principal
2,3,metric_framework,Metric Framework,principal
3,4,model_gallery,Model Gallery,principal
4,5,stability_lab,Stability Lab,principal
5,6,trustworthiness,Trustworthiness,principal
6,7,case_explorer,Case Explorer,principal
7,8,research_archive,Research Archive,principal
8,,focused_portrait_review,Focused Portrait Review,Trustworthiness child


,batch,scope,visual_checkpoint
0,1,Controlled-300 contract and v2 runtime API pre...,none_contract_gate
1,2,Shared museum shell and Exhibition Foyer,exhibition_foyer
2,3,Study Design room,study_design
3,4,Metric Framework room,metric_framework
4,5,Model Gallery room,model_gallery
5,6,Stability Lab room,stability_lab
6,7,Trustworthiness room and D02 child route,trustworthiness_and_focused_portrait_review
7,8,Case Explorer room,case_explorer
8,9,Research Archive room,research_archive
9,10,"Integration, remote/cache/failure tests, deep ...",all_routes_three_viewports_and_final_user_appr...


Controlled-300 N35 contract passed.
Concrete input paths: 30
Input templates: {'painting_partitions': 'outputs/34_final_streamlit_dashboard_assets/data/paintings/{painting_id}.json.gz'}
Pre-existing output files: 0
Canonical files persisted: 0


In [3]:
drop_validation_stages(('batch_1_package', 'batch_1_indexed', 'batch_1_runtime', 'batch_1_governance', 'batch_1_dependency', 'batch_1_completion', 'batch_2_'))

NETWORK_FAILURE = AssertionError(
    "Batch 1 attempted network access."
)

with (
    mock.patch(
        "urllib.request.urlopen",
        side_effect=NETWORK_FAILURE,
    ),
    mock.patch(
        "urllib.request.urlretrieve",
        side_effect=NETWORK_FAILURE,
    ),
    mock.patch(
        "socket.create_connection",
        side_effect=NETWORK_FAILURE,
    ),
):
    PACKAGE = open_dashboard_package(
        PROJECT_ROOT
    )

    FOYER_PARTITION = PACKAGE.load_room(
        "exhibition_foyer"
    )

    P001_PARTITION = PACKAGE.load_painting(
        "p001"
    )

    REPORT_CATALOGUE = (
        PACKAGE.load_reports()
    )

    PACKAGE_AUDIT = (
        audit_dashboard_package(
            PROJECT_ROOT,
            verify_all_files=True,
        )
    )

    INDEXED_PATH_AUDIT = (
        audit_indexed_paths(
            PACKAGE,
            include_all_visuals=True,
        )
    )


for audit_name, audit_frame in (
    (
        "package",
        PACKAGE_AUDIT,
    ),
    (
        "indexed_paths",
        INDEXED_PATH_AUDIT,
    ),
):
    for record in audit_frame.to_dict(
        orient="records"
    ):
        VALIDATION.add(
            validation_stage=(
                "batch_1_"
                f"{record['validation_stage']}"
            ),
            check_id=(
                f"{audit_name}__"
                f"{record['check_id']}"
            ),
            check_description=(
                record[
                    "check_description"
                ]
            ),
            severity=record["severity"],
            expected=record["expected"],
            observed=record["observed"],
            passed=bool(
                record["passed"]
            ),
            details=record["details"],
        )


EXPECTED_RUNTIME_POPULATION = {
    "painting_count": (
        EXPECTED_POPULATION[
            "paintings"
        ]
    ),
    "registered_case_count": (
        EXPECTED_POPULATION[
            "registered_cases"
        ]
    ),
    "four_model_eligible_case_count": (
        EXPECTED_POPULATION[
            "four_model_eligible_cases"
        ]
    ),
    "four_model_primary_candidate_count": (
        EXPECTED_POPULATION[
            "four_model_primary_candidates"
        ]
    ),
    "indexed_inspectable_candidate_count": (
        EXPECTED_POPULATION[
            "indexed_inspectable_candidates"
        ]
    ),
    "repeated_seed_group_count": (
        EXPECTED_POPULATION[
            "repeated_seed_groups"
        ]
    ),
}

OBSERVED_RUNTIME_POPULATION = {
    key: PACKAGE.population[key]
    for key in EXPECTED_RUNTIME_POPULATION
}

PACKAGE_SIZE_BYTES = sum(
    path.stat().st_size
    for path in PACKAGE.package_root.rglob("*")
    if path.is_file()
)

UPSTREAM_BLOCKING_FAILURES = int(
    (
        ~PACKAGE.upstream_checks[
            "passed"
        ].map(truthy)
        & PACKAGE.upstream_checks[
            "severity"
        ]
        .astype(str)
        .str.casefold()
        .isin(
            {
                "blocking",
                "error",
            }
        )
    ).sum()
)

PACKAGE_CHECKS = (
    (
        "release_id",
        "The exact N34 release is loaded",
        N34_RELEASE["release_id"],
        PACKAGE.release_id,
        (
            PACKAGE.release_id
            == N34_RELEASE["release_id"]
        ),
    ),
    (
        "producer_run_id",
        "The exact completed N34 run is loaded",
        N34_RELEASE[
            "producer_run_id"
        ],
        PACKAGE.runtime_manifest[
            "run_id"
        ],
        (
            PACKAGE.runtime_manifest[
                "run_id"
            ]
            == N34_RELEASE[
                "producer_run_id"
            ]
        ),
    ),
    (
        "package_hash",
        "The deterministic package identity matches",
        N34_RELEASE[
            "deterministic_package_sha256"
        ],
        PACKAGE.runtime_manifest[
            "deterministic_package_sha256"
        ],
        (
            PACKAGE.runtime_manifest[
                "deterministic_package_sha256"
            ]
            == N34_RELEASE[
                "deterministic_package_sha256"
            ]
        ),
    ),
    (
        "runtime_population",
        "The main Controlled-300 populations match",
        EXPECTED_RUNTIME_POPULATION,
        OBSERVED_RUNTIME_POPULATION,
        (
            EXPECTED_RUNTIME_POPULATION
            == OBSERVED_RUNTIME_POPULATION
        ),
    ),
    (
        "runtime_local_artifacts",
        "All runtime-declared local artifacts are indexed",
        EXPECTED_PACKAGE[
            "local_artifacts"
        ],
        len(
            PACKAGE.artifact_index
        ),
        (
            len(
                PACKAGE.artifact_index
            )
            == EXPECTED_PACKAGE[
                "local_artifacts"
            ]
        ),
    ),
    (
        "physical_package_files",
        "The promoted package has the exact file count",
        EXPECTED_PACKAGE[
            "package_files"
        ],
        sum(
            1
            for path
            in PACKAGE.package_root.rglob(
                "*"
            )
            if path.is_file()
        ),
        (
            sum(
                1
                for path
                in PACKAGE.package_root.rglob(
                    "*"
                )
                if path.is_file()
            )
            == EXPECTED_PACKAGE[
                "package_files"
            ]
        ),
    ),
    (
        "physical_package_bytes",
        "The promoted package has the exact byte size",
        EXPECTED_PACKAGE[
            "package_size_bytes"
        ],
        PACKAGE_SIZE_BYTES,
        (
            PACKAGE_SIZE_BYTES
            == EXPECTED_PACKAGE[
                "package_size_bytes"
            ]
        ),
    ),
    (
        "upstream_validation_rows",
        "All N34 validation rows are present",
        N34_RELEASE[
            "upstream_validation_checks"
        ],
        len(
            PACKAGE.upstream_checks
        ),
        (
            len(
                PACKAGE.upstream_checks
            )
            == N34_RELEASE[
                "upstream_validation_checks"
            ]
        ),
    ),
    (
        "upstream_blocking_failures",
        "N34 has no blocking validation failure",
        0,
        UPSTREAM_BLOCKING_FAILURES,
        (
            UPSTREAM_BLOCKING_FAILURES
            == 0
        ),
    ),
    (
        "foyer_partition",
        "The Exhibition Foyer partition loads lazily",
        {
            "room_id": (
                "exhibition_foyer"
            ),
            "rows": (
                EXPECTED_PACKAGE[
                    "room_rows"
                ][
                    "exhibition_foyer"
                ]
            ),
        },
        {
            "room_ids": sorted(
                FOYER_PARTITION[
                    "room_id"
                ]
                .astype(str)
                .unique()
                .tolist()
            ),
            "rows": len(
                FOYER_PARTITION
            ),
        },
        (
            len(
                FOYER_PARTITION
            )
            == EXPECTED_PACKAGE[
                "room_rows"
            ][
                "exhibition_foyer"
            ]
            and set(
                FOYER_PARTITION[
                    "room_id"
                ].astype(str)
            )
            == {
                "exhibition_foyer"
            }
        ),
    ),
    (
        "painting_partition",
        "A single painting shard preserves its identity",
        "p001",
        P001_PARTITION[
            "painting_id"
        ],
        (
            P001_PARTITION[
                "painting_id"
            ]
            == "p001"
        ),
    ),
    (
        "report_catalogue",
        "The compact report catalogue is complete",
        EXPECTED_POPULATION[
            "report_index_records"
        ],
        REPORT_CATALOGUE[
            "report_count"
        ],
        (
            REPORT_CATALOGUE[
                "report_count"
            ]
            == EXPECTED_POPULATION[
                "report_index_records"
            ]
        ),
    ),
)

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in PACKAGE_CHECKS:
    VALIDATION.add(
        validation_stage=(
            "batch_1_runtime_package"
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else (
                "The immutable N34 "
                "package contract drifted."
            )
        ),
    )

VALIDATION.raise_for_blocking()

PACKAGE_FACTS_VIEW = pd.DataFrame(
    [
        {
            "fact": "release_id",
            "value": PACKAGE.release_id,
        },
        {
            "fact": "paintings",
            "value": len(
                PACKAGE.painting_lookup
            ),
        },
        {
            "fact": "room_partitions",
            "value": len(
                PACKAGE.room_partitions
            ),
        },
        {
            "fact": "display_ids",
            "value": (
                PACKAGE.display_components[
                    "display_id"
                ].nunique()
            ),
        },
        {
            "fact": "display_bindings",
            "value": len(
                PACKAGE.display_components
            ),
        },
        {
            "fact": "logical_assets",
            "value": len(
                PACKAGE.dashboard_assets
            ),
        },
        {
            "fact": "renditions",
            "value": len(
                PACKAGE.renditions
            ),
        },
        {
            "fact": "locators",
            "value": len(
                PACKAGE.asset_locators
            ),
        },
        {
            "fact": "reports",
            "value": (
                REPORT_CATALOGUE[
                    "report_count"
                ]
            ),
        },
        {
            "fact": "physical_bytes",
            "value": PACKAGE_SIZE_BYTES,
        },
    ]
)

display(PACKAGE_FACTS_VIEW)
display(PACKAGE_AUDIT)
display(INDEXED_PATH_AUDIT)

print("Controlled-300 package preflight passed.")
print(
    "Runtime-declared artifacts verified:",
    len(PACKAGE.artifact_index),
)
print(
    "Package audit:",
    f"{int(PACKAGE_AUDIT['passed'].sum())}/"
    f"{len(PACKAGE_AUDIT)} passed",
)
print(
    "Indexed-path audit:",
    f"{int(INDEXED_PATH_AUDIT['passed'].sum())}/"
    f"{len(INDEXED_PATH_AUDIT)} passed",
)
print("Network requests made: 0")

,fact,value
0,release_id,release_c5edf0d0ec99affbfa403726
1,paintings,300
2,room_partitions,9
3,display_ids,100
4,display_bindings,170
5,logical_assets,56
6,renditions,48
7,locators,95
8,reports,337
9,physical_bytes,7978024


,validation_stage,check_id,check_description,severity,expected,observed,passed,details
0,package_contract,runtime_schema,Controlled-300 package satisfies runtime schema,blocking,dashboard_runtime_manifest.v1,dashboard_runtime_manifest.v1,True,
1,package_contract,package_schema,Controlled-300 package satisfies package schema,blocking,dashboard_package.v2,dashboard_package.v2,True,
2,package_contract,principal_rooms,Controlled-300 package satisfies principal rooms,blocking,"[""exhibition_foyer"", ""study_design"", ""metric_f...","[""exhibition_foyer"", ""study_design"", ""metric_f...",True,
3,package_contract,child_rooms,Controlled-300 package satisfies child rooms,blocking,"[""focused_portrait_review""]","[""focused_portrait_review""]",True,
4,package_contract,paintings,Controlled-300 package satisfies paintings,blocking,300,300,True,
5,package_contract,room_partitions,Controlled-300 package satisfies room partitions,blocking,9,9,True,
6,package_contract,display_ids,Controlled-300 package satisfies display ids,blocking,100,100,True,
7,package_contract,display_bindings,Controlled-300 package satisfies display bindings,blocking,170,170,True,
8,package_contract,logical_assets,Controlled-300 package satisfies logical assets,blocking,56,56,True,
9,package_contract,renditions,Controlled-300 package satisfies renditions,blocking,48,48,True,


,validation_stage,check_id,check_description,severity,expected,observed,passed,details
0,indexed_paths,registered_renditions,Every registered local rendition exists and ma...,blocking,0,0,True,[]
1,indexed_paths,local_exact_assets,Registered local exact assets remain repositor...,blocking,0,0,True,[]


Controlled-300 package preflight passed.
Runtime-declared artifacts verified: 373
Package audit: 17/17 passed
Indexed-path audit: 2/2 passed
Network requests made: 0


In [5]:
drop_validation_stages(('batch_1_governance', 'batch_1_dependency', 'batch_1_completion', 'batch_2_'))

INVENTORY_RUN_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "inventory"
    / "inventory_run.json"
)

PROJECT_PATHS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "inventory"
    / "project_paths.json"
)

with INVENTORY_RUN_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    INVENTORY_RUN = json.load(
        handle
    )

with PROJECT_PATHS_PATH.open(
    "r",
    encoding="utf-8-sig",
) as handle:
    PROJECT_PATHS_REGISTRY = (
        json.load(handle)
    )

PROJECT_PATH_ERRORS = (
    validate_project_paths_registry(
        PROJECT_PATHS_REGISTRY
    )
)

PREPARATION_PATHS = {
    "config": (
        PROJECT_ROOT
        / "config"
        / "evaluation"
        / "dashboard_validation.yaml"
    ),
    "coverage": (
        PROJECT_ROOT
        / "config"
        / "evaluation"
        / "evidence_coverage.yaml"
    ),
    "roadmap": (
        PROJECT_ROOT
        / "docs"
        / "final_notebook_roadmap.md"
    ),
    "guidelines": (
        PROJECT_ROOT
        / "docs"
        / "refactoring_implementation_guidelines.md"
    ),
    "audit": (
        PROJECT_ROOT
        / "docs"
        / "evidence_dependency_audit.md"
    ),
    "helper": (
        PROJECT_ROOT
        / "src"
        / "restoration_eval"
        / "dashboard_application.py"
    ),
    "application": (
        PROJECT_ROOT
        / APPLICATION["entrypoint"]
    ),
    "focused_tests": (
        PROJECT_ROOT
        / "tests"
        / "test_dashboard_application.py"
    ),
}

PREPARATION_PATHS.update({
    "room_runtime": PROJECT_ROOT / "src/restoration_eval/room_runtime.py",
    "visitor_renderer": PROJECT_ROOT / "src/restoration_eval/museum_visit.py",
    "visitor_controller": PROJECT_ROOT / "streamlit_assets/museum_visit/controller.js",
    "visitor_styles": PROJECT_ROOT / "streamlit_assets/museum_visit/visit.css",
    "visitor_route": PROJECT_ROOT / "streamlit_assets/museum_visit/tour.json",
    "visitor_freeze": PROJECT_ROOT / "config/publication/museum_visit_freeze.json",
    "archive_freeze": PROJECT_ROOT / "config/publication/research_archive_freeze.json",
})

MISSING_PREPARATION_PATHS = sorted(
    name
    for name, path
    in PREPARATION_PATHS.items()
    if not path.is_file()
)

INVENTORY_COMPLETED_AT = (
    datetime.fromisoformat(
        str(
            INVENTORY_RUN[
                "completed_at_utc"
            ]
        ).replace(
            "Z",
            "+00:00",
        )
    )
)

LATEST_PREPARATION_MTIME = max(
    datetime.fromtimestamp(
        path.stat().st_mtime,
        tz=timezone.utc,
    )
    for path
    in PREPARATION_PATHS.values()
)

INVENTORY_IS_FRESH = (
    INVENTORY_COMPLETED_AT
    >= LATEST_PREPARATION_MTIME
)

ROADMAP_TEXT = (
    PREPARATION_PATHS[
        "roadmap"
    ]
    .read_text(
        encoding="utf-8-sig"
    )
    .casefold()
)

GUIDELINES_TEXT = (
    PREPARATION_PATHS[
        "guidelines"
    ]
    .read_text(
        encoding="utf-8-sig"
    )
    .casefold()
)

AUDIT_TEXT = (
    PREPARATION_PATHS[
        "audit"
    ]
    .read_text(
        encoding="utf-8-sig"
    )
    .casefold()
)

COVERAGE_TEXT = (
    PREPARATION_PATHS[
        "coverage"
    ]
    .read_text(
        encoding="utf-8-sig"
    )
    .casefold()
)

HELPER_SOURCE = (
    PREPARATION_PATHS[
        "helper"
    ].read_text(
        encoding="utf-8"
    )
)

APPLICATION_SOURCE = (
    PREPARATION_PATHS[
        "application"
    ].read_text(
        encoding="utf-8"
    )
)

SOURCE_SYNTAX_ERRORS = {}

for source_name, source_text, source_path in (
    (
        "dashboard_application",
        HELPER_SOURCE,
        PREPARATION_PATHS[
            "helper"
        ],
    ),
    (
        "streamlit_app",
        APPLICATION_SOURCE,
        PREPARATION_PATHS[
            "application"
        ],
    ),
):
    try:
        ast.parse(
            source_text,
            filename=str(
                source_path
            ),
        )
    except SyntaxError as exc:
        SOURCE_SYNTAX_ERRORS[
            source_name
        ] = (
            f"{exc.msg} at "
            f"{exc.lineno}:"
            f"{exc.offset}"
        )


TEST_ENV = os.environ.copy()

TEST_ENV["PYTHONPATH"] = (
    str(SOURCE_ROOT)
    + (
        os.pathsep
        + TEST_ENV["PYTHONPATH"]
        if TEST_ENV.get(
            "PYTHONPATH"
        )
        else ""
    )
)

FOCUSED_TEST_RESULT = (
    subprocess.run(
        [
            sys.executable,
            "-m",
            "unittest",
            "discover",
            "-s",
            "tests",
            "-p",
            (
                "test_dashboard_"
                "application.py"
            ),
            "-v",
        ],
        cwd=PROJECT_ROOT,
        env=TEST_ENV,
        capture_output=True,
        text=True,
        timeout=120,
        check=False,
    )
)

FOCUSED_TEST_TRANSCRIPT = (
    (
        FOCUSED_TEST_RESULT.stdout
        + "\n"
        + FOCUSED_TEST_RESULT.stderr
    )
    .strip()
)

REQUIREMENTS_PATH = (
    PROJECT_ROOT
    / "requirements.txt"
)

REQUIREMENT_LINES = [
    line.strip()
    for line in REQUIREMENTS_PATH.read_text(
        encoding="utf-8-sig"
    ).splitlines()
    if (
        line.strip()
        and not line.lstrip().startswith(
            "#"
        )
    )
]

REQUIREMENT_PINS = {
    line.split(
        "==",
        maxsplit=1,
    )[0]
    .strip()
    .casefold(): line.split(
        "==",
        maxsplit=1,
    )[1]
    .strip()
    for line in REQUIREMENT_LINES
    if "==" in line
}

DISTRIBUTION_NAMES = {
    "streamlit": "streamlit",
    "pandas": "pandas",
    "plotly": "plotly",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
    "pillow": "Pillow",
    "pyyaml": "PyYAML",
}

TARGET_VERSIONS = {
    str(package).casefold(): str(
        version
    )
    for package, version
    in RUNTIME[
        "packages"
    ].items()
}

INSTALLED_VERSIONS = {}

for package, distribution in (
    DISTRIBUTION_NAMES.items()
):
    try:
        INSTALLED_VERSIONS[
            package
        ] = metadata.version(
            distribution
        )
    except metadata.PackageNotFoundError:
        INSTALLED_VERSIONS[
            package
        ] = "not_installed"

MISSING_PACKAGES = sorted(
    package
    for package, version
    in INSTALLED_VERSIONS.items()
    if version == "not_installed"
)

MISSING_OR_DIFFERENT_PINS = sorted(
    package
    for package, target
    in TARGET_VERSIONS.items()
    if REQUIREMENT_PINS.get(
        package
    )
    != target
)

VERSION_DIFFERENCES = [
    {
        "package": package,
        "target_version": (
            TARGET_VERSIONS[
                package
            ]
        ),
        "installed_version": (
            INSTALLED_VERSIONS[
                package
            ]
        ),
    }
    for package
    in sorted(TARGET_VERSIONS)
    if (
        TARGET_VERSIONS[
            package
        ]
        != INSTALLED_VERSIONS[
            package
        ]
    )
]

GOVERNANCE_CHECKS = (
    (
        "inventory_status",
        "The refreshed inventory completed",
        "completed",
        INVENTORY_RUN.get(
            "status"
        ),
        (
            INVENTORY_RUN.get(
                "status"
            )
            == "completed"
        ),
    ),
    (
        "inventory_read_errors",
        "The inventory contains no read error",
        0,
        int(
            INVENTORY_RUN[
                "summary"
            ][
                "read_error_count"
            ]
        ),
        (
            int(
                INVENTORY_RUN[
                    "summary"
                ][
                    "read_error_count"
                ]
            )
            == 0
        ),
    ),
    (
        "inventory_repository",
        "The inventory belongs to this repository",
        str(PROJECT_ROOT),
        INVENTORY_RUN.get(
            "repository_root"
        ),
        (
            Path(
                INVENTORY_RUN[
                    "repository_root"
                ]
            ).resolve()
            == PROJECT_ROOT
        ),
    ),
    (
        "inventory_freshness",
        "The inventory is newer than the preparation files",
        True,
        INVENTORY_IS_FRESH,
        INVENTORY_IS_FRESH,
    ),
    (
        "preparation_files",
        "Every preparation file exists",
        [],
        MISSING_PREPARATION_PATHS,
        not MISSING_PREPARATION_PATHS,
    ),
    (
        "project_paths_registry",
        "The project-path registry remains valid",
        {
            "schema": (
                PROJECT_PATHS_SCHEMA_VERSION
            ),
            "errors": [],
        },
        {
            "schema": (
                PROJECT_PATHS_REGISTRY.get(
                    "registry_schema_version"
                )
            ),
            "errors": (
                PROJECT_PATH_ERRORS
            ),
        },
        (
            PROJECT_PATHS_REGISTRY.get(
                "registry_schema_version"
            )
            == PROJECT_PATHS_SCHEMA_VERSION
            and not PROJECT_PATH_ERRORS
        ),
    ),
    (
        "roadmap_preparation_status",
        "The roadmap records the Controlled-300 N35 preparation",
        True,
        (
            "controlled-300 implementation status"
            in ROADMAP_TEXT
            and "ten-batch delivery"
            in ROADMAP_TEXT
        ),
        (
            "controlled-300 implementation status"
            in ROADMAP_TEXT
            and "ten-batch delivery"
            in ROADMAP_TEXT
        ),
    ),
    (
        "manual_notebook_policy",
        "The notebook remains user-pasted and user-executed",
        True,
        (
            "complete notebook markdown and code replacements remain"
            in GUIDELINES_TEXT
        ),
        (
            "complete notebook markdown and code replacements remain"
            in GUIDELINES_TEXT
        ),
    ),
    (
        "room_approval_policy",
        "The room-by-room visual approval policy is recorded",
        True,
        (
            "room-by-room visual approval workflow"
            in GUIDELINES_TEXT
        ),
        (
            "room-by-room visual approval workflow"
            in GUIDELINES_TEXT
        ),
    ),
    (
        "audit_transition_record",
        "The evidence audit records the N35 transition",
        True,
        (
            "controlled-300 notebook 35 preparation"
            in AUDIT_TEXT
        ),
        (
            "controlled-300 notebook 35 preparation"
            in AUDIT_TEXT
        ),
    ),
    (
        "coverage_target_record",
        "The machine-readable coverage registry records the N35 target",
        True,
        (
            "controlled_300_target:"
            in COVERAGE_TEXT
            and (
                "dashboard_package.v2"
                in COVERAGE_TEXT
            )
        ),
        (
            "controlled_300_target:"
            in COVERAGE_TEXT
            and (
                "dashboard_package.v2"
                in COVERAGE_TEXT
            )
        ),
    ),
    (
        "source_syntax",
        "The helper and application entrypoint parse as Python",
        {},
        SOURCE_SYNTAX_ERRORS,
        not SOURCE_SYNTAX_ERRORS,
    ),
    (
        "focused_tests",
        "The focused Controlled-300 helper tests pass",
        0,
        FOCUSED_TEST_RESULT.returncode,
        (
            FOCUSED_TEST_RESULT.returncode
            == 0
        ),
    ),
    (
        "runtime_packages_available",
        "All required dashboard packages are installed",
        [],
        MISSING_PACKAGES,
        not MISSING_PACKAGES,
    ),
    (
        "requirements_pins",
        "requirements.txt contains the deployment target pins",
        [],
        MISSING_OR_DIFFERENT_PINS,
        not MISSING_OR_DIFFERENT_PINS,
    ),
)

for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in GOVERNANCE_CHECKS:
    VALIDATION.add(
        validation_stage=(
            "batch_1_governance"
        ),
        check_id=check_id,
        check_description=description,
        severity="blocking",
        expected=expected,
        observed=observed,
        passed=passed,
        details=(
            ""
            if passed
            else (
                "Preparation governance "
                "or focused testing failed."
            )
        ),
    )


for record in VERSION_DIFFERENCES:
    VALIDATION.add(
        validation_stage=(
            "batch_1_dependency_versions"
        ),
        check_id=(
            "version__"
            f"{record['package']}"
        ),
        check_description=(
            "Installed package version "
            "matches the deployment pin"
        ),
        severity="warning",
        expected=record[
            "target_version"
        ],
        observed=record[
            "installed_version"
        ],
        passed=False,
        details=(
            "Development may continue, "
            "but the deployment environment "
            "must be reconciled before the "
            "final N35 readiness claim."
        ),
    )


display(pd.DataFrame([{"inventory_completed_utc": INVENTORY_COMPLETED_AT.isoformat(),
    "latest_preparation_mtime_utc": LATEST_PREPARATION_MTIME.isoformat(),
    "fresh": INVENTORY_IS_FRESH}]))
if not INVENTORY_IS_FRESH:
    print("Inventory is stale. Save notebook edits, run tools/build_project_inventory.py --root . "
          "from the project root, then rerun this cell. Do not bypass this blocking check.")
if FOCUSED_TEST_RESULT.returncode:
    print(FOCUSED_TEST_TRANSCRIPT)
VALIDATION.raise_for_blocking()

DEPENDENCY_VIEW = pd.DataFrame(
    [
        {
            "package": package,
            "target_version": (
                TARGET_VERSIONS[
                    package
                ]
            ),
            "installed_version": (
                INSTALLED_VERSIONS[
                    package
                ]
            ),
            "exact_match": (
                TARGET_VERSIONS[
                    package
                ]
                == INSTALLED_VERSIONS[
                    package
                ]
            ),
        }
        for package
        in sorted(TARGET_VERSIONS)
    ]
)

display(DEPENDENCY_VIEW)

print("Focused test transcript:")
print(FOCUSED_TEST_TRANSCRIPT)
print()
print(
    "Inventory run ID:",
    INVENTORY_RUN[
        "inventory_run_id"
    ],
)
print(
    "Indexed files:",
    INVENTORY_RUN[
        "summary"
    ][
        "file_count"
    ],
)
print(
    "Configuration SHA-256:",
    configuration_checksum(
        PROJECT_ROOT
    ),
)
print(
    "Dependency-version warnings:",
    len(VERSION_DIFFERENCES),
)

,inventory_completed_utc,latest_preparation_mtime_utc,fresh
0,2026-10-01T06:54:36.043299+00:00,2026-10-01T06:52:57.230591+00:00,True


,package,target_version,installed_version,exact_match
0,numpy,1.26.4,1.26.4,True
1,pandas,2.3.3,2.3.3,True
2,pillow,12.3.0,9.5.0,False
3,plotly,6.9.0,6.8.0,False
4,pyarrow,20.0.0,24.0.0,False
5,pyyaml,6.0.3,6.0.3,True
6,streamlit,1.56.0,1.59.0,False


Focused test transcript:
test_asset_rendition_and_locator_apis_preserve_identity (test_dashboard_application.DashboardApplicationV2Tests.test_asset_rendition_and_locator_apis_preserve_identity) ... ok
test_case_explorer_default_is_exact_and_cross_experiment_safe (test_dashboard_application.DashboardApplicationV2Tests.test_case_explorer_default_is_exact_and_cross_experiment_safe) ... ok
test_curated_case_metrics_resolve_only_for_exact_p018_identity (test_dashboard_application.DashboardApplicationV2Tests.test_curated_case_metrics_resolve_only_for_exact_p018_identity) ... ok
test_curated_threshold_is_exact_and_mismatch_never_substitutes (test_dashboard_application.DashboardApplicationV2Tests.test_curated_threshold_is_exact_and_mismatch_never_substitutes) ... ok
test_every_room_and_d02_partition_loads_exact_identity (test_dashboard_application.DashboardApplicationV2Tests.test_every_room_and_d02_partition_loads_exact_identity) ... ok
test_exact_rooms_child_route_and_population (test_dashboa

In [6]:
drop_validation_stages(('batch_1_completion', 'batch_2_'))

REQUIRED_BATCH_1_STAGES = {
    "batch_1_contract", "batch_1_package_contract", "batch_1_package_integrity",
    "batch_1_indexed_paths", "batch_1_runtime_package", "batch_1_governance",
}
MISSING_BATCH_1_STAGES = REQUIRED_BATCH_1_STAGES - {
    c.validation_stage for c in VALIDATION.checks
}
if MISSING_BATCH_1_STAGES:
    raise RuntimeError("Run every Batch 1 cell in order. Missing stages: "
                       + str(sorted(MISSING_BATCH_1_STAGES)))

CANONICAL_FILES_AFTER_BATCH = (
    sorted(
        path.relative_to(
            OUTPUT_ROOT
        ).as_posix()
        for path in OUTPUT_ROOT.rglob(
            "*"
        )
        if path.is_file()
    )
)

VALIDATION.add(
    validation_stage=(
        "batch_1_completion"
    ),
    check_id=(
        "no_canonical_files_written"
    ),
    check_description=(
        "Batch 1 writes no canonical "
        "Notebook 35 artifact"
    ),
    severity="blocking",
    expected=[],
    observed=(
        CANONICAL_FILES_AFTER_BATCH
    ),
    passed=(
        not CANONICAL_FILES_AFTER_BATCH
    ),
    details=(
        ""
        if not CANONICAL_FILES_AFTER_BATCH
        else (
            "Batch 1 is a read-only "
            "preflight and must not persist "
            "canonical outputs."
        )
    ),
)

VALIDATION.raise_for_blocking()

VALIDATION_FRAME = (
    VALIDATION.to_dataframe()
)

VALIDATION_STAGE_SUMMARY = (
    VALIDATION_FRAME
    .groupby(
        [
            "validation_stage",
            "severity",
        ],
        dropna=False,
    )
    .agg(
        checks=("check_id", "size"),
        passed=("passed", "sum"),
    )
    .reset_index()
)

VALIDATION_STAGE_SUMMARY[
    "failed"
] = (
    VALIDATION_STAGE_SUMMARY[
        "checks"
    ]
    - VALIDATION_STAGE_SUMMARY[
        "passed"
    ]
)

WARNING_RESULTS = (
    VALIDATION_FRAME.loc[
        (
            ~VALIDATION_FRAME[
                "passed"
            ]
        )
        & (
            VALIDATION_FRAME[
                "severity"
            ]
            == "warning"
        ),
        [
            "check_id",
            "expected",
            "observed",
            "details",
        ],
    ]
    .reset_index(drop=True)
)

display(
    VALIDATION_STAGE_SUMMARY
)

if not WARNING_RESULTS.empty:
    display(WARNING_RESULTS)

print("Batch 1 preflight passed.")
print(
    "Validation checks:",
    len(VALIDATION_FRAME),
)
print(
    "Blocking failures:",
    len(
        VALIDATION.blocking_failures
    ),
)
print(
    "Dependency-version warnings:",
    len(WARNING_RESULTS),
)
print(
    "N34 package audit:",
    f"{int(PACKAGE_AUDIT['passed'].sum())}/"
    f"{len(PACKAGE_AUDIT)}",
)
print(
    "Runtime-declared artifacts:",
    len(PACKAGE.artifact_index),
)
print(
    "Paintings:",
    len(PACKAGE.painting_lookup),
)
print(
    "Principal rooms:",
    len(PRINCIPAL_ROOM_IDS),
)
print(
    "Child routes:",
    len(ALL_ROOM_IDS)
    - len(PRINCIPAL_ROOM_IDS),
)
print(
    "Canonical N35 files persisted:",
    len(
        CANONICAL_FILES_AFTER_BATCH
    ),
)
print(
    "Next validation gate (presentation already approved):",
    (
        "Batch 2 — shared museum shell "
        "and Exhibition Foyer"
    ),
)

,validation_stage,severity,checks,passed,failed
0,batch_1_completion,blocking,1,1,0
1,batch_1_contract,blocking,15,15,0
2,batch_1_dependency_versions,warning,4,0,4
3,batch_1_governance,blocking,15,15,0
4,batch_1_indexed_paths,blocking,2,2,0
5,batch_1_package_contract,blocking,16,16,0
6,batch_1_package_integrity,blocking,1,1,0
7,batch_1_runtime_package,blocking,12,12,0


,check_id,expected,observed,details
0,version__pillow,12.3.0,9.5.0,"Development may continue, but the deployment e..."
1,version__plotly,6.9.0,6.8.0,"Development may continue, but the deployment e..."
2,version__pyarrow,20.0.0,24.0.0,"Development may continue, but the deployment e..."
3,version__streamlit,1.56.0,1.59.0,"Development may continue, but the deployment e..."


Batch 1 preflight passed.
Validation checks: 66
Blocking failures: 0
Dependency-version warnings: 4
N34 package audit: 17/17
Runtime-declared artifacts: 373
Paintings: 300
Principal rooms: 8
Child routes: 1
Canonical N35 files persisted: 0
Next validation gate (presentation already approved): Batch 2 — shared museum shell and Exhibition Foyer


## Batch 2 — Validate the approved shared shell and Exhibition Foyer

The Foyer is already visually approved. This batch checks its current implementation without modifying the application, CSS, artwork, or frozen evidence.

Checks cover the eight-room navigation and seven previews, verified p001 hero, collection counts, pinned decorative shell, keyboard-access declarations, responsive CSS structure, and the real eight-stop/six-minute passport plus free-exploration entry points. Freeze manifests verify the approved app integration and Archive bytes.

The old exact-1050px assertion is replaced by structural checks for the existing Foyer reflow and compact navigation rules. This establishes that responsive rules exist; it does NOT prove viewport rendering, clipping, keyboard usability, Firefox reliability, or zoom/second-monitor behavior. Those browser checks remain pending for Batch 10.

The visitor guide is now a JavaScript overlay injected through a zero-height component. Streamlit AppTest can check its delivered payload, but cannot execute its JavaScript or prove its dialogs work. We therefore check the real transport rather than looking for the removed server-rendered tour card. Same-room navigation uses a dedicated component; unexpected native buttons remain a failure, not an indiscriminate exception.

Server-side Foyer smoke runs below do not start a web server or touch your manually running Streamlit process. They make no permitted network request and write no canonical notebook output. They do not establish cold-cloud startup latency, memory limits, or deployment readiness.


In [7]:
drop_validation_stages(('batch_2_',))

from restoration_eval.dashboard_application import (
    audit_streamlit_source,
)


APP_PATH = PROJECT_ROOT / "streamlit_app.py"
APP_SOURCE = APP_PATH.read_text(
    encoding="utf-8"
)
APP_TREE = ast.parse(
    APP_SOURCE,
    filename=str(APP_PATH),
)


def literal_assignment(variable_name):
    for node in APP_TREE.body:
        if not isinstance(node, ast.Assign):
            continue

        target_names = {
            target.id
            for target in node.targets
            if isinstance(target, ast.Name)
        }

        if variable_name in target_names:
            return ast.literal_eval(node.value)

    raise KeyError(
        "Literal application assignment "
        f"not found: {variable_name}"
    )


def replace_validation_stage(
    validation_stage,
    records,
):
    """Replace one validation stage so the cell is safe to rerun."""

    global VALIDATION

    retained_checks = [
        check
        for check in VALIDATION.checks
        if (
            check.validation_stage
            != validation_stage
        )
    ]

    refreshed_validation = ValidationCollector()
    refreshed_validation.extend(
        retained_checks
    )

    for record in records:
        refreshed_validation.add(
            validation_stage=validation_stage,
            **record,
        )

    VALIDATION = refreshed_validation


def application_source_check(
    check_id,
    description,
    expected,
    observed,
    passed,
):
    return {
        "check_id": check_id,
        "check_description": description,
        "severity": "blocking",
        "expected": expected,
        "observed": observed,
        "passed": bool(passed),
        "details": (
            ""
            if passed
            else (
                "The Batch 2 application "
                "source contract differs."
            )
        ),
    }


APP_BUILD = literal_assignment(
    "APP_BUILD"
)

APP_ROOM_LABELS = literal_assignment(
    "ROOM_LABELS"
)

APP_ROUTE_PREVIEW_ASSETS = (
    literal_assignment(
        "ROUTE_PREVIEW_ASSETS"
    )
)

EXPECTED_ROOM_LABELS = dict(
    APPLICATION["display_names"]
)

EXPECTED_PREVIEW_DESTINATIONS = set(
    PRINCIPAL_ROOM_IDS[1:]
)

SOURCE_AUDIT = audit_streamlit_source(
    PROJECT_ROOT
)


# ---------------------------------------------------------------------------
# Exact visual-shell requirements
# ---------------------------------------------------------------------------

LOCAL_PACKAGE_API_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        "open_dashboard_package(",
        "package.load_room(",
    )
)

ENTRY_MODES_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        "Take the guided tour",
        "Explore freely",
    )
)

# Inspect actual Foyer reflow rules; do not demand an obsolete exact breakpoint.
# This is source evidence only. Browser viewport rendering is a Batch 10 gate.
def css_media_blocks(source):
    blocks = []
    for match in re.finditer(r"@media\s*\(\s*max-width\s*:\s*(\d+)px\s*\)\s*\{", source):
        depth, end = 1, match.end()
        while end < len(source) and depth:
            depth += (source[end] == "{") - (source[end] == "}")
            end += 1
        if depth:
            raise ValueError("Unbalanced responsive CSS rule")
        blocks.append((int(match.group(1)), source[match.end():end - 1]))
    return blocks

RESPONSIVE_BLOCKS = css_media_blocks(APP_SOURCE)
FOYER_REFLOW_BLOCKS = [width for width, body in RESPONSIVE_BLOCKS
    if 640 < width <= 1100 and all(s in body for s in
        (".foyer-stage", ".foyer-actions", "position: relative", "height: auto"))]
COMPACT_NAV_BLOCKS = [width for width, body in RESPONSIVE_BLOCKS
    if width <= 640 and ".museum-nav" in body]
RESPONSIVE_LAYOUT_PRESENT = bool(FOYER_REFLOW_BLOCKS and COMPACT_NAV_BLOCKS)
print("Declared Foyer reflow widths:", FOYER_REFLOW_BLOCKS,
      "compact navigation widths:", COMPACT_NAV_BLOCKS)


KEYBOARD_PREVIEW_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        'tabindex="0"',
        "focus-visible",
        "focus-within",
    )
)

MOCKUP_RUNTIME_DEPENDENCY = any(
    fragment in APP_SOURCE
    for fragment in (
        "approved_current/",
        "dashboard_design_finalists/",
    )
)

PINNED_DECORATIVE_SHELL_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        "FOYER_SHELL_SHA256",
        (
            "file_data_uri("
            "str(FOYER_SHELL_PATH), "
            "FOYER_SHELL_SHA256"
            ")"
        ),
        'class="foyer-shell"',
        'aria-hidden="true"',
    )
)

LIVE_HERO_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        "bound_asset_id(",
        "local_asset_uri(",
        "clip-path: polygon(",
        "object-fit: cover",
        "hero-live",
    )
)

ZIGZAG_ROUTE_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        '<polyline points="62,8 174,22',
        "route-stop:nth-child(3)",
        "route-stop:nth-child(9)",
    )
)

HOVER_ONLY_ROUTE_PREVIEWS = (
    "visibility: hidden"
    in APP_SOURCE
    and (
        ".route-stop:hover "
        ".route-preview"
        in APP_SOURCE
    )
    and (
        ".route-stop:focus-within "
        ".route-preview"
        in APP_SOURCE
    )
    and "route-stop.default"
    not in APP_SOURCE
    and "default_class"
    not in APP_SOURCE
)

WALL_PERSPECTIVE_PRESENT = all(
    fragment in APP_SOURCE
    for fragment in (
        "perspective(24cqw)",
        "rotateZ(-1.55deg)",
        "perspective(20cqw)",
        "rotateZ(-1.85deg)",
        "perspective(18cqw)",
        "rotateZ(-2.05deg)",
        (
            "Art<br>meets<br>evidence"
            "<br>across<br>time"
        ),
        "To<br>Study Design",
    )
)

CURRENT_MARKER_ALIGNED = (
    "left: calc(50% - .725cqw)"
    in APP_SOURCE
)


# ---------------------------------------------------------------------------
# Shared application and Exhibition Foyer checks
# ---------------------------------------------------------------------------

APPLICATION_SOURCE_CHECKS = (
    (
        "application_build",
        (
            "The application exposes the final "
            "approved Exhibition Foyer build marker"
        ),
        "n35.batch2.foyer.v4",
        APP_BUILD,
        (
            APP_BUILD
            == "n35.batch2.foyer.v4"
        ),
    ),
    (
        "room_labels",
        (
            "The application declares the approved "
            "eight rooms and D02 child label"
        ),
        EXPECTED_ROOM_LABELS,
        APP_ROOM_LABELS,
        (
            APP_ROOM_LABELS
            == EXPECTED_ROOM_LABELS
        ),
    ),
    (
        "preview_destinations",
        (
            "The Foyer declares one exact preview "
            "for every later principal room"
        ),
        sorted(
            EXPECTED_PREVIEW_DESTINATIONS
        ),
        sorted(
            APP_ROUTE_PREVIEW_ASSETS
        ),
        (
            set(
                APP_ROUTE_PREVIEW_ASSETS
            )
            == EXPECTED_PREVIEW_DESTINATIONS
        ),
    ),
    (
        "local_package_api",
        (
            "The application opens the immutable "
            "Notebook 34 package through its helper API"
        ),
        True,
        LOCAL_PACKAGE_API_PRESENT,
        LOCAL_PACKAGE_API_PRESENT,
    ),
    (
        "foyer_renderer",
        (
            "The application has an explicit "
            "Exhibition Foyer renderer"
        ),
        True,
        (
            "def render_exhibition_foyer("
            in APP_SOURCE
        ),
        (
            "def render_exhibition_foyer("
            in APP_SOURCE
        ),
    ),
    (
        "guided_and_free_entry",
        (
            "Both approved Foyer entry modes "
            "are implemented"
        ),
        True,
        ENTRY_MODES_PRESENT,
        ENTRY_MODES_PRESENT,
    ),
    (
        "responsive_rules_declared",
        (
            "Desktop, compact and mobile "
            "layout rules are declared"
        ),
        True,
        RESPONSIVE_LAYOUT_PRESENT,
        RESPONSIVE_LAYOUT_PRESENT,
    ),
    (
        "keyboard_preview_access",
        (
            "Room previews support keyboard "
            "focus as well as pointer hover"
        ),
        True,
        KEYBOARD_PREVIEW_PRESENT,
        KEYBOARD_PREVIEW_PRESENT,
    ),
    (
        "no_mockup_runtime_dependency",
        (
            "Approved reference images are not "
            "loaded directly at application runtime"
        ),
        False,
        MOCKUP_RUNTIME_DEPENDENCY,
        not MOCKUP_RUNTIME_DEPENDENCY,
    ),
    (
        "pinned_decorative_shell",
        (
            "The local decorative museum shell "
            "is checksum-pinned and non-scientific"
        ),
        True,
        PINNED_DECORATIVE_SHELL_PRESENT,
        PINNED_DECORATIVE_SHELL_PRESENT,
    ),
    (
        "live_hero_evidence",
        (
            "The dominant framed painting is a "
            "live registered asset fitted to the frame"
        ),
        True,
        LIVE_HERO_PRESENT,
        LIVE_HERO_PRESENT,
    ),
    (
        "zigzag_museum_route",
        (
            "The museum map follows the approved "
            "staggered zigzag route"
        ),
        True,
        ZIGZAG_ROUTE_PRESENT,
        ZIGZAG_ROUTE_PRESENT,
    ),
    (
        "hover_only_route_previews",
        (
            "Room cards remain closed at rest and "
            "open only through hover or keyboard focus"
        ),
        True,
        HOVER_ONLY_ROUTE_PREVIEWS,
        HOVER_ONLY_ROUTE_PREVIEWS,
    ),
    (
        "wall_perspective_and_current_marker",
        (
            "Wall inscriptions follow their wall "
            "planes and the current marker identifies "
            "the Exhibition Foyer"
        ),
        {
            "wall_perspective": True,
            "current_marker_aligned": True,
        },
        {
            "wall_perspective": (
                WALL_PERSPECTIVE_PRESENT
            ),
            "current_marker_aligned": (
                CURRENT_MARKER_ALIGNED
            ),
        },
        (
            WALL_PERSPECTIVE_PRESENT
            and CURRENT_MARKER_ALIGNED
        ),
    ),
)


SOURCE_STAGE_RECORDS = []

for record in SOURCE_AUDIT.to_dict(
    orient="records"
):
    SOURCE_STAGE_RECORDS.append(
        {
            "check_id": (
                "source__"
                + str(record["check_id"])
            ),
            "check_description": (
                record[
                    "check_description"
                ]
            ),
            "severity": (
                record["severity"]
            ),
            "expected": (
                record["expected"]
            ),
            "observed": (
                record["observed"]
            ),
            "passed": bool(
                record["passed"]
            ),
            "details": (
                record["details"]
            ),
        }
    )


for (
    check_id,
    description,
    expected,
    observed,
    passed,
) in APPLICATION_SOURCE_CHECKS:
    SOURCE_STAGE_RECORDS.append(
        application_source_check(
            check_id,
            description,
            expected,
            observed,
            passed,
        )
    )



# Reconcile the completed visitor overlay without modifying frozen source files.
VISITOR_ROOT = PROJECT_ROOT / "streamlit_assets/museum_visit"
VISITOR_DATA = json.loads((VISITOR_ROOT / "tour.json").read_text(encoding="utf-8"))
VISITOR_SCRIPT = (VISITOR_ROOT / "controller.js").read_text(encoding="utf-8")
VISITOR_CSS = (VISITOR_ROOT / "visit.css").read_text(encoding="utf-8")
VISITOR_ROOMS = [s["room"] for s in VISITOR_DATA["stops"]]
FREEZE_AUDIT_RECORDS = []
for manifest_name in ("research_archive_freeze.json", "museum_visit_freeze.json"):
    manifest = json.loads((PROJECT_ROOT / "config/publication" / manifest_name).read_text(encoding="utf-8"))
    if not manifest["files"] or manifest["status"] != "user_approved_visual_freeze":
        raise ValueError("Missing user-approved freeze records: " + manifest_name)
    for item in manifest["files"]:
        path = (PROJECT_ROOT / item["path"]).resolve()
        safe = path.is_relative_to(PROJECT_ROOT)
        raw = path.read_bytes() if safe and path.is_file() else None
        if raw is not None and item["hash_mode"] == "lf_normalized":
            raw = raw.replace(b"\r\n", b"\n")
        observed = hashlib.sha256(raw).hexdigest() if raw is not None else "missing_or_unsafe"
        FREEZE_AUDIT_RECORDS.append({"manifest": manifest_name, "path": item["path"],
            "expected": item["sha256"], "observed": observed, "passed": observed == item["sha256"]})
FREEZE_AUDIT = pd.DataFrame(FREEZE_AUDIT_RECORDS)

VISITOR_SOURCE_CHECKS = (
    ("approved_freeze_bytes", "All approved Archive and visitor release files retain their hashes",
     0, int((~FREEZE_AUDIT["passed"]).sum()), bool(FREEZE_AUDIT["passed"].all())),
    ("visitor_room_order", "The actual visitor asset follows all eight principal rooms",
     list(PRINCIPAL_ROOM_IDS), VISITOR_ROOMS, VISITOR_ROOMS == list(PRINCIPAL_ROOM_IDS)),
    ("visitor_duration", "The suggested visit totals six minutes",
     360, sum(s["seconds"] for s in VISITOR_DATA["stops"]),
     sum(s["seconds"] for s in VISITOR_DATA["stops"]) == 360),
    ("visitor_detour", "Portrait review is optional and outside the eight stamps",
     CHILD_ROOM_ID, VISITOR_DATA["detour"]["room"],
     VISITOR_DATA["detour"]["room"] == CHILD_ROOM_ID and CHILD_ROOM_ID not in VISITOR_ROOMS),
    ("visitor_invitation_routes", "Four free-exploration invitations use known rooms",
     True, len(VISITOR_DATA["invitations"]) == 4 and all(i["room"] in VISITOR_ROOMS for i in VISITOR_DATA["invitations"]),
     len(VISITOR_DATA["invitations"]) == 4 and all(i["room"] in VISITOR_ROOMS for i in VISITOR_DATA["invitations"])),
    ("visitor_overlay_wiring", "The approved dialog hooks and renderer remain connected",
     True, all(s in APP_SOURCE for s in ('data-museum-visit="tour"', 'data-museum-visit="free"', 'render_museum_visit(room_id,')),
     all(s in APP_SOURCE for s in ('data-museum-visit="tour"', 'data-museum-visit="free"', 'render_museum_visit(room_id,'))),
    ("visitor_source_boundaries", "No automatic timer or persistent localStorage is declared",
     False, bool(re.search(r"\b(?:setTimeout|setInterval|localStorage)\b", VISITOR_SCRIPT)),
     not bool(re.search(r"\b(?:setTimeout|setInterval|localStorage)\b", VISITOR_SCRIPT))),
)
SOURCE_STAGE_RECORDS.extend(application_source_check(*r) for r in VISITOR_SOURCE_CHECKS)
display(FREEZE_AUDIT.loc[~FREEZE_AUDIT["passed"]] if not FREEZE_AUDIT["passed"].all()
        else FREEZE_AUDIT.groupby("manifest").agg(files=("path", "size"), passed=("passed", "sum")))

replace_validation_stage(
    "batch_2_application_source",
    SOURCE_STAGE_RECORDS,
)

VALIDATION.raise_for_blocking()

display(SOURCE_AUDIT)

print("Batch 2 application source passed.")
print("Application build:", APP_BUILD)
print(
    "Principal navigation rooms:",
    len(PRINCIPAL_ROOM_IDS),
)
print(
    "Foyer destination previews:",
    len(APP_ROUTE_PREVIEW_ASSETS),
)
print(
    "Application source checks:",
    len(SOURCE_STAGE_RECORDS),
)

Declared Foyer reflow widths: [780] compact navigation widths: [640]


,files,passed
manifest,,
museum_visit_freeze.json,11,11
research_archive_freeze.json,52,52


,validation_stage,check_id,check_description,severity,expected,observed,passed,details
0,application_static,python_syntax,Streamlit entrypoint parses as Python,blocking,valid,valid,True,
1,application_static,prohibited__outputs_dashboard,Application excludes prohibited source fragmen...,blocking,False,False,True,
2,application_static,prohibited__outputs_metrics,Application excludes prohibited source fragmen...,blocking,False,False,True,
3,application_static,prohibited__outputs_reports,Application excludes prohibited source fragmen...,blocking,False,False,True,
4,application_static,prohibited__outputs_figures,Application excludes prohibited source fragmen...,blocking,False,False,True,
5,application_static,prohibited__combined_uncertainty_index,Application excludes prohibited source fragmen...,blocking,False,False,True,
6,application_static,prohibited__controlled_50,Application excludes prohibited source fragmen...,blocking,False,False,True,
7,application_boundaries,no_external_font_or_css_cdn,Application styling is self-contained and star...,blocking,[],[],True,
8,application_boundaries,read_only_application,Application source contains no filesystem/data...,blocking,[],[],True,


Batch 2 application source passed.
Application build: n35.batch2.foyer.v4
Principal navigation rooms: 8
Foyer destination previews: 7
Application source checks: 30


In [8]:
drop_validation_stages(('batch_2_foyer_contract', 'batch_2_streamlit_smoke', 'batch_2_final_gate'))

FOYER_ROOM = PACKAGE.load_room(
    "exhibition_foyer"
)

EXPECTED_FOYER_DISPLAY_IDS = {
    "foyer.hero.p001",
    "foyer.scope.paintings",
    "foyer.scope.categories",
    "foyer.scope.cases",
    "foyer.scope.methods",
    "foyer.route.previews",
    "foyer.tour.frames",
}

OBSERVED_FOYER_DISPLAY_IDS = set(
    FOYER_ROOM[
        "display_id"
    ].astype(str)
)

FOYER_OPENING = PACKAGE.bootstrap[
    "opening"
]

EXPECTED_FOYER_OPENING = {
    "painting_id": "p001",
    "question": (
        "How should we judge an "
        "AI-restored painting?"
    ),
    "boundary": (
        "Visual plausibility is not the "
        "same as restoration trustworthiness."
    ),
}

SCOPE_INDICATORS = {
    str(record["indicator_id"]): int(
        record["value"]
    )
    for record in PACKAGE.bootstrap[
        "scope_indicators"
    ]
}

EXPECTED_SCOPE_INDICATORS = {
    "paintings": 300,
    "visual_categories": 5,
    "registered_cases": 3425,
    "methods": 4,
}

GUIDED_TOUR = PACKAGE.bootstrap[
    "guided_tour"
]

HERO_BINDINGS = PACKAGE.bindings_for(
    display_id="foyer.hero.p001",
    room_id="exhibition_foyer",
)

HERO_ASSET_BINDINGS = HERO_BINDINGS[
    HERO_BINDINGS[
        "payload_ref"
    ].astype(str).str.startswith(
        "asset:"
    )
]

HERO_ASSET_ID = (
    str(
        HERO_ASSET_BINDINGS.iloc[0][
            "payload_ref"
        ]
    ).removeprefix("asset:")
    if len(HERO_ASSET_BINDINGS) == 1
    else ""
)

HERO_ASSET_RECORD = (
    PACKAGE.asset_record(
        HERO_ASSET_ID
    )
    if HERO_ASSET_ID
    else None
)

HERO_RENDITION_PATH = (
    PACKAGE.local_rendition(
        HERO_ASSET_ID,
        profiles=(
            "standard",
            "thumb",
        ),
        verify=True,
    )
    if HERO_ASSET_ID
    else None
)


PREVIEW_AUDIT_RECORDS = []

for (
    destination_room,
    preview_contract,
) in APP_ROUTE_PREVIEW_ASSETS.items():
    (
        source_room,
        asset_id,
    ) = preview_contract

    source_bindings = (
        PACKAGE.bindings_for(
            room_id=source_room
        )
    )

    registered = (
        f"asset:{asset_id}"
        in set(
            source_bindings[
                "payload_ref"
            ].astype(str)
        )
    )

    rendition_path = (
        PACKAGE.local_rendition(
            asset_id,
            profiles=(
                "thumb",
                "standard",
                "diagnostic_preview",
            ),
            verify=True,
        )
    )

    PREVIEW_AUDIT_RECORDS.append(
        {
            "destination_room": (
                destination_room
            ),
            "source_room": source_room,
            "asset_id": asset_id,
            "registered": registered,
            "rendition_path": (
                rendition_path.relative_to(
                    PROJECT_ROOT
                ).as_posix()
                if rendition_path
                is not None
                else ""
            ),
            "passed": (
                registered
                and rendition_path
                is not None
                and rendition_path.is_file()
            ),
        }
    )


FOYER_PREVIEW_AUDIT = pd.DataFrame(
    PREVIEW_AUDIT_RECORDS
)


# ---------------------------------------------------------------------------
# Cross-room prerequisite 1:
# deterministic LaMa must preserve an explicit uncertainty N/A state.
# ---------------------------------------------------------------------------

CASE_DEFAULT_SELECTION = (
    PACKAGE.default_selection_for_room(
        "case_explorer"
    )
)

CASE_UNCERTAINTY_BINDINGS = (
    PACKAGE.bindings_for(
        display_id=(
            "case.anchor.uncertainty"
        ),
        room_id="case_explorer",
    )
)

CASE_UNCERTAINTY_ASSETS = (
    CASE_UNCERTAINTY_BINDINGS[
        CASE_UNCERTAINTY_BINDINGS[
            "payload_ref"
        ].astype(str).str.startswith(
            "asset:"
        )
    ]
)

CASE_UNCERTAINTY_PARENT = (
    CASE_UNCERTAINTY_BINDINGS[
        CASE_UNCERTAINTY_BINDINGS[
            "slot_id"
        ].astype(str).eq(
            "case.anchor.uncertainty"
        )
    ]
)

CASE_NA_STATE_PASSED = (
    CASE_DEFAULT_SELECTION.model_id
    == "lama"
    and CASE_UNCERTAINTY_ASSETS.empty
    and len(
        CASE_UNCERTAINTY_PARENT
    ) == 1
    and str(
        CASE_UNCERTAINTY_PARENT.iloc[0][
            "applicability_state"
        ]
    )
    == (
        "not_applicable_"
        "deterministic_method"
    )
)


# ---------------------------------------------------------------------------
# Cross-room prerequisite 2:
# numerical displays must resolve typed records, never prose-parsed values.
# ---------------------------------------------------------------------------

TYPED_PAYLOAD_RECORDS = []

for display_id in (
    "case.metric.ssim",
    "case.metric.lpips",
    "case.metric.colour",
):
    payload = (
        PACKAGE.resolve_display_payload(
            display_id,
            CASE_DEFAULT_SELECTION,
        )
    )

    TYPED_PAYLOAD_RECORDS.append(
        {
            "display_id": display_id,
            "status": payload.status,
            "payload_kind": (
                payload.payload_kind
            ),
            "record_count": len(
                payload.records
            ),
            "source_sha256": (
                payload.source_sha256
            ),
            "passed": (
                payload.status
                == "available"
                and payload.payload_kind
                == "curated_typed_record"
                and len(
                    payload.records
                ) == 1
            ),
        }
    )


TYPED_PAYLOAD_AUDIT = pd.DataFrame(
    TYPED_PAYLOAD_RECORDS
)

NARRATIVE_PAYLOAD = (
    PACKAGE.resolve_display_payload(
        "metric.lens.pixel"
    )
)


FOYER_BINDING_CHECKS = (
    (
        "foyer_partition",
        (
            "The Foyer partition has exactly "
            "its seven approved display records"
        ),
        sorted(
            EXPECTED_FOYER_DISPLAY_IDS
        ),
        sorted(
            OBSERVED_FOYER_DISPLAY_IDS
        ),
        (
            OBSERVED_FOYER_DISPLAY_IDS
            == EXPECTED_FOYER_DISPLAY_IDS
        ),
    ),
    (
        "opening_contract",
        (
            "The Foyer uses the approved p001 "
            "introduction and scientific boundary"
        ),
        EXPECTED_FOYER_OPENING,
        {
            key: FOYER_OPENING[key]
            for key in (
                "painting_id",
                "question",
                "boundary",
            )
        },
        all(
            FOYER_OPENING[key]
            == value
            for key, value
            in EXPECTED_FOYER_OPENING.items()
        ),
    ),
    (
        "scope_indicators",
        (
            "The compact Foyer scope strip uses "
            "the Controlled-300 population"
        ),
        EXPECTED_SCOPE_INDICATORS,
        SCOPE_INDICATORS,
        (
            SCOPE_INDICATORS
            == EXPECTED_SCOPE_INDICATORS
        ),
    ),
    (
        "guided_tour_order",
        (
            "The guided tour follows all eight "
            "principal rooms in approved order"
        ),
        list(PRINCIPAL_ROOM_IDS),
        GUIDED_TOUR["room_ids"],
        (
            GUIDED_TOUR["room_ids"]
            == list(
                PRINCIPAL_ROOM_IDS
            )
        ),
    ),
    (
        "hero_binding",
        (
            "The dominant Foyer painting resolves "
            "to one verified p001 clean reference"
        ),
        {
            "asset_bindings": 1,
            "source_row_id": "p001",
            "rendition_available": True,
        },
        {
            "asset_bindings": len(
                HERO_ASSET_BINDINGS
            ),
            "source_row_id": (
                str(
                    HERO_ASSET_RECORD[
                        "source_row_id"
                    ]
                )
                if HERO_ASSET_RECORD
                is not None
                else ""
            ),
            "rendition_available": (
                HERO_RENDITION_PATH
                is not None
                and HERO_RENDITION_PATH.is_file()
            ),
        },
        (
            len(
                HERO_ASSET_BINDINGS
            ) == 1
            and HERO_ASSET_RECORD
            is not None
            and str(
                HERO_ASSET_RECORD[
                    "source_row_id"
                ]
            ) == "p001"
            and HERO_RENDITION_PATH
            is not None
            and HERO_RENDITION_PATH.is_file()
        ),
    ),
    (
        "route_preview_assets",
        (
            "All seven destination previews use "
            "explicit registered and verified assets"
        ),
        {
            "preview_count": 7,
            "failed": 0,
        },
        {
            "preview_count": len(
                FOYER_PREVIEW_AUDIT
            ),
            "failed": int(
                (
                    ~FOYER_PREVIEW_AUDIT[
                        "passed"
                    ]
                ).sum()
            ),
        },
        (
            len(
                FOYER_PREVIEW_AUDIT
            ) == 7
            and bool(
                FOYER_PREVIEW_AUDIT[
                    "passed"
                ].all()
            )
        ),
    ),
    (
        "case_uncertainty_na",
        (
            "The default deterministic LaMa case "
            "preserves an explicit uncertainty N/A state"
        ),
        {
            "model_id": "lama",
            "uncertainty_assets": 0,
            "applicability_state": (
                "not_applicable_"
                "deterministic_method"
            ),
        },
        {
            "model_id": (
                CASE_DEFAULT_SELECTION.model_id
            ),
            "uncertainty_assets": len(
                CASE_UNCERTAINTY_ASSETS
            ),
            "applicability_state": (
                str(
                    CASE_UNCERTAINTY_PARENT.iloc[
                        0
                    ][
                        "applicability_state"
                    ]
                )
                if len(
                    CASE_UNCERTAINTY_PARENT
                ) == 1
                else ""
            ),
        },
        CASE_NA_STATE_PASSED,
    ),
    (
        "typed_numeric_payloads",
        (
            "The three opening Case Explorer "
            "metrics resolve as typed records"
        ),
        {
            "payloads": 3,
            "failed": 0,
        },
        {
            "payloads": len(
                TYPED_PAYLOAD_AUDIT
            ),
            "failed": int(
                (
                    ~TYPED_PAYLOAD_AUDIT[
                        "passed"
                    ]
                ).sum()
            ),
        },
        (
            len(
                TYPED_PAYLOAD_AUDIT
            ) == 3
            and bool(
                TYPED_PAYLOAD_AUDIT[
                    "passed"
                ].all()
            )
        ),
    ),
    (
        "narrative_not_number_parsed",
        (
            "Narrative room specifications are "
            "not parsed as scientific measurements"
        ),
        {
            "status": "narrative",
            "record_count": 0,
        },
        {
            "status": (
                NARRATIVE_PAYLOAD.status
            ),
            "record_count": len(
                NARRATIVE_PAYLOAD.records
            ),
        },
        (
            NARRATIVE_PAYLOAD.status
            == "narrative"
            and not NARRATIVE_PAYLOAD.records
        ),
    ),
)


FOYER_STAGE_RECORDS = [
    {
        "check_id": check_id,
        "check_description": description,
        "severity": "blocking",
        "expected": expected,
        "observed": observed,
        "passed": passed,
        "details": (
            ""
            if passed
            else (
                "The Foyer or prerequisite "
                "binding contract differs."
            )
        ),
    }
    for (
        check_id,
        description,
        expected,
        observed,
        passed,
    ) in FOYER_BINDING_CHECKS
]


replace_validation_stage(
    "batch_2_foyer_contract",
    FOYER_STAGE_RECORDS,
)

VALIDATION.raise_for_blocking()

display(FOYER_ROOM)
display(FOYER_PREVIEW_AUDIT)
display(TYPED_PAYLOAD_AUDIT)

print("Exhibition Foyer evidence contract passed.")
print("Foyer display records:", len(FOYER_ROOM))
print("Hero asset:", HERO_ASSET_ID)
print(
    "Verified destination previews:",
    int(
        FOYER_PREVIEW_AUDIT[
            "passed"
        ].sum()
    ),
)
print(
    "Default Case Explorer uncertainty:",
    CASE_UNCERTAINTY_PARENT.iloc[0][
        "interpretation"
    ],
)

,room_record_id,room_id,subroute_id,section_id,display_id,slot_id,display_order,record_type,record_key,label,...,interpretation,limitation,denominator,applicability_state,source_notebook_ids_json,source_paths_json,source_row_ids_json,schema_version,status,issue
0,room_record_1ededde46bc32fb70cafc307,exhibition_foyer,None,foyer.hero,foyer.hero.p001,foyer.hero.p001,1,image,foyer.hero.p001,Dominant Juan de Pareja exhibit,...,Dominant Juan de Pareja exhibit,Visual plausibility is not the same as restora...,None,available,"[""02"", ""01""]","[""docs/dashboard_design_finalists/controlled_3...","[""foyer.hero.p001""]",dashboard_room_partition.v1,ok,
1,room_record_9189afa5e69e6ce431a28304,exhibition_foyer,None,foyer.scope,foyer.scope.paintings,foyer.scope.paintings,2,metric,foyer.scope.paintings,300 paintings,...,300 paintings,Visual plausibility is not the same as restora...,300 paintings,available,"[""01""]","[""docs/dashboard_design_finalists/controlled_3...","[""foyer.scope.paintings""]",dashboard_room_partition.v1,ok,
2,room_record_ab4c312365a121e829f70863,exhibition_foyer,None,foyer.scope,foyer.scope.categories,foyer.scope.categories,3,metric,foyer.scope.categories,"Five categories, 60 each",...,"Five categories, 60 each",Visual plausibility is not the same as restora...,300 paintings across five categories,available,"[""01""]","[""docs/dashboard_design_finalists/controlled_3...","[""foyer.scope.categories""]",dashboard_room_partition.v1,ok,
3,room_record_e6b6e8b73f47033b316dd9b6,exhibition_foyer,None,foyer.scope,foyer.scope.cases,foyer.scope.cases,4,metric,foyer.scope.cases,"3,425 registered cases",...,"3,425 registered cases",Visual plausibility is not the same as restora...,"3,425 registered cases",available,"[""08""]","[""docs/dashboard_design_finalists/controlled_3...","[""foyer.scope.cases""]",dashboard_room_partition.v1,ok,
4,room_record_d8975b9f77b2f97c5a7a254a,exhibition_foyer,None,foyer.scope,foyer.scope.methods,foyer.scope.methods,5,metric,foyer.scope.methods,Four full-scope methods plus bounded SDXL,...,Four full-scope methods plus bounded SDXL,Visual plausibility is not the same as restora...,None,available,"[""09"", ""10"", ""12A"", ""11"", ""12""]","[""docs/dashboard_design_finalists/controlled_3...","[""foyer.scope.methods""]",dashboard_room_partition.v1,ok,
5,room_record_8ea76a25cff8382c8e07dad9,exhibition_foyer,None,foyer.route,foyer.route.previews,foyer.route.previews,6,route,foyer.route.previews,Seven destination-room preview thumbnails and ...,...,Seven destination-room preview thumbnails and ...,Visual plausibility is not the same as restora...,None,available,[],"[""docs/dashboard_design_finalists/controlled_3...","[""foyer.route.previews""]",dashboard_room_partition.v1,ok,
6,room_record_f6b2dd96a713b3476a3e272d,exhibition_foyer,None,foyer.tour,foyer.tour.frames,foyer.tour.frames,7,route,foyer.tour.frames,"One visual, takeaway and limitation per room",...,"One visual, takeaway and limitation per room",Visual plausibility is not the same as restora...,None,available,"[""33""]","[""docs/dashboard_design_finalists/controlled_3...","[""foyer.tour.frames""]",dashboard_room_partition.v1,ok,


,destination_room,source_room,asset_id,registered,rendition_path,passed
0,study_design,exhibition_foyer,asset_51c9c8fd90b375cfcdcc0934,True,outputs/34_final_streamlit_dashboard_assets/as...,True
1,metric_framework,metric_framework,asset_e8837231a30b63c6cda95404,True,outputs/34_final_streamlit_dashboard_assets/as...,True
2,model_gallery,model_gallery,asset_0b881fb2b09b7b805308cb18,True,outputs/34_final_streamlit_dashboard_assets/as...,True
3,stability_lab,stability_lab,asset_9dee66286e792c92af5d7200,True,outputs/34_final_streamlit_dashboard_assets/as...,True
4,trustworthiness,trustworthiness,asset_abecb3b93925d2ed8d45aa07,True,outputs/34_final_streamlit_dashboard_assets/as...,True
5,case_explorer,case_explorer,asset_1c9654b8fe6387ee4a4e6bc8,True,outputs/34_final_streamlit_dashboard_assets/as...,True
6,research_archive,research_archive,asset_d8b3392fc1c1f8e2612c7212,True,outputs/34_final_streamlit_dashboard_assets/as...,True


,display_id,status,payload_kind,record_count,source_sha256,passed
0,case.metric.ssim,available,curated_typed_record,1,a9a1250c7ecd1339dbb4150d8f42336ea3d1b0567648f6...,True
1,case.metric.lpips,available,curated_typed_record,1,0d2b125a5045459db4d07648ac52af14faeb5c4ae91873...,True
2,case.metric.colour,available,curated_typed_record,1,662dd76152d26a3b6e317067ac49d33eaca249bf724bbd...,True


Exhibition Foyer evidence contract passed.
Foyer display records: 7
Hero asset: asset_51c9c8fd90b375cfcdcc0934
Verified destination previews: 7
Default Case Explorer uncertainty: Not applicable — deterministic method


In [9]:
# Server-side smoke only: AppTest does not execute browser JavaScript.
drop_validation_stages(("batch_2_streamlit_smoke", "batch_2_final_gate"))
from streamlit.testing.v1 import AppTest
from contextlib import ExitStack
import time

def foyer_smoke(query):
    started = time.perf_counter()
    requests = []
    def reject_network(*args, **kwargs):
        requests.append("blocked network attempt")
        raise AssertionError("Batch 2 server smoke attempted network access")
    with ExitStack() as stack:
        for target in ("urllib.request.urlopen", "urllib.request.urlretrieve",
                       "socket.create_connection", "socket.socket.connect",
                       "socket.socket.connect_ex"):
            stack.enter_context(mock.patch(target, side_effect=reject_network))
        app = AppTest.from_file(str(APP_PATH), default_timeout=60)
        for key, value in query.items():
            app.query_params[key] = value
        app.run()
    markup = "\n".join(str(element.value) for element in app.markdown)
    # components.html is represented by an iframe proto, not a Markdown card.
    frames = [str(element.proto.srcdoc) for element in app.get("iframe")]
    return {
        "app": app, "markup": markup, "frames": frames,
        "errors": [str(e.value) for e in app.exception],
        "buttons": [b.label for b in app.button],
        "network_attempts": len(requests),
        "elapsed_seconds": round(time.perf_counter() - started, 3),
    }

INITIAL_SMOKE = foyer_smoke({"room": "exhibition_foyer"})
TOUR_SMOKE = foyer_smoke({"room": "exhibition_foyer", "tour": "1"})
INITIAL_APP_TEST, TOUR_APP_TEST = INITIAL_SMOKE["app"], TOUR_SMOKE["app"]
INITIAL_MARKDOWN, TOUR_MARKDOWN = INITIAL_SMOKE["markup"], TOUR_SMOKE["markup"]
INITIAL_EXCEPTIONS, TOUR_EXCEPTIONS = INITIAL_SMOKE["errors"], TOUR_SMOKE["errors"]
INITIAL_BUTTON_LABELS = INITIAL_SMOKE["buttons"]
EXPECTED_PRINCIPAL_LABELS = [APP_ROOM_LABELS[r] for r in PRINCIPAL_ROOM_IDS]
LOCAL_IMAGE_COUNT = INITIAL_MARKDOWN.count('src="data:image/')
ROUTE_PREVIEW_COUNT = INITIAL_MARKDOWN.count('class="route-preview"')
QUESTION_FRAGMENTS = ("How should we", "judge an AI-restored", "painting?")

def visitor_payload(smoke, legacy):
    frames = [f for f in smoke["frames"] if "MuseumVisit.install(window," in f]
    if len(frames) != 1:
        return False
    expected = dict(VISITOR_DATA, room="exhibition_foyer", legacyLaunch=legacy)
    encoded = json.dumps(expected, ensure_ascii=True).replace("</", "<\\/")
    css_encoded = json.dumps(VISITOR_CSS).replace("</", "<\\/")
    return (
        frames[0] == "<script>" + VISITOR_SCRIPT + "\nMuseumVisit.install(window,"
        + encoded + "," + css_encoded + ");</script>"
    )

APP_TEST_CHECKS = (
    ("foyer_startup", "Both Foyer server-side smoke runs have no exception",
     [], INITIAL_EXCEPTIONS + TOUR_EXCEPTIONS,
     not INITIAL_EXCEPTIONS and not TOUR_EXCEPTIONS),
    ("semantic_entry_controls", "Both approved links carry visitor-dialog hooks",
     True, all(s in INITIAL_MARKDOWN for s in (
         "Take the guided tour", "Explore freely", 'data-museum-visit="tour"',
         'data-museum-visit="free"', 'aria-haspopup="dialog"')),
     all(s in INITIAL_MARKDOWN for s in (
         "Take the guided tour", "Explore freely", 'data-museum-visit="tour"',
         'data-museum-visit="free"', 'aria-haspopup="dialog"'))),
    ("foyer_opening_copy", "The approved question, p001 and local evidence render",
     True, all(s in INITIAL_MARKDOWN for s in (*QUESTION_FRAGMENTS, "p001", "data:image/")),
     all(s in INITIAL_MARKDOWN for s in (*QUESTION_FRAGMENTS, "p001", "data:image/"))),
    ("principal_navigation", "All eight approved principal labels render",
     EXPECTED_PRINCIPAL_LABELS,
     [s for s in EXPECTED_PRINCIPAL_LABELS if s in INITIAL_MARKDOWN],
     all(s in INITIAL_MARKDOWN for s in EXPECTED_PRINCIPAL_LABELS)),
    ("route_preview_content", "Seven destination previews render", 7,
     ROUTE_PREVIEW_COUNT, ROUTE_PREVIEW_COUNT == 7),
    ("local_verified_images", "Shell, hero and seven previews use local data images",
     9, LOCAL_IMAGE_COUNT, LOCAL_IMAGE_COUNT == 9),
    ("scientific_boundary", "Trustworthiness, historical and category limits render",
     True, all(s in INITIAL_MARKDOWN for s in (
         FOYER_OPENING["boundary"], "historically correct", "not historical styles")),
     all(s in INITIAL_MARKDOWN for s in (
         FOYER_OPENING["boundary"], "historically correct", "not historical styles"))),
    ("zigzag_route_runtime", "The approved SVG route is delivered",
     True, 'class="route-line"' in INITIAL_MARKDOWN and "<polyline" in INITIAL_MARKDOWN,
     'class="route-line"' in INITIAL_MARKDOWN and "<polyline" in INITIAL_MARKDOWN),
    ("visitor_default_payload", "Default entry delivers the exact inactive visitor payload",
     True, visitor_payload(INITIAL_SMOKE, False), visitor_payload(INITIAL_SMOKE, False)),
    ("visitor_legacy_payload", "Legacy tour query delivers the exact launch payload",
     True, visitor_payload(TOUR_SMOKE, True), visitor_payload(TOUR_SMOKE, True)),
    ("no_stray_native_buttons", "The Foyer has no unexpected native Streamlit buttons",
     [], INITIAL_BUTTON_LABELS + TOUR_SMOKE["buttons"],
     not INITIAL_BUTTON_LABELS and not TOUR_SMOKE["buttons"]),
    ("server_network_attempts", "The guarded smoke makes zero network attempts",
     0, INITIAL_SMOKE["network_attempts"] + TOUR_SMOKE["network_attempts"],
     INITIAL_SMOKE["network_attempts"] + TOUR_SMOKE["network_attempts"] == 0),
)
APP_TEST_STAGE_RECORDS = [application_source_check(*r) for r in APP_TEST_CHECKS]
replace_validation_stage("batch_2_streamlit_smoke", APP_TEST_STAGE_RECORDS)
APP_TEST_RESULTS = pd.DataFrame(APP_TEST_STAGE_RECORDS)
display(APP_TEST_RESULTS)
VALIDATION.raise_for_blocking()
print("Batch 2 server-side Foyer smoke passed; browser execution is NOT tested here.")
print("Observed run seconds (not a cold-cloud/p95 benchmark):",
      INITIAL_SMOKE["elapsed_seconds"], TOUR_SMOKE["elapsed_seconds"])


2026-10-01 08:55:06.152 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 08:55:07.936 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 08:55:07.950 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 08:55:09.087 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


,check_id,check_description,severity,expected,observed,passed,details
0,foyer_startup,Both Foyer server-side smoke runs have no exce...,blocking,[],[],True,
1,semantic_entry_controls,Both approved links carry visitor-dialog hooks,blocking,True,True,True,
2,foyer_opening_copy,"The approved question, p001 and local evidence...",blocking,True,True,True,
3,principal_navigation,All eight approved principal labels render,blocking,"[Exhibition Foyer, Study Design, Metric Framew...","[Exhibition Foyer, Study Design, Metric Framew...",True,
4,route_preview_content,Seven destination previews render,blocking,7,7,True,
5,local_verified_images,"Shell, hero and seven previews use local data ...",blocking,9,9,True,
6,scientific_boundary,"Trustworthiness, historical and category limit...",blocking,True,True,True,
7,zigzag_route_runtime,The approved SVG route is delivered,blocking,True,True,True,
8,visitor_default_payload,Default entry delivers the exact inactive visi...,blocking,True,True,True,
9,visitor_legacy_payload,Legacy tour query delivers the exact launch pa...,blocking,True,True,True,


Batch 2 server-side Foyer smoke passed; browser execution is NOT tested here.
Observed run seconds (not a cold-cloud/p95 benchmark): 1.796 1.145


In [10]:
drop_validation_stages(('batch_2_final_gate',))

PRE_FINAL_VALIDATION = (
    VALIDATION.to_dataframe()
)

PRE_FINAL_BATCH_2 = (
    PRE_FINAL_VALIDATION[
        PRE_FINAL_VALIDATION[
            "validation_stage"
        ].str.startswith(
            "batch_2_"
        )
    ].copy()
)

EXPECTED_BATCH_2_STAGES = {
    "batch_2_application_source", "batch_2_foyer_contract", "batch_2_streamlit_smoke"
}
OBSERVED_BATCH_2_STAGES = set(PRE_FINAL_BATCH_2["validation_stage"])
if OBSERVED_BATCH_2_STAGES != EXPECTED_BATCH_2_STAGES:
    raise RuntimeError("Run every Batch 2 cell in order; missing/unexpected stages: "
                       + str(EXPECTED_BATCH_2_STAGES ^ OBSERVED_BATCH_2_STAGES))

PRE_FINAL_BLOCKING_FAILURES = (
    PRE_FINAL_BATCH_2[
        (
            PRE_FINAL_BATCH_2[
                "severity"
            ]
            == "blocking"
        )
        & (
            ~PRE_FINAL_BATCH_2[
                "passed"
            ].map(truthy)
        )
    ]
)

CANONICAL_FILES_AFTER_BATCH_2 = (
    sorted(
        path.relative_to(
            OUTPUT_ROOT
        ).as_posix()
        for path
        in OUTPUT_ROOT.rglob("*")
        if path.is_file()
    )
    if OUTPUT_ROOT.exists()
    else []
)


BATCH_2_FINAL_RECORDS = [
    {
        "check_id": (
            "prior_batch_2_checks"
        ),
        "check_description": (
            "All Batch 2 source, evidence "
            "and runtime checks passed"
        ),
        "severity": "blocking",
        "expected": 0,
        "observed": len(
            PRE_FINAL_BLOCKING_FAILURES
        ),
        "passed": (
            PRE_FINAL_BLOCKING_FAILURES.empty
        ),
        "details": (
            ""
            if PRE_FINAL_BLOCKING_FAILURES.empty
            else (
                PRE_FINAL_BLOCKING_FAILURES[
                    [
                        "validation_stage",
                        "check_id",
                    ]
                ].to_dict(
                    orient="records"
                )
            )
        ),
    },
    {
        "check_id": (
            "no_canonical_files_written"
        ),
        "check_description": (
            "Batch 2 has not persisted "
            "canonical Notebook 35 outputs"
        ),
        "severity": "blocking",
        "expected": [],
        "observed": (
            CANONICAL_FILES_AFTER_BATCH_2
        ),
        "passed": not (
            CANONICAL_FILES_AFTER_BATCH_2
        ),
        "details": (
            ""
            if not (
                CANONICAL_FILES_AFTER_BATCH_2
            )
            else (
                "Canonical N35 files must "
                "remain absent until the "
                "final consolidation batch."
            )
        ),
    },
]


replace_validation_stage(
    "batch_2_final_gate",
    BATCH_2_FINAL_RECORDS,
)

VALIDATION.raise_for_blocking()

VALIDATION_FRAME = (
    VALIDATION.to_dataframe()
)

BATCH_2_STAGE_SUMMARY = (
    VALIDATION_FRAME[
        VALIDATION_FRAME[
            "validation_stage"
        ].str.startswith(
            "batch_2_"
        )
    ]
    .groupby(
        [
            "validation_stage",
            "severity",
        ],
        dropna=False,
    )
    .agg(
        checks=(
            "check_id",
            "size",
        ),
        passed=(
            "passed",
            "sum",
        ),
    )
    .reset_index()
)

BATCH_2_STAGE_SUMMARY[
    "failed"
] = (
    BATCH_2_STAGE_SUMMARY[
        "checks"
    ]
    - BATCH_2_STAGE_SUMMARY[
        "passed"
    ]
)

display(BATCH_2_STAGE_SUMMARY)

print("Batch 2 local source/binding/server-smoke gate passed.")
print("Presentation was previously user-approved; fresh browser validation is NOT claimed.")
print("N35 completion: NO. External deployment validation: NOT RUN.")
print("STOP HERE. Do not run the unreconciled Batch 3+ / historical pilot cells.")
print(
    "Batch 2 checks:",
    len(
        VALIDATION_FRAME[
            VALIDATION_FRAME[
                "validation_stage"
            ].str.startswith(
                "batch_2_"
            )
        ]
    ),
)
print(
    "Blocking failures:",
    len(
        VALIDATION.blocking_failures
    ),
)
print(
    "Canonical files persisted:",
    len(
        CANONICAL_FILES_AFTER_BATCH_2
    ),
)
print()
print("Visual checkpoint command:")
print(
    "python -m streamlit run "
    "streamlit_app.py"
)
print(
    "Open: "
    "http://localhost:8501/"
    "?room=exhibition_foyer"
)
PENDING_N35_GATES = pd.DataFrame([
    {"gate": label, "status": "pending_current_release_validation"}
    for label in (
        "Batches 3-9 current room validation reconciliation",
        "Browser tour/free-popup actions, focus and keyboard behavior",
        "Desktop/tablet/mobile rendering, Firefox, zoom and second monitor",
        "Clean-checkout assets and exact remote retrieval",
        "Cold/warm cache, integrity failure and retry behavior",
        "Startup/RSS/latency/concurrency budgets",
        "Deployment dependency alignment and live-site checks",
        "Batch 10 canonical readiness report and persistence",
    )
])
display(PENDING_N35_GATES)


,validation_stage,severity,checks,passed,failed
0,batch_2_application_source,blocking,30,30,0
1,batch_2_final_gate,blocking,2,2,0
2,batch_2_foyer_contract,blocking,9,9,0
3,batch_2_streamlit_smoke,blocking,12,12,0


Batch 2 local source/binding/server-smoke gate passed.
Presentation was previously user-approved; fresh browser validation is NOT claimed.
N35 completion: NO. External deployment validation: NOT RUN.
STOP HERE. Do not run the unreconciled Batch 3+ / historical pilot cells.
Batch 2 checks: 53
Blocking failures: 0
Canonical files persisted: 0

Visual checkpoint command:
python -m streamlit run streamlit_app.py
Open: http://localhost:8501/?room=exhibition_foyer


,gate,status
0,Batches 3-9 current room validation reconcilia...,pending_current_release_validation
1,"Browser tour/free-popup actions, focus and key...",pending_current_release_validation
2,"Desktop/tablet/mobile rendering, Firefox, zoom...",pending_current_release_validation
3,Clean-checkout assets and exact remote retrieval,pending_current_release_validation
4,"Cold/warm cache, integrity failure and retry b...",pending_current_release_validation
5,Startup/RSS/latency/concurrency budgets,pending_current_release_validation
6,Deployment dependency alignment and live-site ...,pending_current_release_validation
7,Batch 10 canonical readiness report and persis...,pending_current_release_validation


## Batch 3 — Study Design: current-release validation

The approved dashboard is frozen. This batch validates it without changing its layout:
13 exact N34 room records, five category representatives, 105 visible saved case routes,
four parallel experiment paths, six Study Notes, shared D02 routing and native query callbacks.
The current visitor-overlay freeze replaces the obsolete tour-function assertion.

Requires successful Batches 1–2 in this kernel. Use all five code cells in order.
Rerunning an earlier cell invalidates later gates. No notebook editing, producer execution,
network access, server launch or canonical N35 persistence is performed by these cells.
AppTest checks Python-rendered markup and callbacks, not browser JavaScript or visual alignment.
The exact evidence/relationship cells are retained, with rerun and offline guards added.


In [11]:
# Shared read-only room validation harness. Requires the completed Batch 2 kernel.
import importlib
TOOLS_ROOT = str(PROJECT_ROOT / "tools")
if TOOLS_ROOT not in sys.path:
    sys.path.insert(0, TOOLS_ROOT)
import n35_room_validation as n35
n35 = importlib.reload(n35)
n35.required_stages(VALIDATION.checks, ["batch_2_final_gate"])
drop_validation_stages(tuple(f"batch_{i}_" for i in range(3, 11)))
VALIDATION.raise_for_blocking()

N35_BATCH_START = {}
N35_PROTECTED_PATHS = sorted({
    APP_PATH,
    *list((PROJECT_ROOT / "src/restoration_eval").glob("*.py")),
    *list((PROJECT_ROOT / "streamlit_assets").rglob("*.js")),
    *list((PROJECT_ROOT / "streamlit_assets").rglob("*.css")),
    PROJECT_ROOT / "streamlit_assets/room_runtime/index.html",
})

def protected_snapshot():
    return {p.relative_to(PROJECT_ROOT).as_posix(): n35.digest(p)
            for p in N35_PROTECTED_PATHS}

def start_room_batch(number):
    n35.required_stages(VALIDATION.checks, [f"batch_{number - 1}_final_gate"])
    VALIDATION.raise_for_blocking()
    if n35.snapshot(OUTPUT_ROOT):
        raise RuntimeError("Canonical N35 files must remain absent until Batch 10.")
    N35_BATCH_START[number] = (n35.snapshot(OUTPUT_ROOT), protected_snapshot())

def room_check(key, expected, observed, passed=None, description=None):
    return application_source_check(key, description or key.replace("_", " "),
        expected, observed, expected == observed if passed is None else passed)

def record_room_stage(stage, rows):
    if not rows:
        raise RuntimeError("An empty room stage cannot pass.")
    replace_validation_stage(stage, rows)
    display(pd.DataFrame(rows)[["check_id", "passed", "expected", "observed"]])
    VALIDATION.raise_for_blocking()

def test_record(group):
    print(f"Running {group} local contract tests; no inference or artifact generation.")
    result = n35.run_suite(PROJECT_ROOT, group)
    print(result["transcript"])
    print("Historical checks replaced:", result["excluded_historical_checks"])
    return room_check("focused_tests", True, result["passed"],
        description=json.dumps({k: v for k, v in result.items() if k != "transcript"}))

def finish_room_batch(number, expected_stages):
    n35.required_stages(VALIDATION.checks, expected_stages)
    observed = {c.validation_stage for c in VALIDATION.checks
                if c.validation_stage.startswith(f"batch_{number}_")
                and c.validation_stage != f"batch_{number}_final_gate"}
    if observed != set(expected_stages):
        raise RuntimeError(f"Unexpected/missing stages: {observed ^ set(expected_stages)}")
    before = N35_BATCH_START[number]
    after = (n35.snapshot(OUTPUT_ROOT), protected_snapshot())
    record_room_stage(f"batch_{number}_final_gate", [
        room_check("read_only_outputs_and_dashboard", before, after),
        room_check("canonical_files_absent", {}, after[0]),
    ])
    frame = VALIDATION.to_dataframe()
    display(frame[frame.validation_stage.str.startswith(f"batch_{number}_")]
        .groupby("validation_stage").agg(checks=("check_id", "size"), passed=("passed", "sum")))
    print(f"Batch {number}: local source/data/server checks passed. Canonical outputs: 0.")
    print("Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.")
    print("N35 completion: NO. These cells do not certify external deployment readiness.")

start_room_batch(3)
APP_SOURCE = APP_PATH.read_text(encoding="utf-8")
APP_TREE = ast.parse(APP_SOURCE)
N35_APP_CONSTANTS = n35.literals(APP_PATH)
for name in ("STUDY_SHELL_RELATIVE_PATH", "STUDY_SHELL_SHA256", "STUDY_CLEAN_ASSET_ID",
             "STUDY_CORE_ASSETS", "STUDY_CATEGORY_LABELS", "STUDY_CATEGORY_ANCHORS",
             "STUDY_PATH_DEFAULT_OPTIONS", "STUDY_OTHER_CHANGE_OPTIONS", "STUDY_NOTE_COPY"):
    globals()["APP_" + name] = N35_APP_CONSTANTS[name]

# Existing tests normalize ONLY the approved visitor-overlay delta; no new
# fingerprints are invented to bless drift in the Foyer or Study layout.
record_room_stage("batch_3_application_source", [
    room_check("study_build", "n35.batch3.study.v4", N35_APP_CONSTANTS["STUDY_BUILD"]),
    test_record("study"),
])


Running study local contract tests; no inference or artifact generation.
test_asset_rendition_and_locator_apis_preserve_identity (n35_checks_test_dashboard_application.DashboardApplicationV2Tests.test_asset_rendition_and_locator_apis_preserve_identity) ... ok
test_case_explorer_default_is_exact_and_cross_experiment_safe (n35_checks_test_dashboard_application.DashboardApplicationV2Tests.test_case_explorer_default_is_exact_and_cross_experiment_safe) ... ok
test_curated_case_metrics_resolve_only_for_exact_p018_identity (n35_checks_test_dashboard_application.DashboardApplicationV2Tests.test_curated_case_metrics_resolve_only_for_exact_p018_identity) ... ok
test_curated_threshold_is_exact_and_mismatch_never_substitutes (n35_checks_test_dashboard_application.DashboardApplicationV2Tests.test_curated_threshold_is_exact_and_mismatch_never_substitutes) ... ok
test_every_room_and_d02_partition_loads_exact_identity (n35_checks_test_dashboard_application.DashboardApplicationV2Tests.test_every_room_a

,check_id,passed,expected,observed
0,study_build,True,n35.batch3.study.v4,n35.batch3.study.v4
1,focused_tests,True,True,True


In [12]:
drop_validation_stages(('batch_3_study_evidence', 'batch_3_study_relationships', 'batch_3_streamlit_fragment_smoke', 'batch_3_final_gate', 'batch_4_', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_3_application_source'])
with n35.offline() as STUDY_EVIDENCE_NETWORK:
    STUDY_ROOM = PACKAGE.load_room(
        "study_design"
    )
    STUDY_BINDINGS = PACKAGE.bindings_for(
        room_id="study_design"
    )

    EXPECTED_STUDY_DISPLAY_IDS = {
        "study.anchor.clean",
        "study.core.conditions",
        "study.branch.damage_size",
        "study.branch.mask_placement",
        "study.branch.other_changes",
        "study.routing.status",
        "study.scope.catalogues",
        "study.collection.drawers",
        "study.notes.selection",
        "study.notes.preprocessing",
        "study.notes.damage",
        "study.notes.policy",
        "study.d02.plaque",
    }
    OBSERVED_STUDY_DISPLAY_IDS = set(
        STUDY_ROOM[
            "display_id"
        ].astype(str)
    )

    STUDY_CATALOGUE_MATCHES = [
        row
        for row
        in PACKAGE.room_catalogue["rooms"]
        if str(row.get("room_id"))
        == "study_design"
    ]
    STUDY_CATALOGUE = (
        STUDY_CATALOGUE_MATCHES[0]
        if len(STUDY_CATALOGUE_MATCHES) == 1
        else {}
    )
    EXPECTED_STUDY_OPENING = {
        "question": (
            "What did we test, and how did we "
            "keep the comparison fair?"
        ),
        "introduction": (
            "We started with 300 paintings, divided "
            "evenly across five broad visual categories. "
            "Every painting received the same core tests, "
            "while smaller balanced experiments examined "
            "damage size, mask placement and other image "
            "changes."
        ),
        "boundary": (
            "Equal category sizes make comparisons easier. "
            "They do not make the collection representative "
            "of every artistic period, culture or style."
        ),
    }

    EXPECTED_STUDY_POPULATION = {
        "painting_count": 300,
        "visual_category_count": 5,
        "paintings_per_category": 60,
        "registered_case_count": 3425,
        "four_model_eligible_case_count": 2620,
        "d02_portraits_screened": 60,
        "d02_matched_cases": 45,
        "d02_matched_painting_count": 20,
    }
    OBSERVED_STUDY_POPULATION = {
        key: int(PACKAGE.population[key])
        for key
        in EXPECTED_STUDY_POPULATION
    }

    EXPECTED_MAPPING_STATE_COUNTS = {
        "existing_direct": 5,
        "existing_filtered": 5,
        "n34_derived_registered": 3,
    }
    OBSERVED_MAPPING_STATE_COUNTS = {
        str(key): int(value)
        for key, value
        in (
            STUDY_BINDINGS[
                "mapping_state"
            ]
            .astype(str)
            .value_counts()
            .to_dict()
            .items()
        )
    }

    PRESENTATION = CONFIG["presentation"]
    STUDY_SHELL_CONTRACT = (
        PRESENTATION[
            "decorative_shells"
        ]["study_design"]
    )
    STUDY_SHELL_PATH = (
        PROJECT_ROOT
        / STUDY_SHELL_CONTRACT["path"]
    ).resolve()
    STUDY_SHELL_OBSERVED_SHA256 = (
        hashlib.sha256(
            STUDY_SHELL_PATH.read_bytes()
        ).hexdigest()
    )
    STUDY_SHELL_PASSED = (
        STUDY_SHELL_PATH.is_relative_to(
            PROJECT_ROOT
        )
        and STUDY_SHELL_PATH.is_file()
        and STUDY_SHELL_OBSERVED_SHA256
        == STUDY_SHELL_CONTRACT["sha256"]
        == APP_STUDY_SHELL_SHA256
        and STUDY_SHELL_CONTRACT["path"]
        == APP_STUDY_SHELL_RELATIVE_PATH
        and STUDY_SHELL_CONTRACT[
            "classification"
        ] == "n35_decorative_ambience_only"
        and not STUDY_SHELL_CONTRACT[
            "scientific_content_allowed"
        ]
        and STUDY_SHELL_CONTRACT[
            "live_overlay_required"
        ]
    )

    STUDY_OPENING_ASSET_CONTRACT = (
        PRESENTATION[
            "study_design_exact_opening_assets"
        ]
    )
    STUDY_OPENING_ASSET_RECORDS = []

    for condition_id, record in (
        STUDY_OPENING_ASSET_CONTRACT[
            "assets"
        ].items()
    ):
        deployed_path = (
            PROJECT_ROOT
            / record["path"]
        ).resolve()
        source_path = (
            PROJECT_ROOT
            / record["source_path"]
        ).resolve()
        deployed_sha256 = hashlib.sha256(
            deployed_path.read_bytes()
        ).hexdigest()
        source_sha256 = hashlib.sha256(
            source_path.read_bytes()
        ).hexdigest()
        app_record = (
            APP_STUDY_CORE_ASSETS.get(
                condition_id,
                {},
            )
        )
        STUDY_OPENING_ASSET_RECORDS.append(
            {
                "condition_id": condition_id,
                "deployed_path": record["path"],
                "source_path": (
                    record["source_path"]
                ),
                "expected_sha256": (
                    record["sha256"]
                ),
                "deployed_sha256": (
                    deployed_sha256
                ),
                "source_sha256": source_sha256,
                "passed": (
                    deployed_path.is_relative_to(
                        PROJECT_ROOT
                    )
                    and source_path.is_relative_to(
                        PROJECT_ROOT
                    )
                    and deployed_path.is_file()
                    and source_path.is_file()
                    and deployed_sha256
                    == source_sha256
                    == record["sha256"]
                    and app_record.get("path")
                    == record["path"]
                    and app_record.get(
                        "source_path"
                    ) == record["source_path"]
                    and app_record.get("sha256")
                    == record["sha256"]
                ),
            }
        )

    STUDY_OPENING_ASSET_AUDIT = (
        pd.DataFrame(
            STUDY_OPENING_ASSET_RECORDS
        )
    )

    STUDY_CLEAN_ASSET_RECORD = (
        PACKAGE.asset_record(
            APP_STUDY_CLEAN_ASSET_ID
        )
    )
    STUDY_CLEAN_RENDITION_PATH = (
        PACKAGE.local_rendition(
            APP_STUDY_CLEAN_ASSET_ID,
            profiles=(
                "standard",
                "thumb",
            ),
            verify=True,
        )
    )
    STUDY_CLEAN_ASSET_PASSED = (
        str(
            STUDY_CLEAN_ASSET_RECORD[
                "source_row_id"
            ]
        ) == "p001"
        and STUDY_CLEAN_RENDITION_PATH
        is not None
        and STUDY_CLEAN_RENDITION_PATH.is_file()
    )

    STUDY_D02_SELECTION = (
        PACKAGE.validate_selection(
            {
                "room_id": (
                    "focused_portrait_review"
                ),
                "return_room": "study_design",
            }
        )
    )
    EXPECTED_STUDY_D02_SELECTION = {
        "room_id": "focused_portrait_review",
        "return_room": "study_design",
    }
    OBSERVED_STUDY_D02_SELECTION = (
        STUDY_D02_SELECTION.as_identifiers()
    )

    STUDY_EVIDENCE_CHECKS = (
        (
            "room_partition",
            (
                "The Study Design partition contains "
                "exactly 13 approved records"
            ),
            EXPECTED_STUDY_DISPLAY_IDS,
            OBSERVED_STUDY_DISPLAY_IDS,
            (
                len(STUDY_ROOM) == 13
                and OBSERVED_STUDY_DISPLAY_IDS
                == EXPECTED_STUDY_DISPLAY_IDS
                and set(
                    STUDY_ROOM[
                        "status"
                    ].astype(str)
                ) == {"ok"}
            ),
        ),
        (
            "required_bindings",
            (
                "Every Study Design record has one "
                "available required binding"
            ),
            {
                "rows": 13,
                "requiredness": {"required"},
                "applicability": {"available"},
            },
            {
                "rows": len(STUDY_BINDINGS),
                "requiredness": set(
                    STUDY_BINDINGS[
                        "requiredness"
                    ].astype(str)
                ),
                "applicability": set(
                    STUDY_BINDINGS[
                        "applicability_state"
                    ].astype(str)
                ),
            },
            (
                len(STUDY_BINDINGS) == 13
                and set(
                    STUDY_BINDINGS[
                        "display_id"
                    ].astype(str)
                ) == EXPECTED_STUDY_DISPLAY_IDS
                and set(
                    STUDY_BINDINGS[
                        "requiredness"
                    ].astype(str)
                ) == {"required"}
                and set(
                    STUDY_BINDINGS[
                        "applicability_state"
                    ].astype(str)
                ) == {"available"}
            ),
        ),
        (
            "mapping_states",
            (
                "Study Design preserves its exact "
                "N34 mapping-state composition"
            ),
            EXPECTED_MAPPING_STATE_COUNTS,
            OBSERVED_MAPPING_STATE_COUNTS,
            (
                OBSERVED_MAPPING_STATE_COUNTS
                == EXPECTED_MAPPING_STATE_COUNTS
            ),
        ),
        (
            "opening_copy",
            (
                "The approved Study Design question, "
                "introduction and boundary are exact"
            ),
            EXPECTED_STUDY_OPENING,
            {
                key: STUDY_CATALOGUE.get(key)
                for key
                in EXPECTED_STUDY_OPENING
            },
            (
                len(STUDY_CATALOGUE_MATCHES)
                == 1
                and all(
                    STUDY_CATALOGUE.get(key)
                    == value
                    for key, value
                    in EXPECTED_STUDY_OPENING.items()
                )
            ),
        ),
        (
            "population_scope",
            (
                "The collection and D02 summary use "
                "exact Controlled-300 counts"
            ),
            EXPECTED_STUDY_POPULATION,
            OBSERVED_STUDY_POPULATION,
            (
                OBSERVED_STUDY_POPULATION
                == EXPECTED_STUDY_POPULATION
            ),
        ),
        (
            "decorative_shell",
            (
                "The Study Design shell is checksum-"
                "pinned ambience with live overlays"
            ),
            True,
            STUDY_SHELL_PASSED,
            STUDY_SHELL_PASSED,
        ),
        (
            "exact_opening_asset_copies",
            (
                "All five N04 opening conditions "
                "are exact byte copies"
            ),
            0,
            int(
                (
                    ~STUDY_OPENING_ASSET_AUDIT[
                        "passed"
                    ]
                ).sum()
            ),
            (
                STUDY_OPENING_ASSET_CONTRACT[
                    "painting_id"
                ] == "p001"
                and STUDY_OPENING_ASSET_CONTRACT[
                    "source_notebook_id"
                ] == "04"
                and STUDY_OPENING_ASSET_CONTRACT[
                    "evidence_classification"
                ] == "exact_byte_copy_not_generated"
                and len(
                    STUDY_OPENING_ASSET_AUDIT
                ) == 5
                and bool(
                    STUDY_OPENING_ASSET_AUDIT[
                        "passed"
                    ].all()
                )
            ),
        ),
        (
            "registered_p001_clean_asset",
            (
                "The fixed p001 clean anchor resolves "
                "to a verified local rendition"
            ),
            True,
            STUDY_CLEAN_ASSET_PASSED,
            STUDY_CLEAN_ASSET_PASSED,
        ),
        (
            "shared_d02_child_route",
            (
                "The portrait plaque resolves to the "
                "single shared D02 child route"
            ),
            EXPECTED_STUDY_D02_SELECTION,
            OBSERVED_STUDY_D02_SELECTION,
            (
                OBSERVED_STUDY_D02_SELECTION
                == EXPECTED_STUDY_D02_SELECTION
            ),
        ),
    )

    STUDY_EVIDENCE_RECORDS = [
        {
            "check_id": check_id,
            "check_description": description,
            "severity": "blocking",
            "expected": expected,
            "observed": observed,
            "passed": bool(passed),
            "details": (
                ""
                if passed
                else (
                    "The immutable Study Design "
                    "evidence contract drifted."
                )
            ),
        }
        for (
            check_id,
            description,
            expected,
            observed,
            passed,
        ) in STUDY_EVIDENCE_CHECKS
    ]

    replace_validation_stage(
        "batch_3_study_evidence",
        STUDY_EVIDENCE_RECORDS,
    )
    VALIDATION.raise_for_blocking()

    STUDY_ROOM_VIEW = STUDY_ROOM[
        [
            "display_id",
            "record_type",
            "applicability_state",
            "interpretation",
        ]
    ].copy()

    display(STUDY_ROOM_VIEW)
    display(STUDY_OPENING_ASSET_AUDIT)

    print("Study Design evidence contract passed.")
    print("Room records:", len(STUDY_ROOM))
    print("Required bindings:", len(STUDY_BINDINGS))
    print(
        "Exact N04 copies:",
        int(
            STUDY_OPENING_ASSET_AUDIT[
                "passed"
            ].sum()
        ),
    )
    print(
        "D02 route:",
        OBSERVED_STUDY_D02_SELECTION,
    )
record_room_stage("batch_3_evidence_network", [room_check("network", [], STUDY_EVIDENCE_NETWORK)])


,display_id,record_type,applicability_state,interpretation
0,study.anchor.clean,metric,available,Selected clean reference
1,study.core.conditions,metric,available,"Unchanged control, scratch, small loss, large ..."
2,study.branch.damage_size,metric,available,Seven damage-size levels
3,study.branch.mask_placement,metric,available,Five valid mask variants for three family-area...
4,study.branch.other_changes,metric,available,Procedural degradation specimens
5,study.routing.status,selector,available,Suitable for restoration testing / studied as ...
6,study.scope.catalogues,selector,available,"300, five conditions, 3,425 and 2,620"
7,study.collection.drawers,selector,available,Five category drawers
8,study.notes.selection,metric,available,Selection and metadata drawer
9,study.notes.preprocessing,metric,available,Preparation drawer


,condition_id,deployed_path,source_path,expected_sha256,deployed_sha256,source_sha256,passed
0,zero_control,streamlit_assets/evidence/study_design/p001_ze...,outputs/04_canonical_damaged_image_generation/...,b5852f0337c8342a1a70268f9e9f682534177581d23871...,b5852f0337c8342a1a70268f9e9f682534177581d23871...,b5852f0337c8342a1a70268f9e9f682534177581d23871...,True
1,scratch_thin,streamlit_assets/evidence/study_design/p001_sc...,outputs/04_canonical_damaged_image_generation/...,aa28fe11ad325c143cb952d46883b2cbb1c1d5627e1f49...,aa28fe11ad325c143cb952d46883b2cbb1c1d5627e1f49...,aa28fe11ad325c143cb952d46883b2cbb1c1d5627e1f49...,True
2,loss_small,streamlit_assets/evidence/study_design/p001_lo...,outputs/04_canonical_damaged_image_generation/...,7683f8017f2d0f846e07ac5d4746564a233f6812056500...,7683f8017f2d0f846e07ac5d4746564a233f6812056500...,7683f8017f2d0f846e07ac5d4746564a233f6812056500...,True
3,loss_large,streamlit_assets/evidence/study_design/p001_lo...,outputs/04_canonical_damaged_image_generation/...,95c2dca591d41cd5d5462a5f8d52e815a9d907698128f1...,95c2dca591d41cd5d5462a5f8d52e815a9d907698128f1...,95c2dca591d41cd5d5462a5f8d52e815a9d907698128f1...,True
4,mixed_damage,streamlit_assets/evidence/study_design/p001_mi...,outputs/04_canonical_damaged_image_generation/...,4272fe588bb1af895fc4da909794688fcb105353051720...,4272fe588bb1af895fc4da909794688fcb105353051720...,4272fe588bb1af895fc4da909794688fcb105353051720...,True


Study Design evidence contract passed.
Room records: 13
Required bindings: 13
Exact N04 copies: 5
D02 route: {'room_id': 'focused_portrait_review', 'return_room': 'study_design'}


,check_id,passed,expected,observed
0,network,True,[],[]


In [13]:
drop_validation_stages(('batch_3_study_relationships', 'batch_3_streamlit_fragment_smoke', 'batch_3_final_gate', 'batch_4_', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_3_study_evidence', 'batch_3_evidence_network'])
with n35.offline() as STUDY_RELATIONSHIP_NETWORK:
    EXPECTED_STUDY_CATEGORY_LABELS = {
        "abstraction_surrealism": (
            "Abstraction / Surrealism"
        ),
        "architecture_structured": (
            "Architecture / Structured"
        ),
        "high_texture_brushwork": (
            "High Texture / Brushwork"
        ),
        "landscape_natural": (
            "Landscape / Natural"
        ),
        "portrait_figure": "Portrait / Figure",
    }

    EXPECTED_STUDY_ANCHORS = {
        "abstraction_surrealism": {
            "painting_id": "p039",
            "title": "Painting with Troika",
            "display_title": (
                "Painting with Troika"
            ),
            "artist": "Wassily Kandinsky",
            "clean_sha256": (
                "adc7c740921e37111c263492b14ccf28e"
                "00ec71749fd54ff92a6a8b1440b58f2"
            ),
            "content_box": (0, 102, 768, 666),
            "matte_rgb": (137, 113, 93),
            "frame_fit": "slice",
        },
        "architecture_structured": {
            "painting_id": "p026",
            "title": (
                "View of a Village along a River"
            ),
            "display_title": (
                "Village along a River"
            ),
            "artist": (
                "Attributed to Jan Brueghel I"
            ),
            "clean_sha256": (
                "48819acb8a3147c0230f636c58eaaa127"
                "b2bdde3b71bb7e4f101b4e2ac97b3ee"
            ),
            "content_box": (0, 119, 768, 648),
            "matte_rgb": (86, 111, 93),
            "frame_fit": "slice",
        },
        "high_texture_brushwork": {
            "painting_id": "p043",
            "title": "The Card Players",
            "display_title": "The Card Players",
            "artist": "Paul Cézanne",
            "clean_sha256": (
                "e18e79fb0a876169e1a315935be5e67a"
                "403e0c089c60841a2f34e1b6ba3b3fcd"
            ),
            "content_box": (0, 78, 768, 689),
            "matte_rgb": (82, 79, 73),
            "frame_fit": "slice",
        },
        "landscape_natural": {
            "painting_id": "p018",
            "title": (
                "Classical Landscape with Figures"
            ),
            "display_title": (
                "Landscape with Figures"
            ),
            "artist": "Henri Mauperché",
            "clean_sha256": (
                "3e223322b3ca2c6f52ebe54a266991be"
                "897dd8d6c670ad52393f3262e15574ac"
            ),
            "content_box": (0, 130, 768, 637),
            "matte_rgb": (90, 74, 44),
            "frame_fit": "slice",
        },
        "portrait_figure": {
            "painting_id": "p001",
            "title": "Juan de Pareja",
            "display_title": "Juan de Pareja",
            "artist": "Diego Velázquez",
            "clean_sha256": (
                "b5852f0337c8342a1a70268f9e9f682"
                "534177581d238713f67bf28e0033482a2"
            ),
            "content_box": (52, 0, 715, 768),
            "matte_rgb": (33, 28, 20),
            "frame_fit": "slice",
        },
    }

    EXPECTED_EXPERIMENT_COUNTS = {
        "canonical_missing_region": 5,
        "damage_size_sensitivity": 7,
        "mask_robustness": 15,
        "synthetic_degradation": 33,
    }
    EXPECTED_STUDY_BRANCH_CONTRACT = {
        "core": {
            "default_option": "mixed_damage",
            "visible_options": 6,
        },
        "damage_size": {
            "default_option": "size_10pct",
            "visible_options": 7,
        },
        "mask_placement": {
            "default_option": "variant_01",
            "visible_options": 5,
        },
        "other_changes": {
            "default_option": "dirt_dust__mild",
            "visible_options": 4,
        },
    }
    EXPECTED_OTHER_OPTION_IDS = (
        "dirt_dust__mild",
        "partial_transparency__moderate",
        "water_stain__severe",
        "water_stain_dirt__moderate",
    )
    OBSERVED_OTHER_OPTION_IDS = tuple(
        option_id
        for option_id, _label, _severity
        in APP_STUDY_OTHER_CHANGE_OPTIONS
    )
    EXPECTED_STUDY_NOTE_IDS = {
        "selection",
        "preprocessing",
        "damage",
        "focused_tests",
        "non_restoration",
        "portrait_audit",
    }

    APPROVED_STUDY_IMAGE_ROOTS = tuple(
        (
            PROJECT_ROOT
            / relative_path
        ).resolve()
        for relative_path in (
            (
                "outputs/02_image_preprocessing/"
                "images/clean"
            ),
            (
                "outputs/04_canonical_damaged_"
                "image_generation/images/damaged"
            ),
            (
                "outputs/05_damage_size_sensitivity_"
                "dataset_generation/images/damaged"
            ),
            (
                "outputs/06_mask_robustness_dataset_"
                "generation/images/damaged"
            ),
            (
                "outputs/07_synthetic_degradation_"
                "dataset_generation/images/degraded"
            ),
        )
    )

    STUDY_REPRESENTATIVE_RECORDS = []

    for category_id, anchor in (
        EXPECTED_STUDY_ANCHORS.items()
    ):
        painting_id = anchor["painting_id"]
        payload = PACKAGE.load_painting(
            painting_id
        )
        painting = payload["painting"]
        cases = payload["cases"]
        case_by_id = {
            str(row["case_id"]): row
            for row in cases
        }

        experiment_counts = {}
        for row in cases:
            experiment_id = str(
                row["experiment_id"]
            )
            experiment_counts[experiment_id] = (
                experiment_counts.get(
                    experiment_id,
                    0,
                )
                + 1
            )

        visible_case_ids = [
            (
                f"canonical__{painting_id}__"
                f"{condition_id}"
            )
            for condition_id in (
                "zero_control",
                "scratch_thin",
                "loss_small",
                "loss_large",
                "mixed_damage",
            )
        ]
        visible_case_ids.extend(
            (
                f"damage_size__{painting_id}__"
                "loss_large__"
                f"size_{percent:02d}pct"
            )
            for percent
            in (2, 4, 6, 8, 10, 15, 20)
        )
        visible_case_ids.extend(
            (
                "mask_robustness__"
                f"{painting_id}__loss_large__"
                "target_12p5pct__"
                f"variant_{number:02d}"
            )
            for number in range(1, 6)
        )
        visible_case_ids.extend(
            (
                "synthetic_degradation__"
                f"{painting_id}__{option_id}"
            )
            for option_id
            in EXPECTED_OTHER_OPTION_IDS
        )

        visible_paths = [
            (
                PROJECT_ROOT
                / str(
                    case_by_id[
                        case_id
                    ]["input_image_path"]
                )
            ).resolve()
            for case_id in visible_case_ids
            if case_id in case_by_id
        ]
        clean_path = (
            PROJECT_ROOT
            / str(
                painting["clean_image_path"]
            )
        ).resolve()
        clean_sha256 = hashlib.sha256(
            clean_path.read_bytes()
        ).hexdigest()
        other_eligible = all(
            bool(
                case_by_id[
                    (
                        "synthetic_degradation__"
                        f"{painting_id}__{option_id}"
                    )
                ]["four_method_eligible"]
            )
            for option_id
            in EXPECTED_OTHER_OPTION_IDS
        )

        passed = (
            APP_STUDY_CATEGORY_ANCHORS.get(
                category_id
            ) == anchor
            and str(
                painting["painting_id"]
            ) == painting_id
            and str(
                painting["category"]
            ) == category_id
            and str(
                painting["title"]
            ) == anchor["title"]
            and str(
                painting["artist"]
            ).casefold() == str(
                anchor["artist"]
            ).casefold()
            and int(
                painting["case_count"]
            ) == 60
            and len(cases) == 60
            and len(case_by_id) == 60
            and experiment_counts
            == EXPECTED_EXPERIMENT_COUNTS
            and len(visible_case_ids) == 21
            and set(
                visible_case_ids
            ).issubset(case_by_id)
            and len(visible_paths) == 21
            and all(
                path.is_file()
                for path in visible_paths
            )
            and all(
                any(
                    path.is_relative_to(root)
                    for root
                    in APPROVED_STUDY_IMAGE_ROOTS
                )
                for path in visible_paths
            )
            and clean_path.is_file()
            and clean_path.is_relative_to(
                APPROVED_STUDY_IMAGE_ROOTS[0]
            )
            and clean_sha256
            == anchor["clean_sha256"]
            and other_eligible
        )

        STUDY_REPRESENTATIVE_RECORDS.append(
            {
                "category_id": category_id,
                "painting_id": painting_id,
                "case_count": len(cases),
                "visible_case_count": len(
                    visible_case_ids
                ),
                "other_options_eligible": (
                    other_eligible
                ),
                "clean_sha256": clean_sha256,
                "passed": passed,
            }
        )

    STUDY_REPRESENTATIVE_AUDIT = (
        pd.DataFrame(
            STUDY_REPRESENTATIVE_RECORDS
        )
    )

    OBSERVED_STUDY_BRANCH_CONTRACT = {
        path_id: {
            "default_option": (
                APP_STUDY_PATH_DEFAULT_OPTIONS[
                    path_id
                ]
            ),
            "visible_options": {
                "core": 6,
                "damage_size": 7,
                "mask_placement": 5,
                "other_changes": len(
                    OBSERVED_OTHER_OPTION_IDS
                ),
            }[path_id],
        }
        for path_id
        in EXPECTED_STUDY_BRANCH_CONTRACT
    }

    STUDY_RELATIONSHIP_RECORDS = [
        {
            "check_id": (
                "representative__"
                f"{row.category_id}"
            ),
            "check_description": (
                "The fixed category representative has "
                "60 cases and all 21 visible exact routes: "
                f"{row.category_id}"
            ),
            "severity": "blocking",
            "expected": {
                "painting_id": row.painting_id,
                "case_count": 60,
                "visible_case_count": 21,
                "other_options_eligible": True,
            },
            "observed": {
                "painting_id": row.painting_id,
                "case_count": int(row.case_count),
                "visible_case_count": int(
                    row.visible_case_count
                ),
                "other_options_eligible": bool(
                    row.other_options_eligible
                ),
            },
            "passed": bool(row.passed),
            "details": (
                ""
                if row.passed
                else (
                    "A fixed Study Design representative "
                    "or exact case route drifted."
                )
            ),
        }
        for row in (
            STUDY_REPRESENTATIVE_AUDIT
            .itertuples(index=False)
        )
    ]

    STUDY_RELATIONSHIP_RECORDS.extend(
        [
            {
                "check_id": "category_contract",
                "check_description": (
                    "Five category drawers preserve "
                    "their exact labels and anchors"
                ),
                "severity": "blocking",
                "expected": {
                    "labels": (
                        EXPECTED_STUDY_CATEGORY_LABELS
                    ),
                    "anchors": (
                        EXPECTED_STUDY_ANCHORS
                    ),
                },
                "observed": {
                    "labels": (
                        APP_STUDY_CATEGORY_LABELS
                    ),
                    "anchors": (
                        APP_STUDY_CATEGORY_ANCHORS
                    ),
                },
                "passed": (
                    APP_STUDY_CATEGORY_LABELS
                    == EXPECTED_STUDY_CATEGORY_LABELS
                    and APP_STUDY_CATEGORY_ANCHORS
                    == EXPECTED_STUDY_ANCHORS
                ),
                "details": "",
            },
            {
                "check_id": "parallel_branch_contract",
                "check_description": (
                    "All four parallel paths preserve "
                    "their option counts and defaults"
                ),
                "severity": "blocking",
                "expected": (
                    EXPECTED_STUDY_BRANCH_CONTRACT
                ),
                "observed": (
                    OBSERVED_STUDY_BRANCH_CONTRACT
                ),
                "passed": (
                    OBSERVED_STUDY_BRANCH_CONTRACT
                    == EXPECTED_STUDY_BRANCH_CONTRACT
                ),
                "details": "",
            },
            {
                "check_id": "eligible_other_changes",
                "check_description": (
                    "Only the four policy-approved "
                    "other-change options are exposed"
                ),
                "severity": "blocking",
                "expected": EXPECTED_OTHER_OPTION_IDS,
                "observed": OBSERVED_OTHER_OPTION_IDS,
                "passed": (
                    OBSERVED_OTHER_OPTION_IDS
                    == EXPECTED_OTHER_OPTION_IDS
                ),
                "details": "",
            },
            {
                "check_id": "study_note_drawers",
                "check_description": (
                    "The six approved Study Notes "
                    "drawers are present"
                ),
                "severity": "blocking",
                "expected": EXPECTED_STUDY_NOTE_IDS,
                "observed": set(
                    APP_STUDY_NOTE_COPY
                ),
                "passed": (
                    set(APP_STUDY_NOTE_COPY)
                    == EXPECTED_STUDY_NOTE_IDS
                    and len(
                        APP_STUDY_NOTE_COPY
                    ) == 6
                ),
                "details": "",
            },
        ]
    )

    replace_validation_stage(
        "batch_3_study_relationships",
        STUDY_RELATIONSHIP_RECORDS,
    )
    VALIDATION.raise_for_blocking()

    display(STUDY_REPRESENTATIVE_AUDIT)

    print("Study Design relationships passed.")
    print(
        "Category representatives:",
        int(
            STUDY_REPRESENTATIVE_AUDIT[
                "passed"
            ].sum()
        ),
    )
    print(
        "Visible exact cases:",
        int(
            STUDY_REPRESENTATIVE_AUDIT[
                "visible_case_count"
            ].sum()
        ),
    )
    print(
        "Parallel paths:",
        len(EXPECTED_STUDY_BRANCH_CONTRACT),
    )
    print(
        "Study Notes:",
        len(APP_STUDY_NOTE_COPY),
    )
record_room_stage("batch_3_relationship_network", [room_check("network", [], STUDY_RELATIONSHIP_NETWORK)])


,category_id,painting_id,case_count,visible_case_count,other_options_eligible,clean_sha256,passed
0,abstraction_surrealism,p039,60,21,True,adc7c740921e37111c263492b14ccf28e00ec71749fd54...,True
1,architecture_structured,p026,60,21,True,48819acb8a3147c0230f636c58eaaa127b2bdde3b71bb7...,True
2,high_texture_brushwork,p043,60,21,True,e18e79fb0a876169e1a315935be5e67a403e0c089c6084...,True
3,landscape_natural,p018,60,21,True,3e223322b3ca2c6f52ebe54a266991be897dd8d6c670ad...,True
4,portrait_figure,p001,60,21,True,b5852f0337c8342a1a70268f9e9f682534177581d23871...,True


Study Design relationships passed.
Category representatives: 5
Visible exact cases: 105
Parallel paths: 4
Study Notes: 6


,check_id,passed,expected,observed
0,network,True,[],[]


In [14]:
drop_validation_stages(('batch_3_streamlit_fragment_smoke', 'batch_3_final_gate', 'batch_4_', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_3_study_relationships', 'batch_3_relationship_network'])
from streamlit.testing.v1 import (
    AppTest,
)


def apptest_query_scalar(app_test, key):
    value = app_test.query_params.get(
        key,
        "",
    )
    if isinstance(value, (list, tuple)):
        value = (
            value[-1]
            if value
            else ""
        )
    return str(value)


def apptest_markdown(app_test):
    return n35.app_html(app_test)


STUDY_PATH_RUNTIME_EXPECTATIONS = {
    "core": {
        "option_id": "mixed_damage",
        "specimen_count": 6,
        "case_id": (
            "canonical__p001__mixed_damage"
        ),
    },
    "damage_size": {
        "option_id": "size_10pct",
        "specimen_count": 7,
        "case_id": (
            "damage_size__p001__loss_large__"
            "size_10pct"
        ),
    },
    "mask_placement": {
        "option_id": "variant_01",
        "specimen_count": 5,
        "case_id": (
            "mask_robustness__p001__loss_large__"
            "target_12p5pct__variant_01"
        ),
    },
    "other_changes": {
        "option_id": "dirt_dust__mild",
        "specimen_count": 4,
        "case_id": (
            "synthetic_degradation__p001__"
            "dirt_dust__mild"
        ),
    },
}

STUDY_CATEGORY_RUNTIME_EXPECTATIONS = {
    "abstraction_surrealism": "p039",
    "architecture_structured": "p026",
    "high_texture_brushwork": "p043",
    "landscape_natural": "p018",
    "portrait_figure": "p001",
}

STUDY_OTHER_RUNTIME_OPTIONS = (
    "dirt_dust__mild",
    "partial_transparency__moderate",
    "water_stain__severe",
    "water_stain_dirt__moderate",
)

STUDY_RUNTIME_RECORDS = []


def capture_study_runtime_state(
    app_test,
    *,
    check_id,
    category_id,
    painting_id,
    path_id,
    option_id,
    specimen_count,
    case_id,
):
    markdown = apptest_markdown(app_test)
    exceptions = [
        str(exception.value)
        for exception in list(app_test.exception) + list(app_test.error)
    ]
    observed = {
        "room": apptest_query_scalar(
            app_test,
            "room",
        ),
        "study_category": apptest_query_scalar(
            app_test,
            "study_category",
        ),
        "study_path": apptest_query_scalar(
            app_test,
            "study_path",
        ),
        "study_option": apptest_query_scalar(
            app_test,
            "study_option",
        ),
        "scene_category": (
            f'data-study-category="{category_id}"'
            in markdown
        ),
        "scene_painting": (
            f'data-painting-id="{painting_id}"'
            in markdown
        ),
        "scene_path": (
            f'data-study-path="{path_id}"'
            in markdown
        ),
        "scene_option": (
            f'data-study-option="{option_id}"'
            in markdown
        ),
        "scene_case": (
            f'data-case-id="{case_id}"'
            in markdown
        ),
        "specimen_count": (
            f'data-specimen-count="{specimen_count}"'
            in markdown
        ),
        "native_bridge_buttons": len(
            app_test.button
        ),
        "exceptions": exceptions,
    }
    expected = {
        "room": "study_design",
        "study_category": category_id,
        "study_path": path_id,
        "study_option": option_id,
        "scene_category": True,
        "scene_painting": True,
        "scene_path": True,
        "scene_option": True,
        "scene_case": True,
        "specimen_count": True,
        "native_bridge_buttons": (
            9 + specimen_count
        ),
        "exceptions": [],
    }
    passed = observed == expected
    STUDY_RUNTIME_RECORDS.append(
        {
            "check_id": check_id,
            "check_description": (
                "The fragment bridge preserves "
                "exact Study Design query and "
                "evidence identity"
            ),
            "severity": "blocking",
            "expected": expected,
            "observed": observed,
            "passed": passed,
            "details": (
                ""
                if passed
                else (
                    "Study Design query/fragment "
                    "state drifted."
                )
            ),
        }
    )


with n35.offline() as STUDY_NETWORK_ATTEMPTS:
    STUDY_APP_TEST = AppTest.from_file(
        str(APP_PATH),
        default_timeout=45,
    )
    STUDY_APP_TEST.query_params[
        "room"
    ] = "study_design"
    STUDY_APP_TEST.query_params[
        "study_category"
    ] = "portrait_figure"
    STUDY_APP_TEST.query_params[
        "study_path"
    ] = "core"
    STUDY_APP_TEST.query_params[
        "study_option"
    ] = "mixed_damage"
    STUDY_APP_TEST.run()

    capture_study_runtime_state(
        STUDY_APP_TEST,
        check_id="initial_core_state",
        category_id="portrait_figure",
        painting_id="p001",
        path_id="core",
        **STUDY_PATH_RUNTIME_EXPECTATIONS[
            "core"
        ],
    )

    for path_id in (
        "damage_size",
        "mask_placement",
        "other_changes",
    ):
        STUDY_APP_TEST.button(
            key=(
                "study_bridge_path_"
                f"{path_id}"
            )
        ).click().run()
        capture_study_runtime_state(
            STUDY_APP_TEST,
            check_id=f"path__{path_id}",
            category_id="portrait_figure",
            painting_id="p001",
            path_id=path_id,
            **STUDY_PATH_RUNTIME_EXPECTATIONS[
                path_id
            ],
        )

    for category_id, painting_id in (
        STUDY_CATEGORY_RUNTIME_EXPECTATIONS
        .items()
    ):
        STUDY_APP_TEST.button(
            key=(
                "study_bridge_category_"
                f"{category_id}"
            )
        ).click().run()
        capture_study_runtime_state(
            STUDY_APP_TEST,
            check_id=f"category__{category_id}",
            category_id=category_id,
            painting_id=painting_id,
            path_id="other_changes",
            option_id="dirt_dust__mild",
            specimen_count=4,
            case_id=(
                "synthetic_degradation__"
                f"{painting_id}__dirt_dust__mild"
            ),
        )

    if apptest_query_scalar(
        STUDY_APP_TEST,
        "study_category",
    ) != "portrait_figure":
        STUDY_APP_TEST.button(
            key=(
                "study_bridge_category_"
                "portrait_figure"
            )
        ).click().run()

    for option_id in (
        STUDY_OTHER_RUNTIME_OPTIONS
    ):
        STUDY_APP_TEST.button(
            key=(
                "study_bridge_option_"
                f"{option_id}"
            )
        ).click().run()
        capture_study_runtime_state(
            STUDY_APP_TEST,
            check_id=(
                "other_option__"
                f"{option_id}"
            ),
            category_id="portrait_figure",
            painting_id="p001",
            path_id="other_changes",
            option_id=option_id,
            specimen_count=4,
            case_id=(
                "synthetic_degradation__p001__"
                f"{option_id}"
            ),
        )

    FINAL_STUDY_MARKDOWN = (
        apptest_markdown(STUDY_APP_TEST)
    )
    STUDY_SEMANTIC_EXPECTED = {
        "parallel_path_links": 4,
        "category_drawers": 5,
        "study_note_buttons": 6,
        "popover_notes": 6,
        "focused_review_route": True,
        "d02_reuse_boundary": True,
        "remote_image_sources": 0,
        "scope_300": True,
        "scope_3425": True,
        "scope_2620": True,
    }
    STUDY_SEMANTIC_OBSERVED = {
        "parallel_path_links": (
            FINAL_STUDY_MARKDOWN.count(
                '<a class="study-path'
            )
        ),
        "category_drawers": (
            FINAL_STUDY_MARKDOWN.count(
                '<a class="collection-drawer'
            )
        ),
        "study_note_buttons": (
            FINAL_STUDY_MARKDOWN.count(
                (
                    '<button class="study-note" '
                    'type="button"'
                )
            )
        ),
        "popover_notes": (
            FINAL_STUDY_MARKDOWN.count(
                'popover="auto"'
            )
        ),
        "focused_review_route": (
            (
                "?room=focused_portrait_review&amp;"
                "return_room=study_design"
            )
            in FINAL_STUDY_MARKDOWN
        ),
        "d02_reuse_boundary": (
            "Reuses existing results."
            in FINAL_STUDY_MARKDOWN
        ),
        "remote_image_sources": (
            FINAL_STUDY_MARKDOWN.count(
                'src="http'
            )
        ),
        "scope_300": (
            ">300</strong><span>paintings"
            in FINAL_STUDY_MARKDOWN
        ),
        "scope_3425": (
            (
                ">3,425</strong><span>"
                "registered cases"
            )
            in FINAL_STUDY_MARKDOWN
        ),
        "scope_2620": (
            (
                ">2,620</strong><span>"
                "method-eligible cases"
            )
            in FINAL_STUDY_MARKDOWN
        ),
    }
    semantic_passed = (
        STUDY_SEMANTIC_OBSERVED
        == STUDY_SEMANTIC_EXPECTED
    )
    STUDY_RUNTIME_RECORDS.append(
        {
            "check_id": "semantic_room_contract",
            "check_description": (
                "The rendered Study Design room exposes "
                "all approved controls, notes, scope "
                "facts and the single D02 route"
            ),
            "severity": "blocking",
            "expected": STUDY_SEMANTIC_EXPECTED,
            "observed": STUDY_SEMANTIC_OBSERVED,
            "passed": semantic_passed,
            "details": (
                ""
                if semantic_passed
                else (
                    "The rendered semantic room "
                    "contract drifted."
                )
            ),
        }
    )


STUDY_RUNTIME_RECORDS.append(room_check("outbound_network_attempts", [], STUDY_NETWORK_ATTEMPTS))

replace_validation_stage(
    "batch_3_streamlit_fragment_smoke",
    STUDY_RUNTIME_RECORDS,
)
VALIDATION.raise_for_blocking()

STUDY_RUNTIME_AUDIT = pd.DataFrame(
    [
        {
            "check_id": row["check_id"],
            "passed": row["passed"],
        }
        for row
        in STUDY_RUNTIME_RECORDS
    ]
)

display(STUDY_RUNTIME_AUDIT)

print(
    "Study Design fragment/query smoke passed."
)
print(
    "Runtime states:",
    len(STUDY_RUNTIME_RECORDS) - 1,
)
print(
    "Final query:",
    dict(STUDY_APP_TEST.query_params),
)
print(
    "Streamlit exceptions:",
    len(STUDY_APP_TEST.exception),
)
print(
    "Fresh browser same-document behavior: NOT checked; "
    "AppTest validates the native "
    "fragment callbacks, not browser JavaScript."
)


2026-10-01 09:19:31.717 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 09:19:33.669 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:19:34.972 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:19:36.047 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:19:36.962 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:19:38.016 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:19:39.075 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:19:40.135 Please rep

,check_id,passed
0,initial_core_state,True
1,path__damage_size,True
2,path__mask_placement,True
3,path__other_changes,True
4,category__abstraction_surrealism,True
5,category__architecture_structured,True
6,category__high_texture_brushwork,True
7,category__landscape_natural,True
8,category__portrait_figure,True
9,other_option__dirt_dust__mild,True


Study Design fragment/query smoke passed.
Runtime states: 14
Final query: {'room': ['study_design'], 'study_path': ['other_changes'], 'study_category': ['portrait_figure'], 'study_option': ['water_stain_dirt__moderate']}
Streamlit exceptions: 0
Fresh browser same-document behavior: NOT checked; AppTest validates the native fragment callbacks, not browser JavaScript.


In [15]:
drop_validation_stages(('batch_3_final_gate', 'batch_4_', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

finish_room_batch(3, {
            "batch_3_application_source", "batch_3_study_evidence", "batch_3_study_relationships",
            "batch_3_evidence_network", "batch_3_relationship_network", "batch_3_streamlit_fragment_smoke"})
print("Continue with the replacement Batch 4 below, not historical pilot cells.")


,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_3_application_source,2,2
batch_3_evidence_network,1,1
batch_3_final_gate,2,2
batch_3_relationship_network,1,1
batch_3_streamlit_fragment_smoke,15,15
batch_3_study_evidence,9,9
batch_3_study_relationships,9,9


Batch 3: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.
Continue with the replacement Batch 4 below, not historical pilot cells.


## Batch 4 — Metric Framework: saved diagnostic companions

Validate the frozen teaching room: one canonical mixed-damage LaMa candidate per painting,
7 lenses × 7 regions, 37 allowed pairs and 12 prohibited pairs. These spatial companions
are inspection diagnostics, not replacements for frozen benchmark scalar metrics.

Check all 300 manifests for identity, policy and local path availability. Rehash saved
sources/maps/supports for five category representatives; use focused arithmetic and
corruption-rejection tests on the saved p018 exhibit. No diagnostic maps are regenerated.
Historical Controlled-50 dashboard-metrics tests are not used as evidence for this release.

AppTest checks the opening, native painting selector and Surprise me callback. It cannot
operate the browser-only draggable lens or certify Firefox, viewport layout or hosted assets.
Those remain Batch 10 gates. All four code cells are required; no canonical outputs are written.


In [18]:
drop_validation_stages(('batch_4_', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))


start_room_batch(4)
from restoration_eval.metric_inspection import public_contract, LENSES, REGIONS, VERSION
from restoration_eval.metric_inspection_view import load_inspection
METRIC_POLICY = public_contract()
with n35.offline() as METRIC_SOURCE_NETWORK:
    METRIC_ROOM = PACKAGE.load_room("metric_framework")
    METRIC_BINDINGS = PACKAGE.bindings_for(room_id="metric_framework")
metric_shell = CONFIG["presentation"]["decorative_shells"]["metric_framework"]
metric_shell_path = n35.safe_path(PROJECT_ROOT, metric_shell["path"])
record_room_stage("batch_4_source_contract", [
    room_check("room_rows", 12, len(METRIC_ROOM)),
    room_check("unique_room_ids", 12, METRIC_ROOM.display_id.nunique()),
    room_check("binding_ids", set(METRIC_ROOM.display_id), set(METRIC_BINDINGS.display_id)),
    room_check("binding_count", 21, len(METRIC_BINDINGS)),
    room_check("local_package_network", [], METRIC_SOURCE_NETWORK),
    room_check("shell_checksum", metric_shell["sha256"], n35.digest(metric_shell_path)),
    room_check("shell_is_not_evidence", False, metric_shell["scientific_content_allowed"]),
    room_check("lens_region_pairs", (7, 7, 49, 37, 12),
        (len(LENSES), len(REGIONS), len(METRIC_POLICY),
         sum(p["allowed"] for p in METRIC_POLICY.values()),
         sum(not p["allowed"] for p in METRIC_POLICY.values()))),
    test_record("metric"),
])


Running metric local contract tests; no inference or artifact generation.
test_arrakis_art_and_surprise_tooltip_cleanup_are_decorative (n35_checks_test_metric_inspection.InspectionControllerTests.test_arrakis_art_and_surprise_tooltip_cleanup_are_decorative) ... ok
test_cabinet_icons_and_conservation_study_are_scoped (n35_checks_test_metric_inspection.InspectionControllerTests.test_cabinet_icons_and_conservation_study_are_scoped) ... ok
test_cabinet_paper_angles_and_label_offset_are_scoped (n35_checks_test_metric_inspection.InspectionControllerTests.test_cabinet_paper_angles_and_label_offset_are_scoped) ... ok
test_cabinet_reuses_shell_drawers_and_reference_style_icons (n35_checks_test_metric_inspection.InspectionControllerTests.test_cabinet_reuses_shell_drawers_and_reference_style_icons) ... ok
test_controller_has_fail_closed_state_and_patch_geometry (n35_checks_test_metric_inspection.InspectionControllerTests.test_controller_has_fail_closed_state_and_patch_geometry) ... ok
test_dune_b

,check_id,passed,expected,observed
0,room_rows,True,12,12
1,unique_room_ids,True,12,12
2,binding_ids,True,"{metric.lens.local, metric.quality.anchors, me...","{metric.lens.local, metric.quality.anchors, me..."
3,binding_count,True,21,21
4,local_package_network,True,[],[]
5,shell_checksum,True,a5182c9f6eb136b1c4be215d5477a40a8eeb0b33e6f156...,a5182c9f6eb136b1c4be215d5477a40a8eeb0b33e6f156...
6,shell_is_not_evidence,True,False,False
7,lens_region_pairs,True,"(7, 7, 49, 37, 12)","(7, 7, 49, 37, 12)"
8,focused_tests,True,True,True


In [19]:
drop_validation_stages(('batch_4_saved_evidence', 'batch_4_server_smoke', 'batch_4_final_gate', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_4_source_contract'])

# All 300 saved manifests: exact teaching identity and public pair policy.
# Full source/map checksums are sampled across the five category anchors below;
# this is not a claim that all corpus image bytes have been rehashed/downloaded.
METRIC_MANIFEST_AUDIT = []
with n35.offline() as METRIC_DATA_NETWORK:
    for pid in PACKAGE.painting_lookup.painting_id.astype(str):
        folder = PROJECT_ROOT / "streamlit_assets/evidence/metric_framework" / pid
        meta = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
        case_id = f"canonical__{pid}__mixed_damage"
        candidate_id = f"candidate__lama__{case_id}__c00"
        identity_ok = (meta["version"] == VERSION and meta["painting_id"] == pid
                       and meta["case_id"] == case_id and meta["candidate_id"] == candidate_id)
        policy_ok = (set(meta["pairs"]) == set(METRIC_POLICY)
                     and set(meta["regions"]) == set(REGIONS))
        for key, expected in METRIC_POLICY.items():
            actual = meta["pairs"].get(key, {})
            policy_ok &= all(actual.get(k) == expected[k]
                             for k in ("lens", "region", "canonical_region", "allowed", "role"))
            if expected["allowed"]:
                policy_ok &= (actual.get("recipe") == expected["recipe"]
                              and bool(actual.get("maps"))
                              and all(k in meta["assets"] for k in actual.get("maps", [])))
            else:
                policy_ok &= not actual.get("maps")
        path_ok = all(n35.safe_path(PROJECT_ROOT, a["path"]).is_file()
                      for collection in ("sources", "assets", "regions")
                      for a in meta[collection].values())
        METRIC_MANIFEST_AUDIT.append(dict(painting_id=pid, identity=identity_ok,
                                        policy=policy_ok, local_paths=path_ok))
    METRIC_SAMPLE_AUDIT = []
    for pid in ("p001", "p018", "p026", "p039", "p043"):
        meta = load_inspection(PROJECT_ROOT, pid)
        METRIC_SAMPLE_AUDIT.append(dict(painting_id=pid, sources=len(meta["sources"]),
            maps=len(meta["assets"]), supports=len(meta["regions"]),
            candidate_id=meta["candidate_id"], loader=meta["loader_version"]))

METRIC_MANIFEST_AUDIT = pd.DataFrame(METRIC_MANIFEST_AUDIT)
record_room_stage("batch_4_saved_evidence", [
    room_check("manifest_coverage", (300, 300),
        (len(METRIC_MANIFEST_AUDIT), METRIC_MANIFEST_AUDIT.painting_id.nunique())),
    room_check("exact_identities", 300, int(METRIC_MANIFEST_AUDIT.identity.sum())),
    room_check("complete_policy", 300, int(METRIC_MANIFEST_AUDIT.policy.sum())),
    room_check("local_registered_paths", 300, int(METRIC_MANIFEST_AUDIT.local_paths.sum())),
    room_check("five_checked_samples", 5, len(METRIC_SAMPLE_AUDIT)),
    room_check("outbound_network_attempts", [], METRIC_DATA_NETWORK),
])
display(pd.DataFrame(METRIC_SAMPLE_AUDIT))
print("Saved diagnostic companions checked; no metric producer, model download or inference ran.")


,check_id,passed,expected,observed
0,manifest_coverage,True,"(300, 300)","(300, 300)"
1,exact_identities,True,300,300
2,complete_policy,True,300,300
3,local_registered_paths,True,300,300
4,five_checked_samples,True,5,5
5,outbound_network_attempts,True,[],[]


,painting_id,sources,maps,supports,candidate_id,loader
0,p001,4,29,7,candidate__lama__canonical__p001__mixed_damage...,metric_view.v2
1,p018,4,29,7,candidate__lama__canonical__p018__mixed_damage...,metric_view.v2
2,p026,4,29,7,candidate__lama__canonical__p026__mixed_damage...,metric_view.v2
3,p039,4,29,7,candidate__lama__canonical__p039__mixed_damage...,metric_view.v2
4,p043,4,29,7,candidate__lama__canonical__p043__mixed_damage...,metric_view.v2


Saved diagnostic companions checked; no metric producer, model download or inference ran.


In [20]:
drop_validation_stages(('batch_4_server_smoke', 'batch_4_final_gate', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_4_saved_evidence'])

# A fresh in-process AppTest, not the user's running Streamlit process.
METRIC_SMOKE = n35.room_smoke(APP_PATH, {"room": "metric_framework", "metric_painting": "p018"})
METRIC_APP_TEST = METRIC_SMOKE["app"]
METRIC_RUNTIME_RECORDS = []

def capture_metric(check_id, pid, attempts):
    markup = n35.app_html(METRIC_APP_TEST)
    case_id = f"canonical__{pid}__mixed_damage"
    candidate_id = f"candidate__lama__{case_id}__c00"
    expected = {"errors": [], "network": [], "identity": True, "regions": 7,
                "lenses": 7, "selector_paintings": 300, "controller": True}
    observed = {"errors": n35.app_errors(METRIC_APP_TEST), "network": attempts,
        "identity": all(token in markup for token in (
            f'data-painting-id="{pid}"', f'data-case-id="{case_id}"',
            f'data-candidate-id="{candidate_id}"')),
        "regions": markup.count('class="metric-region"'),
        "lenses": markup.count('class="metric-lens-head"'),
        "selector_paintings": len(METRIC_APP_TEST.selectbox(key="metric_painting_selector").options),
        "controller": any("MuseumReadiness.watch" in str(e.proto.srcdoc)
                          for e in METRIC_APP_TEST.get("iframe"))}
    METRIC_RUNTIME_RECORDS.append(room_check(check_id, expected, observed))

capture_metric("opening_p018", "p018", METRIC_SMOKE["network_attempts"])
with n35.offline() as attempts:
    METRIC_APP_TEST.selectbox(key="metric_painting_selector").select("p001").run()
capture_metric("native_painting_callback", "p001", attempts)
painting_ids = tuple(PACKAGE.painting_lookup.painting_id.astype(str))
surprise_pid = painting_ids[(painting_ids.index("p001") + 37) % len(painting_ids)]
with n35.offline() as attempts:
    METRIC_APP_TEST.button(key="metric_surprise_button").click().run()
capture_metric("native_surprise_callback", surprise_pid, attempts)
record_room_stage("batch_4_server_smoke", METRIC_RUNTIME_RECORDS)
print("Three Python-rendered states verified. Lens dragging/region clicks require browser QA in Batch 10.")


2026-10-01 09:20:38.684 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 09:20:40.844 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:20:40.859 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:20:42.435 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:20:42.454 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:20:44.046 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:20:44.062 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


,check_id,passed,expected,observed
0,opening_p018,True,"{'errors': [], 'network': [], 'identity': True...","{'errors': [], 'network': [], 'identity': True..."
1,native_painting_callback,True,"{'errors': [], 'network': [], 'identity': True...","{'errors': [], 'network': [], 'identity': True..."
2,native_surprise_callback,True,"{'errors': [], 'network': [], 'identity': True...","{'errors': [], 'network': [], 'identity': True..."


Three Python-rendered states verified. Lens dragging/region clicks require browser QA in Batch 10.


In [21]:
drop_validation_stages(('batch_4_final_gate', 'batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

finish_room_batch(4, {
    "batch_4_source_contract", "batch_4_saved_evidence", "batch_4_server_smoke"})
print("Continue with replacement Batch 5 below.")


,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_4_final_gate,2,2
batch_4_saved_evidence,6,6
batch_4_server_smoke,3,3
batch_4_source_contract,9,9


Batch 4: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.
Continue with replacement Batch 5 below.


## Batch 5 — Model Gallery: exact saved comparisons

Validate four primary methods across 300 paintings / 2,620 cases, their exact candidate
and experiment identities, saved N13 measurements, empty-mask non-applicability and
case-scoped SDXL availability (24 completed / 35 scheduled, spanning 19 paintings).
SDXL is bounded evidence, not a fifth full-corpus ranking participant.

The seven-file Gallery visual freeze remains mandatory. Normalize only the two approved
non-visual transport substitutions from commit e88feea5 back to its original fingerprint;
separately verify the shared transport against the later museum-visit freeze. No other
changes are accepted. Its historical test asserting an old
whole-notebook SHA is incompatible with normal user notebook edits: replace only that
assertion with a current before/after cell-source and canonical-output snapshot (allowing
normal execution-count/output autosaves), and report
the replacement explicitly. Leave the frozen test file itself unchanged. Every scientific
Gallery test still runs; skipped tests and empty suites cannot produce a passing gate.

This room currently reads local originals and scans the saved N13 CSV (about 144 MB on disk)
once per fresh cache. This is data loading, not metric recomputation. Remote pinned loading,
compact lookup and deployed memory/latency remain explicit deployment gates, not waived by
a local pass. AppTest checks three case states and an invalid-identity route, not JS dialogs.

Run all four code cells. Stop after this batch; historical Batch 6+ cells remain unreconciled.


In [22]:
drop_validation_stages(('batch_5_', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))


start_room_batch(5)
from restoration_eval import model_gallery as GALLERY
# Discard process-local read caches only; no disk cache or running server changes.
GALLERY.catalogue.cache_clear()
GALLERY.metric_index.cache_clear()
GALLERY.painting_geometry.cache_clear()
GALLERY.image_uri.cache_clear()
GALLERY_FREEZE = n35.verify_gallery_freeze(PROJECT_ROOT)
record_room_stage("batch_5_approved_source", [
    room_check("seven_gallery_files_plus_shared_transport", 8, len(GALLERY_FREEZE)),
    *[room_check("frozen__" + r["path"], r["expected"], r["observed"]) for r in GALLERY_FREEZE],
    test_record("gallery"),
])
print("The historical notebook-SHA assertion was replaced by a before/after cell-source and canonical-output check.")
print("Normal notebook output/metadata autosaves are allowed; cell-source edits are not.")
print("Its frozen test file was not edited. All scientific Gallery checks remain required.")
print("Gallery view normalization covers only the two approved e88feea5 transport substitutions.")


Running gallery local contract tests; no inference or artifact generation.
unittest.case.FunctionTestCase (run)
test_four_methods_share_all_primary_cases ... ok
unittest.case.FunctionTestCase (run)
test_all_paintings_have_declared_opening ... ok
unittest.case.FunctionTestCase (run)
test_invalid_identity_never_silently_substituted ... ok
unittest.case.FunctionTestCase (run)
test_default_matches_registered_candidates_and_measurements ... ok
unittest.case.FunctionTestCase (run)
test_no_empty_mask_crop_values_are_invented ... ok
unittest.case.FunctionTestCase (run)
test_every_case_has_valid_evidence_and_existing_images ... ok
unittest.case.FunctionTestCase (run)
test_sdxl_availability_is_case_scoped ... ok
unittest.case.FunctionTestCase (run)
test_source_reports_retain_recorded_hashes ... ok
unittest.case.FunctionTestCase (run)
test_cases_keep_experiment_identity ... ok
unittest.case.FunctionTestCase (run)
test_frame_content_geometry_is_recorded_and_valid ... ok

--------------------------

,check_id,passed,expected,observed
0,seven_gallery_files_plus_shared_transport,True,8,8
1,frozen__src/restoration_eval/model_gallery.py,True,715321d2f3b368f058cbbcf86d235871a52b8bfecb2da8...,715321d2f3b368f058cbbcf86d235871a52b8bfecb2da8...
2,frozen__src/restoration_eval/model_gallery_vie...,True,486ea42cdca7bcb4c37c2f1a11deeb075013730fe9f0e0...,486ea42cdca7bcb4c37c2f1a11deeb075013730fe9f0e0...
3,frozen__streamlit_assets/model_gallery.css,True,e15922f2231df238a406b3818ddc7a966569f2d5132c65...,e15922f2231df238a406b3818ddc7a966569f2d5132c65...
4,frozen__streamlit_assets/model_gallery_control...,True,827fd2a683f6d722d7867e98c1a7bf49d76241a916a6a0...,827fd2a683f6d722d7867e98c1a7bf49d76241a916a6a0...
5,frozen__streamlit_assets/rooms/model_gallery_s...,True,54cafbe20c496e322700445063f3bab32d707462c0241a...,54cafbe20c496e322700445063f3bab32d707462c0241a...
6,frozen__tests/test_model_gallery.py,True,e4f1da85c4bd22f6a95280ea72122345b0995dc54e1e70...,e4f1da85c4bd22f6a95280ea72122345b0995dc54e1e70...
7,frozen__tools/audit_model_gallery_readiness.py,True,f3b99c99ab8ed6201eea351eb7950809875b9fdce80029...,f3b99c99ab8ed6201eea351eb7950809875b9fdce80029...
8,frozen__src/restoration_eval/room_runtime.py,True,16d7110776c7f2d97365eddb397e5fc1056f8bbdfc493c...,16d7110776c7f2d97365eddb397e5fc1056f8bbdfc493c...
9,focused_tests,True,True,True


The historical notebook-SHA assertion was replaced by a before/after cell-source and canonical-output check.
Normal notebook output/metadata autosaves are allowed; cell-source edits are not.
Its frozen test file was not edited. All scientific Gallery checks remain required.
Gallery view normalization covers only the two approved e88feea5 transport substitutions.


In [24]:
drop_validation_stages(('batch_5_exact_evidence', 'batch_5_server_smoke', 'batch_5_final_gate', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_5_approved_source'])

with n35.offline() as GALLERY_DATA_NETWORK:
    cat = GALLERY.catalogue()
    GALLERY_ROOM = PACKAGE.load_room("model_gallery")
    GALLERY_BINDINGS = PACKAGE.bindings_for(room_id="model_gallery")
    opening = GALLERY.case_payload("canonical__p018__mixed_damage")
    without_sdxl = next(c for c in sorted(cat["cases"])
                        if c.startswith("canonical__p018__") and "zero" not in c
                        and c not in cat["models"]["sdxl_inpainting"])
    absent = GALLERY.case_payload(without_sdxl)
    zero = GALLERY.case_payload("canonical__p018__zero_control")
    sd = opening["models"]["stable_diffusion_inpainting"]["candidate"]
    empty_crop_unranked = all(
        entry["metrics"]["crop_ssim"] is None
        or entry["metrics"]["crop_ssim"]["status"] != "ok"
        or entry["metrics"]["crop_ssim"]["restored_value"] is None
        for entry in zero["models"].values())
    invalid_rejected = False
    try:
        GALLERY.selected_case("p018", "canonical__p019__mixed_damage")
    except ValueError:
        invalid_rejected = True

record_room_stage("batch_5_exact_evidence", [
    room_check("room_and_binding_count", (12, 19), (len(GALLERY_ROOM), len(GALLERY_BINDINGS))),
    room_check("binding_ids", set(GALLERY_ROOM.display_id), set(GALLERY_BINDINGS.display_id)),
    room_check("population", (300, 2620, 4), (len(cat["paintings"]), len(cat["cases"]), len(GALLERY.MAIN_MODELS))),
    room_check("sd_primary_identity", ("sd15__p00__s2026__d0cd65cf894a", 2026, True),
        (sd["candidate_id"], sd["seed"], sd["is_primary_candidate"])),
    room_check("sdxl_bounded_coverage", (24, 35, 19),
        (len(cat["models"]["sdxl_inpainting"]), len(cat["sdxl_scheduled"]),
         len({r["painting_id"] for r in cat["models"]["sdxl_inpainting"].values()}))),
    room_check("sdxl_exact_case_available", True, "sdxl_inpainting" in opening["models"]),
    room_check("sdxl_not_substituted_same_painting", False, "sdxl_inpainting" in absent["models"]),
    room_check("zero_crop_not_ranked", True, empty_crop_unranked),
    room_check("cross_painting_identity_rejected", True, invalid_rejected),
    room_check("outbound_network_attempts", [], GALLERY_DATA_NETWORK),
])
GALLERY_LOCAL_SUMMARY = pd.DataFrame([
    {"case_id": p["case_id"], "models": len(p["models"]), "sdxl_state": p["sdxl_status"]}
    for p in (opening, absent, zero)])
display(GALLERY_LOCAL_SUMMARY)
print("Candidate metrics are saved N13 rows, not newly computed scores; no combined ranking is produced.")


,check_id,passed,expected,observed
0,room_and_binding_count,True,"(12, 19)","(12, 19)"
1,binding_ids,True,"{models.output.hint, models.output.lama, model...","{models.output.hint, models.output.lama, model..."
2,population,True,"(300, 2620, 4)","(300, 2620, 4)"
3,sd_primary_identity,True,"(sd15__p00__s2026__d0cd65cf894a, 2026, True)","(sd15__p00__s2026__d0cd65cf894a, 2026, True)"
4,sdxl_bounded_coverage,True,"(24, 35, 19)","(24, 35, 19)"
5,sdxl_exact_case_available,True,True,True
6,sdxl_not_substituted_same_painting,True,False,False
7,zero_crop_not_ranked,True,True,True
8,cross_painting_identity_rejected,True,True,True
9,outbound_network_attempts,True,[],[]


,case_id,models,sdxl_state
0,canonical__p018__mixed_damage,5,completed
1,canonical__p018__loss_large,4,not_scheduled
2,canonical__p018__zero_control,4,not_scheduled


Candidate metrics are saved N13 rows, not newly computed scores; no combined ranking is produced.


In [25]:
drop_validation_stages(('batch_5_server_smoke', 'batch_5_final_gate', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_5_exact_evidence'])

GALLERY_RUNTIME_RECORDS = []
for label, case_id, sdxl_available in (
    ("opening", "canonical__p018__mixed_damage", True),
    ("same_painting_no_sdxl", without_sdxl, False),
    ("zero_control", "canonical__p018__zero_control", False),
):
    smoke = n35.room_smoke(APP_PATH, {"room": "model_gallery", "gallery_painting": "p018",
        "gallery_case": case_id})
    markup = smoke["html"]
    expected = {"errors": [], "network": [], "exact_case": True,
                "four_method_titles": 4, "sdxl_available": sdxl_available, "controller": True}
    observed = {"errors": smoke["errors"], "network": smoke["network_attempts"],
        "exact_case": f'data-case-id="{case_id}"' in markup,
        "four_method_titles": markup.count('class="gallery-method-title '),
        "sdxl_available": 'aria-label="Inspect bounded SDXL result"' in markup,
        "controller": any("MuseumReadiness.watch" in str(e.proto.srcdoc)
                          for e in smoke["app"].get("iframe"))}
    GALLERY_RUNTIME_RECORDS.append(room_check(label, expected, observed))

# A mismatched deep link must visibly fail, never display a different case.
invalid = n35.room_smoke(APP_PATH, {"room": "model_gallery", "gallery_painting": "p018",
    "gallery_case": "canonical__p019__mixed_damage"})
GALLERY_RUNTIME_RECORDS.append(room_check("invalid_identity_fails_closed", True,
    not invalid["app"].exception and not invalid["network_attempts"]
    and any("does not belong" in error for error in invalid["errors"])
    and 'class="model-stage"' not in invalid["html"]))
record_room_stage("batch_5_server_smoke", GALLERY_RUNTIME_RECORDS)


2026-10-01 09:21:59.209 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 09:22:12.477 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:22:12.497 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:22:12.567 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 09:22:14.116 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:22:14.129 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 09:22:14.166 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 09:22:15.693 Please replace `st.components.v1.html`

,check_id,passed,expected,observed
0,opening,True,"{'errors': [], 'network': [], 'exact_case': Tr...","{'errors': [], 'network': [], 'exact_case': Tr..."
1,same_painting_no_sdxl,True,"{'errors': [], 'network': [], 'exact_case': Tr...","{'errors': [], 'network': [], 'exact_case': Tr..."
2,zero_control,True,"{'errors': [], 'network': [], 'exact_case': Tr...","{'errors': [], 'network': [], 'exact_case': Tr..."
3,invalid_identity_fails_closed,True,True,True


In [26]:
drop_validation_stages(('batch_5_final_gate', 'batch_6_', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

finish_room_batch(5, {"batch_5_approved_source", "batch_5_exact_evidence", "batch_5_server_smoke"})
N35_PENDING_GATES = pd.DataFrame([
    {"gate": "Batches 6–9 current room validation", "status": "pending"},
    {"gate": "Browser interactions, reload boundaries, keyboard/focus and Firefox/second monitor", "status": "not tested by AppTest"},
    {"gate": "Responsive rendering at desktop/tablet/mobile viewports", "status": "pending fresh browser QA"},
    {"gate": "Gallery local originals + N13 CSV: pinned remote retrieval/compact lookup", "status": "deployment blocker; not fixed or waived here"},
    {"gate": "Clean-checkout Git/HF assets, SHA verification, failure/retry and cold/warm cache", "status": "pending"},
    {"gate": "Startup, memory, latency, concurrency and deployed dependency alignment", "status": "pending measurement"},
    {"gate": "Live Streamlit site and Batch 10 canonical readiness report", "status": "pending"},
])
display(N35_PENDING_GATES)
print("STOP HERE at the end of Batch 5. Do not run the historical Batch 6+ cells or Run All.")
print("Five of ten batch gates can be complete after your successful execution; N35 is not yet complete.")


,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_5_approved_source,10,10
batch_5_exact_evidence,10,10
batch_5_final_gate,2,2
batch_5_server_smoke,4,4


Batch 5: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.


,gate,status
0,Batches 6–9 current room validation,pending
1,"Browser interactions, reload boundaries, keybo...",not tested by AppTest
2,Responsive rendering at desktop/tablet/mobile ...,pending fresh browser QA
3,Gallery local originals + N13 CSV: pinned remo...,deployment blocker; not fixed or waived here
4,"Clean-checkout Git/HF assets, SHA verification...",pending
5,"Startup, memory, latency, concurrency and depl...",pending measurement
6,Live Streamlit site and Batch 10 canonical rea...,pending


STOP HERE at the end of Batch 5. Do not run the historical Batch 6+ cells or Run All.
Five of ten batch gates can be complete after your successful execution; N35 is not yet complete.


## Batch 6 — Stability Lab — exact perturbations and seed groups

Validate the frozen room, not redesign it. Read saved N05/N06/N07/N18/N22 records:
35 focused paintings, 245 damage-size cases, 525 mask variants, 350 eligible degradation cases
(within 1,155 registered rows), and 1,025 four-seed groups / 6,150 unordered pairs.
Keep prompt identities separate and deterministic-method uncertainty unavailable, never zero.

The Batch 4/5 count corrections are valid: 12 displays have respectively 21 and 19 binding slots.
Recheck distinct slots and display coverage against N34 before continuing. Requires completed
replacement Batch 5 in the current kernel. Reload only the validation helper; never restart the server.

Original freeze hashes remain required. Normalize only the two approved e88feea5 non-visual
transport substitutions, and separately pin the final shared runtime. No fresh baselines are invented.
Tests load saved tables (some large), but run no inference, scientific producers or publication.
All cells in this pack are read-only with respect to the approved dashboard and canonical N35 outputs.
Python renders and offline controller unit tests are not browser/viewport or cloud certification.


In [27]:
# Continue in the kernel that successfully completed the replacement Batch 5.
import importlib
import re
from collections import Counter
n35 = importlib.reload(n35)
n35.required_stages(VALIDATION.checks, ["batch_5_final_gate"])
drop_validation_stages(tuple(f"batch_{i}_" for i in range(6, 11)))
VALIDATION.raise_for_blocking()
start_room_batch(6)

def room_binding_checks(room_id, display_count, binding_count):
    room = PACKAGE.load_room(room_id)
    bindings = PACKAGE.bindings_for(room_id=room_id)
    return [
        room_check(room_id + "__display_rows", display_count, len(room)),
        room_check(room_id + "__distinct_displays", display_count, room.display_id.nunique()),
        room_check(room_id + "__binding_rows", binding_count, len(bindings)),
        room_check(room_id + "__unique_slots", binding_count, bindings.slot_id.nunique()),
        room_check(room_id + "__display_coverage", set(room.display_id), set(bindings.display_id)),
    ]

def freeze_checks(name):
    rows = n35.verify_later_room_freeze(PROJECT_ROOT, name)
    expected_count = {"stability_lab": 23, "trustworthiness": 27,
                      "focused_portrait": 24, "research_archive": 52}[name]
    return [room_check(name + "__freeze_count", expected_count, len(rows))] + [
        room_check(name + "__" + row["path"], row["expected"], row["observed"])
        for row in rows]

def js_checks(paths):
    result = n35.node_checks(PROJECT_ROOT, paths)
    for row in result:
        print(row["path"], row["stdout"], row["stderr"])
    return [room_check("javascript__" + row["path"], True, row["passed"]) for row in result]

def nested_value(data, path):
    for key in path.split("."):
        data = data[key]
    return data

def render_cases(cases):
    """Fresh Python renders with exact controller JSON, not browser-JS execution."""
    result = []
    for spec in cases:
        print("Server render:", spec["name"], flush=True)
        smoke = n35.room_smoke(APP_PATH, spec["query"], timeout=180)
        app = smoke["app"]
        if "error" in spec:
            observed = dict(network=smoke["network_attempts"], exceptions=len(app.exception),
                visible_rejection=any(spec["error"] in text for text in smoke["errors"]),
                substituted_scene=bool(re.search(r'<main\b[^>]*class="' + spec["scene"] + r'"', smoke["html"])))
            expected = dict(network=[], exceptions=0, visible_rejection=True, substituted_scene=False)
        else:
            data = n35.controller_payload(app)
            observed = dict(network=smoke["network_attempts"], errors=smoke["errors"],
                scene=bool(re.search(r'<main\b[^>]*class="' + spec["scene"] + r'"', smoke["html"])),
                identity={key: nested_value(data, key) for key in spec["expect"]})
            expected = dict(network=[], errors=[], scene=True, identity=spec["expect"])
        result.append(room_check(spec["name"], expected, observed))
    return result

# Correct the earlier 1:1 assumption: child asset/map slots are separate bindings.
# Recheck against the immutable N34 package, not just the edited notebook numbers.
with n35.offline() as binding_network:
    corrected_bindings = room_binding_checks("metric_framework", 12, 21)
    corrected_bindings += room_binding_checks("model_gallery", 12, 19)
record_room_stage("batch_6_prior_binding_recheck", corrected_bindings + [
    room_check("outbound_network_attempts", [], binding_network)])

shared_freeze = n35.verify_freeze(PROJECT_ROOT, "config/publication/museum_visit_freeze.json")
record_room_stage("batch_6_source_contract", freeze_checks("stability_lab") + [
    room_check("shared_approved__" + r["path"], r["expected"], r["observed"])
    for r in shared_freeze] + room_binding_checks("stability_lab", 8, 19) + [test_record("stability")])


,check_id,passed,expected,observed
0,metric_framework__display_rows,True,12,12
1,metric_framework__distinct_displays,True,12,12
2,metric_framework__binding_rows,True,21,21
3,metric_framework__unique_slots,True,21,21
4,metric_framework__display_coverage,True,"{metric.lens.local, metric.quality.anchors, me...","{metric.lens.local, metric.quality.anchors, me..."
5,model_gallery__display_rows,True,12,12
6,model_gallery__distinct_displays,True,12,12
7,model_gallery__binding_rows,True,19,19
8,model_gallery__unique_slots,True,19,19
9,model_gallery__display_coverage,True,"{models.output.hint, models.output.lama, model...","{models.output.hint, models.output.lama, model..."


Running stability local contract tests; no inference or artifact generation.
test_all_focused_selectors_are_complete (n35_checks_test_stability_lab.StabilityLabTests.test_all_focused_selectors_are_complete) ... ok
test_damage_seed_overlays_exist (n35_checks_test_stability_lab.StabilityLabTests.test_damage_seed_overlays_exist) ... ok
test_declared_default (n35_checks_test_stability_lab.StabilityLabTests.test_declared_default) ... ok
test_decorative_book_spines (n35_checks_test_stability_lab.StabilityLabTests.test_decorative_book_spines) ... ok
test_default_disabled_and_explicit_seed_payload (n35_checks_test_stability_lab.StabilityLabTests.test_default_disabled_and_explicit_seed_payload) ... ok
test_every_primary_candidate_exists (n35_checks_test_stability_lab.StabilityLabTests.test_every_primary_candidate_exists) ... ok
test_opening_trajectory_golden_values (n35_checks_test_stability_lab.StabilityLabTests.test_opening_trajectory_golden_values) ... ok
test_p05_uses_the_exact_group_candid

,check_id,passed,expected,observed
0,stability_lab__freeze_count,True,23,23
1,stability_lab__src/restoration_eval/stability_...,True,f2f2a262435112729db30ba936c8fac1050df1f74c1217...,f2f2a262435112729db30ba936c8fac1050df1f74c1217...
2,stability_lab__src/restoration_eval/stability_...,True,f0d71ff21522696c6ba8d30925efe624ebb489b27d0c54...,f0d71ff21522696c6ba8d30925efe624ebb489b27d0c54...
3,stability_lab__streamlit_assets/stability_lab.css,True,af9d9c185f6c91c8e501b5587011faa00b38bbb66f8128...,af9d9c185f6c91c8e501b5587011faa00b38bbb66f8128...
4,stability_lab__streamlit_assets/stability_lab_...,True,80efc9f2ce57d7ae8f07e5975b718eb502f69cdf38c03e...,80efc9f2ce57d7ae8f07e5975b718eb502f69cdf38c03e...
5,stability_lab__streamlit_assets/rooms/stabilit...,True,81504d77921d84fe734ef5bc1a116e3dfc0602e56fe2c0...,81504d77921d84fe734ef5bc1a116e3dfc0602e56fe2c0...
6,stability_lab__tests/test_stability_lab.py,True,054dfbbadb12954926d842dc10b1deb7ed210df8b8af24...,054dfbbadb12954926d842dc10b1deb7ed210df8b8af24...
7,stability_lab__tools/audit_stability_lab_readi...,True,25f147a9bcbbf26398410d4e00e4ebb85c0cf1b62989a5...,25f147a9bcbbf26398410d4e00e4ebb85c0cf1b62989a5...
8,stability_lab__tools/audit_stability_lab_publi...,True,946e0ad8150d0f5c537b766fc02d0bfc0e1a6643f6a267...,946e0ad8150d0f5c537b766fc02d0bfc0e1a6643f6a267...
9,stability_lab__docs/dashboard_design_finalists...,True,caa9b181bc8140d9c23536f8b2689e4b634a4b8806fc10...,caa9b181bc8140d9c23536f8b2689e4b634a4b8806fc10...


In [28]:
drop_validation_stages(('batch_6_saved_evidence', 'batch_6_server_smoke', 'batch_6_final_gate', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_6_source_contract'])

from restoration_eval import stability_lab as LAB
with n35.offline() as attempts:
    cat = LAB.test_catalogue()
    groups = LAB.seed_catalogue()
    LAB_OPENING = LAB.payload({})
    LAB_SEED = LAB.payload({"stability_test": "seed", "stability_model": LAB.SD})
    p05 = next(g for g in LAB.seed_options("p018") if g["prompt"] == "p05_scratch_aware")
    p05_payload = LAB.payload({"stability_test": "seed", "stability_model": LAB.SD,
                              "stability_group": p05["id"]})
    LAB_SEED_QUERIES = {"default": LAB_SEED["seed"]["id"], "p05": p05["id"]}
    invalid_rejected = False
    try:
        LAB.selection({"stability_test": "seed", "stability_model": "lama"})
    except ValueError:
        invalid_rejected = True
record_room_stage("batch_6_saved_evidence", [
    room_check("focused_populations", (35, 245, 525, 1155, 350),
        (len(cat["paintings"]), len(cat["size"]), len(cat["mask"]), len(cat["degradation"]),
         sum(r["degradation_family"] in LAB.FAMILIES for r in cat["degradation"]))),
    room_check("seed_population", (1025, 6150),
        (len(groups), sum(len(g["pairs"]) for g in groups.values()))),
    room_check("four_seeds_per_group", True,
        all(set(g["members"]) == {2026, 2027, 2028, 2029} and len(g["pairs"]) == 6 for g in groups.values())),
    room_check("opening_identity", ("p018", "lama", "size", "damage_size__p018__loss_large__size_20pct"),
        tuple(LAB_OPENING[k] for k in ("painting", "model", "test", "selected"))),
    room_check("deterministic_seed_panel_is_unavailable", None, LAB_OPENING["seed"]),
    room_check("seed_group_identity", "ug_1de955f7e0f3d00cf0", LAB_SEED["seed"]["id"]),
    room_check("p05_candidate_not_p00", p05["members"][2026], p05_payload["current"]["candidate_id"]),
    room_check("p05_missing_overlay_not_substituted", None, p05_payload["seed"]["overlay"]),
    room_check("invalid_deterministic_seed_rejected", True, invalid_rejected),
    room_check("outbound_network_attempts", [], attempts),
])
display(pd.DataFrame([{k: p[k] for k in ("case_id", "label", "candidate_id")} | {"saved_metric": p["metric"]}
                      for p in LAB_OPENING["points"]]))


,check_id,passed,expected,observed
0,focused_populations,True,"(35, 245, 525, 1155, 350)","(35, 245, 525, 1155, 350)"
1,seed_population,True,"(1025, 6150)","(1025, 6150)"
2,four_seeds_per_group,True,True,True
3,opening_identity,True,"(p018, lama, size, damage_size__p018__loss_lar...","(p018, lama, size, damage_size__p018__loss_lar..."
4,deterministic_seed_panel_is_unavailable,True,None,None
5,seed_group_identity,True,ug_1de955f7e0f3d00cf0,ug_1de955f7e0f3d00cf0
6,p05_candidate_not_p00,True,sd15__p05__s2026__26d9162591d1,sd15__p05__s2026__26d9162591d1
7,p05_missing_overlay_not_substituted,True,None,None
8,invalid_deterministic_seed_rejected,True,True,True
9,outbound_network_attempts,True,[],[]


,case_id,label,candidate_id,saved_metric
0,damage_size__p018__loss_large__size_02pct,2%,candidate__lama__damage_size__p018__loss_large...,"{'value': 4.1318695438, 'input_value': 88.2655..."
1,damage_size__p018__loss_large__size_04pct,4%,candidate__lama__damage_size__p018__loss_large...,"{'value': 6.3067950775, 'input_value': 95.4160..."
2,damage_size__p018__loss_large__size_06pct,6%,candidate__lama__damage_size__p018__loss_large...,"{'value': 9.6465636489, 'input_value': 103.742..."
3,damage_size__p018__loss_large__size_08pct,8%,candidate__lama__damage_size__p018__loss_large...,"{'value': 9.5337827707, 'input_value': 111.086..."
4,damage_size__p018__loss_large__size_10pct,10%,candidate__lama__damage_size__p018__loss_large...,"{'value': 13.2004297422, 'input_value': 118.31..."
5,damage_size__p018__loss_large__size_15pct,15%,candidate__lama__damage_size__p018__loss_large...,"{'value': 30.9869134483, 'input_value': 132.42..."
6,damage_size__p018__loss_large__size_20pct,20%,candidate__lama__damage_size__p018__loss_large...,"{'value': 39.3301701445, 'input_value': 140.41..."


In [29]:
drop_validation_stages(('batch_6_server_smoke', 'batch_6_final_gate', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_6_saved_evidence'])

STABILITY_RENDER_CASES = [
    dict(name="size_opening", query={"room": "stability_lab"}, scene="stability-stage",
         expect={"painting": "p018", "test": "size", "model": "lama", "selected": "damage_size__p018__loss_large__size_20pct", "seed": None}),
    dict(name="mask_branch", query={"room": "stability_lab", "stability_test": "mask"}, scene="stability-stage",
         expect={"test": "mask", "condition": "loss_large", "selected": "mask_robustness__p018__loss_large__target_12p5pct__variant_01"}),
    dict(name="degradation_branch", query={"room": "stability_lab", "stability_test": "degradation"}, scene="stability-stage",
         expect={"test": "degradation", "family": "dirt_dust", "selected": "synthetic_degradation__p018__dirt_dust__mild"}),
    dict(name="exact_seed_branch", query={"room": "stability_lab", "stability_test": "seed", "stability_model": LAB.SD}, scene="stability-stage",
         expect={"test": "seed", "seed.id": LAB_SEED_QUERIES["default"]}),
    dict(name="wrong_seed_method_rejected", query={"room": "stability_lab", "stability_test": "seed"}, scene="stability-stage",
         error="explicit switch to Stable Diffusion"),
]
record_room_stage("batch_6_server_smoke", render_cases(STABILITY_RENDER_CASES))


Server render: size_opening


2026-10-01 10:15:19.472 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:15:24.428 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:15:24.444 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: mask_branch


2026-10-01 10:15:24.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:15:26.603 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:15:26.615 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: degradation_branch


...

Server render: exact_seed_branch


2026-10-01 10:15:28.918 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:15:38.580 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:15:38.595 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: wrong_seed_method_rejected


2026-10-01 10:15:38.984 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:15:39.901 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


,check_id,passed,expected,observed
0,size_opening,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
1,mask_branch,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
2,degradation_branch,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
3,exact_seed_branch,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
4,wrong_seed_method_rejected,True,"{'network': [], 'exceptions': 0, 'visible_reje...","{'network': [], 'exceptions': 0, 'visible_reje..."


In [30]:
drop_validation_stages(('batch_6_final_gate', 'batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))

finish_room_batch(6, {'batch_6_saved_evidence', 'batch_6_source_contract', 'batch_6_server_smoke', 'batch_6_prior_binding_recheck'})


,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_6_final_gate,2,2
batch_6_prior_binding_recheck,11,11
batch_6_saved_evidence,10,10
batch_6_server_smoke,5,5
batch_6_source_contract,41,41


Batch 6: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.


## Batch 7 — Trustworthiness and the shared D02 review


Validate the 13,879-candidate union, separate primary/uncertainty/bounded roles, exact saved
assignments, threshold provenance and unresolved evidence. Computational flags are not expert
ground truth or conservation approval. Empirical seed disagreement is not calibrated confidence.

Validate D02's 60 screened portraits, 45 matched cases / 20 paintings and 32 blind reviews.
Keep R005's exact candidate and equal disjoint hand/control supports; retain the R003 non-failure
counterexample without borrowing R005 assets. Render the same child route from its three allowed
parents. Rendered lightness is not race or identity; do not combine the separate metrics.

Python source/data tests and dependency-free Node chart tests are local validation only. AppTest
checks exact JSON sent to controllers and visible rejection states; it cannot operate browser dialogs.


In [31]:
drop_validation_stages(('batch_7_', 'batch_8_', 'batch_9_', 'batch_10_'))


start_room_batch(7)
record_room_stage("batch_7_source_contract",
    freeze_checks("trustworthiness") + freeze_checks("focused_portrait")
    + room_binding_checks("trustworthiness", 7, 14)
    + room_binding_checks("focused_portrait_review", 10, 19)
    + [test_record("trust_portrait")]
    + js_checks(["tests/test_trustworthiness_chart.cjs", "tests/test_focused_portrait_chart.cjs"]))


Running trust_portrait local contract tests; no inference or artifact generation.
Loading recorded Tab 6 tables once; no inference or scientific recomputation.
test_all_six_flags_and_unresolved_colour (n35_checks_test_trustworthiness.TrustworthinessTests.test_all_six_flags_and_unresolved_colour) ... ok
test_complete_assignment_provenance (n35_checks_test_trustworthiness.TrustworthinessTests.test_complete_assignment_provenance) ... ok
test_complete_union_roles (n35_checks_test_trustworthiness.TrustworthinessTests.test_complete_union_roles) ... ok
test_exact_opening_identity (n35_checks_test_trustworthiness.TrustworthinessTests.test_exact_opening_identity) ... ok
test_exact_threshold_and_registered_stratum (n35_checks_test_trustworthiness.TrustworthinessTests.test_exact_threshold_and_registered_stratum) ... ok
test_missing_and_ambiguous_threshold_do_not_pass (n35_checks_test_trustworthiness.TrustworthinessTests.test_missing_and_ambiguous_threshold_do_not_pass) ... ok
test_policy_and_imag

,check_id,passed,expected,observed
0,trustworthiness__freeze_count,True,27,27
1,trustworthiness__src/restoration_eval/trustwor...,True,4a500f023e9eea8bae74a0711106e0c8990b56d2c3bc9c...,4a500f023e9eea8bae74a0711106e0c8990b56d2c3bc9c...
2,trustworthiness__src/restoration_eval/trustwor...,True,7289fef5a08135f0a2992292fb885790b568b8b26d9116...,7289fef5a08135f0a2992292fb885790b568b8b26d9116...
3,trustworthiness__streamlit_assets/trustworthin...,True,2aad3dc52e5cc2d28230c5e3b08f4b2a46144b8da44cd8...,2aad3dc52e5cc2d28230c5e3b08f4b2a46144b8da44cd8...
4,trustworthiness__streamlit_assets/trustworthin...,True,575f2c62e7baefaa85824609404fba1d5827002467c0d6...,575f2c62e7baefaa85824609404fba1d5827002467c0d6...
...,...,...,...,...
61,focused_portrait_review__unique_slots,True,19,19
62,focused_portrait_review__display_coverage,True,"{d02.anchor.overlap, d02.anchor.images, d02.an...","{d02.anchor.overlap, d02.anchor.images, d02.an..."
63,focused_tests,True,True,True
64,javascript__tests/test_trustworthiness_chart.cjs,True,True,True


In [32]:
drop_validation_stages(('batch_7_saved_evidence', 'batch_7_server_smoke', 'batch_7_final_gate', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_7_source_contract'])

from restoration_eval import trustworthiness as TRUST
from restoration_eval import focused_portrait as PORTRAIT
with n35.offline() as attempts:
    TRUST_OPENING = TRUST.payload(PACKAGE)
    PORTRAIT_OPENING = PORTRAIT.payload(PACKAGE)
    PORTRAIT_COUNTER = PORTRAIT.payload(PACKAGE, "R003")
    roles = Counter(r["population_role"] for r in TRUST.candidate_index().values())
    threshold = TRUST_OPENING["thresholds"]["texture_smoothing"]["local_texture_error_p95"]
    registered_returns = [PORTRAIT.return_room(p) for p in ("study_design", "trustworthiness", "case_explorer")]
record_room_stage("batch_7_saved_evidence", [
    room_check("candidate_union_roles", {"primary_and_uncertainty": 725, "uncertainty_only": 3375,
        "bounded_sdxl": 24, "primary_comparison": 9755}, dict(roles)),
    room_check("trust_exact_opening", ("sd15__p00__s2026__29ff258ff921", "canonical__p002__loss_large", 2026, "p00_generic"),
        tuple(TRUST_OPENING["candidate"][k] for k in ("candidate_id", "case_id", "seed", "prompt_variant_id"))),
    room_check("category_flag_policy_counts", (14, 11, 23, 4),
        tuple(len(TRUST_OPENING[k]) for k in ("categories", "flags", "policy", "peers"))),
    room_check("saved_threshold", (10.053275680541985, 1.7923734879493693, 4.8924025321006255, "critical"),
        tuple(threshold[k] for k in ("observed", "warning", "critical", "state"))),
    room_check("d02_review_identity", ("R005", "candidate__hint_places2__canonical__p269__mixed_damage__c00"),
        tuple(PORTRAIT_OPENING["review"][k] for k in ("blind_review_code", "candidate_id"))),
    room_check("d02_scope", (60, 45, 20, 32, 8, 25),
        tuple(PORTRAIT_OPENING["scope"][k] for k in ("screened", "hand_cases", "hand_paintings", "reviews", "review_cases", "visible_failures"))),
    room_check("matched_equal_disjoint_pixels", (1196, 1196, 0),
        (len(PORTRAIT_OPENING["hand_indices"]), len(PORTRAIT_OPENING["control_indices"]),
         len(set(PORTRAIT_OPENING["hand_indices"]) & set(PORTRAIT_OPENING["control_indices"])))),
    room_check("counterexample_is_not_failure", False, PORTRAIT_COUNTER["review"]["overall_anatomy_failure"]),
    room_check("counterexample_has_no_r005_asset_substitution", {}, PORTRAIT_COUNTER["registered"]),
    room_check("one_child_three_return_routes", ["study_design", "trustworthiness", "case_explorer"], registered_returns),
    room_check("outbound_network_attempts", [], attempts),
])
display(pd.DataFrame([TRUST_OPENING["candidate"], PORTRAIT_OPENING["review"]])[
    ["painting_id", "candidate_id", "case_id", "model_id"]])


,check_id,passed,expected,observed
0,candidate_union_roles,True,"{'primary_and_uncertainty': 725, 'uncertainty_...","{'bounded_sdxl': 24, 'primary_and_uncertainty'..."
1,trust_exact_opening,True,"(sd15__p00__s2026__29ff258ff921, canonical__p0...","(sd15__p00__s2026__29ff258ff921, canonical__p0..."
2,category_flag_policy_counts,True,"(14, 11, 23, 4)","(14, 11, 23, 4)"
3,saved_threshold,True,"(10.053275680541985, 1.7923734879493693, 4.892...","(10.053275680541985, 1.7923734879493693, 4.892..."
4,d02_review_identity,True,"(R005, candidate__hint_places2__canonical__p26...","(R005, candidate__hint_places2__canonical__p26..."
5,d02_scope,True,"(60, 45, 20, 32, 8, 25)","(60, 45, 20, 32, 8, 25)"
6,matched_equal_disjoint_pixels,True,"(1196, 1196, 0)","(1196, 1196, 0)"
7,counterexample_is_not_failure,True,False,False
8,counterexample_has_no_r005_asset_substitution,True,{},{}
9,one_child_three_return_routes,True,"[study_design, trustworthiness, case_explorer]","[study_design, trustworthiness, case_explorer]"


,painting_id,candidate_id,case_id,model_id
0,p002,sd15__p00__s2026__29ff258ff921,canonical__p002__loss_large,stable_diffusion_inpainting
1,p269,candidate__hint_places2__canonical__p269__mixe...,canonical__p269__mixed_damage,hint_places2


In [33]:
drop_validation_stages(('batch_7_server_smoke', 'batch_7_final_gate', 'batch_8_', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_7_saved_evidence'])

TRUST_RENDER_CASES = [
    dict(name="trust_exact_opening", query={"room": "trustworthiness"}, scene="trust-stage",
         expect={"candidate.candidate_id": TRUST.OPENING, "candidate.seed": 2026}),
    dict(name="trust_invalid_candidate", query={"room": "trustworthiness", "trust_candidate": "not-a-candidate"},
         scene="trust-stage", error="nothing substituted"),
    *[dict(name="r005_return_" + parent,
         query={"room": "focused_portrait_review", "portrait_review": "R005", "return_room": parent},
         scene="fpr-stage", expect={"review.blind_review_code": "R005", "return_room": parent,
                                    "review.candidate_id": "candidate__hint_places2__canonical__p269__mixed_damage__c00"})
      for parent in ("study_design", "trustworthiness", "case_explorer")],
    dict(name="r003_counterexample", query={"room": "focused_portrait_review", "portrait_review": "R003"},
         scene="fpr-stage", expect={"review.blind_review_code": "R003", "review.overall_anatomy_failure": False, "registered": {}}),
    dict(name="invalid_review", query={"room": "focused_portrait_review", "portrait_review": "R999"},
         scene="fpr-stage", error="blind_review_code"),
]
record_room_stage("batch_7_server_smoke", render_cases(TRUST_RENDER_CASES))


Server render: trust_exact_opening


2026-10-01 10:16:12.606 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:16:35.731 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:16:35.750 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: trust_invalid_candidate


2026-10-01 10:16:36.177 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:16:37.103 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: r005_return_study_design


...

Server render: r003_counterexample


2026-10-01 10:16:44.427 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:16:46.089 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:16:46.101 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: invalid_review


2026-10-01 10:16:46.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:16:47.353 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


,check_id,passed,expected,observed
0,trust_exact_opening,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
1,trust_invalid_candidate,True,"{'network': [], 'exceptions': 0, 'visible_reje...","{'network': [], 'exceptions': 0, 'visible_reje..."
2,r005_return_study_design,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
3,r005_return_trustworthiness,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
4,r005_return_case_explorer,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
5,r003_counterexample,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
6,invalid_review,True,"{'network': [], 'exceptions': 0, 'visible_reje...","{'network': [], 'exceptions': 0, 'visible_reje..."


In [34]:
drop_validation_stages(('batch_7_final_gate', 'batch_8_', 'batch_9_', 'batch_10_'))

finish_room_batch(7, {'batch_7_server_smoke', 'batch_7_source_contract', 'batch_7_saved_evidence'})


,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_7_final_gate,2,2
batch_7_saved_evidence,11,11
batch_7_server_smoke,7,7
batch_7_source_contract,66,66


Batch 7: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.


## Batch 8 — Case Explorer — exact candidate and saved evidence


Pin the final approved Case Explorer files to committed revision e88feea5, never the current
worktree as a new baseline. The room has 15 display records and 40 raw binding slots; the
LaMa opening API returns 39 compatible bindings because an SD-only uncertainty binding is excluded.

Verify all 300 compact metric/map shards and their exact candidate/case/model joins. Verify sampled
saved image bytes through the existing readers, keep LaMa uncertainty not applicable, preserve SD
seed identity, and reject cross-painting selections. Retrieval and counterfactuals remain recorded
examples, not automatically valid evidence for the selected case or new scientific calculations.

The 13,879 registered candidates are not 13,879 measured candidates: 11,944 have the three
selected saved metrics; 1,935 have none. Reconcile this against the exact source-table selections,
not merely observed shard counts. The streamed validation scan reads about 1.3 GB locally;
it may take a few minutes, uses bounded memory and produces no scientific outputs.

These checks do not re-download every raster or independently recalculate every scientific metric.
Full deployed source closure, required dynamic numeric parity and browser interaction stay Batch 10 gates.


In [35]:
drop_validation_stages(('batch_8_', 'batch_9_', 'batch_10_'))


start_room_batch(8)
CASE_APPROVED_COMMIT = "e88feea5e0deb6d5e48cb12378d69ff5d63b2e11"
CASE_PINNED_PATHS = [
    "src/restoration_eval/case_explorer.py", "src/restoration_eval/case_explorer_view.py",
    "streamlit_assets/case_explorer.css", "streamlit_assets/case_explorer_controller.js",
    "streamlit_assets/rooms/case_explorer_shell.png", "streamlit_assets/rooms/case_explorer_lettering.png",
    "streamlit_assets/rooms/case_explorer_neutral_layer.png", "streamlit_assets/rooms/case_explorer_title_enhanced.png",
    "tests/test_case_explorer.py", "tests/case_explorer_controller.test.cjs",
    *["streamlit_assets/evidence/case_explorer/" + name for name in
      ("manifest.json", "maps_manifest.json", "inputs_manifest.json", "panels_manifest.json", "choices_manifest.json", "choices.json")],
]
CASE_FREEZE_ROWS = n35.verify_committed_files(PROJECT_ROOT, CASE_APPROVED_COMMIT, CASE_PINNED_PATHS)
# The default-selection API excludes the SD-only uncertainty child binding.
# Raw manifest: 40 rows. LaMa-compatible opening bindings: 39. Neither is 15.
raw_case_bindings = PACKAGE.display_components.loc[PACKAGE.display_components.room_id.eq("case_explorer")]
record_room_stage("batch_8_source_contract", [
    *[room_check("approved__" + r["path"], r["expected"], r["observed"]) for r in CASE_FREEZE_ROWS],
    room_check("raw_registered_binding_count", 40, len(raw_case_bindings)),
] + room_binding_checks("case_explorer", 15, 39) + [test_record("case")]
  + js_checks(["tests/case_explorer_controller.test.cjs", "tests/room_runtime.test.cjs"]))


Running case local contract tests; no inference or artifact generation.
test_all_five_opening_images_verified (n35_checks_test_case_explorer.CaseExplorerTests.test_all_five_opening_images_verified) ... ok
test_compact_diffusion_ledger_labels_and_neutral_layer_patch (n35_checks_test_case_explorer.CaseExplorerTests.test_compact_diffusion_ledger_labels_and_neutral_layer_patch) ... ok
test_content_crop_is_recorded_and_inspection_remains_full_image (n35_checks_test_case_explorer.CaseExplorerTests.test_content_crop_is_recorded_and_inspection_remains_full_image) ... ok
test_deterministic_uncertainty_not_zero (n35_checks_test_case_explorer.CaseExplorerTests.test_deterministic_uncertainty_not_zero) ... ok
test_each_comparison_choice_is_bound_to_its_saved_candidate (n35_checks_test_case_explorer.CaseExplorerTests.test_each_comparison_choice_is_bound_to_its_saved_candidate) ... ok
test_every_original_study_panel_is_present_and_verified (n35_checks_test_case_explorer.CaseExplorerTests.test_every_o

,check_id,passed,expected,observed
0,approved__src/restoration_eval/case_explorer.py,True,6ad0c0dd4601d586307ed5062d93b90f99fb2081f05c2e...,6ad0c0dd4601d586307ed5062d93b90f99fb2081f05c2e...
1,approved__src/restoration_eval/case_explorer_v...,True,8f917e3a5ad9ab0acaa44c487f9a59e212c2028643a707...,8f917e3a5ad9ab0acaa44c487f9a59e212c2028643a707...
2,approved__streamlit_assets/case_explorer.css,True,9ac66f2c9eeaadf871bc27bc93bf7b0615bac60f8f2720...,9ac66f2c9eeaadf871bc27bc93bf7b0615bac60f8f2720...
3,approved__streamlit_assets/case_explorer_contr...,True,614228b7b780129db55ed15a5f86920cd51ca7d5165c91...,614228b7b780129db55ed15a5f86920cd51ca7d5165c91...
4,approved__streamlit_assets/rooms/case_explorer...,True,046ce6a66acf5bbcc2e2b54e2568ed52b147c84daa35be...,046ce6a66acf5bbcc2e2b54e2568ed52b147c84daa35be...
5,approved__streamlit_assets/rooms/case_explorer...,True,af7470f5696e80fa5630b730c440cb2a65d9566e3ed28a...,af7470f5696e80fa5630b730c440cb2a65d9566e3ed28a...
6,approved__streamlit_assets/rooms/case_explorer...,True,68b4f825d2c6b4a8c47837b25fd7643f20cfb1c9eb74d0...,68b4f825d2c6b4a8c47837b25fd7643f20cfb1c9eb74d0...
7,approved__streamlit_assets/rooms/case_explorer...,True,3f390e707d23e4fd93b08f4b88847f777995052aa11edf...,3f390e707d23e4fd93b08f4b88847f777995052aa11edf...
8,approved__tests/test_case_explorer.py,True,0790883b88d49ee0d84361cbf5e796872c2761a2791edb...,0790883b88d49ee0d84361cbf5e796872c2761a2791edb...
9,approved__tests/case_explorer_controller.test.cjs,True,b446570aeea8b9ccb695216d55245b2599b8abccb2b2a8...,b446570aeea8b9ccb695216d55245b2599b8abccb2b2a8...


In [36]:
drop_validation_stages(('batch_8_saved_evidence', 'batch_8_server_smoke', 'batch_8_final_gate', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_8_source_contract'])

import csv
import hashlib
from restoration_eval import case_explorer as CASE
with n35.offline() as attempts:
    CASE_OPENING = CASE.payload(PACKAGE)
    CASE_SD = CASE.payload(PACKAGE, "p018", "sd15__p00__s2026__d0cd65cf894a")
    CASE_NA = CASE.payload(PACKAGE, layer="uncertainty")
    metric_manifest = json.loads((CASE.INDEX / "manifest.json").read_text(encoding="utf-8"))
    map_manifest = json.loads((CASE.INDEX / "maps_manifest.json").read_text(encoding="utf-8"))
    source_hashes = {r["sha256"] for r in metric_manifest["sources"]}
    expected_paintings = set(PACKAGE.painting_lookup.painting_id.astype(str))
    shard_audit, all_candidates, registered_candidates = [], set(), {}
    for pid in sorted(expected_paintings):
        # Public loaders verify each compact shard's checksum before decoding.
        rows, maps = CASE.saved_metrics(pid), CASE.saved_maps(pid)
        identities = PACKAGE.load_painting(pid)["candidates"]
        by_id = {r["candidate_id"]: r for r in identities}
        registered_candidates.update(by_id)
        exact = True
        for row in rows:
            c = by_id.get(row["candidate_id"])
            exact &= bool(c) and row["case_id"] == c["case_id"] and row["model_id"] == c["model_id"]
            exact &= row["source_sha256"] in source_hashes
            all_candidates.add(row["candidate_id"])
        keys = [(r["candidate_id"], r["metric_name"]) for r in rows]
        shard_audit.append(dict(painting_id=pid, rows=len(rows), exact_identity=bool(exact),
            unique_keys=len(keys) == len(set(keys)), maps=len(maps),
            expected_rows=metric_manifest["shards"][pid]["rows"]))
    CASE_SHARD_AUDIT = pd.DataFrame(shard_audit)
    # candidate_count is the permitted N34 universe, NOT metric availability.
    # Independently re-read the three checksum-pinned producer selections.
    # This bounded-memory validation scan is not an application runtime loader.
    selected_specs = {
        "outputs/13_classical_metrics/metrics/classical_metrics.csv": ("ssim", "mask_bbox_crop"),
        "outputs/14_lpips_metrics/metrics/lpips_metrics.csv": ("lpips", "mask_bbox_crop"),
        "outputs/17_local_consistency_metrics/metrics/local_consistency.csv": ("delta_e_ciede2000_mean", "masked_region"),
    }
    assert {s["path"] for s in metric_manifest["sources"]} == set(selected_specs)
    source_candidates = set()
    for source in metric_manifest["sources"]:
        path = n35.safe_path(PROJECT_ROOT, source["path"])
        print("Checking saved metric availability:", source["path"], flush=True)
        with path.open("rb") as handle:
            assert hashlib.file_digest(handle, "sha256").hexdigest() == source["sha256"], "Metric source checksum mismatch"
        metric_name, region_id = selected_specs[source["path"]]
        with path.open(encoding="utf-8-sig", newline="") as handle:
            for row in csv.DictReader(handle):
                if (row["metric_name"] == metric_name and row["region_id"] == region_id
                        and (metric_name != "lpips" or row.get("network") == "alex")
                        and row["candidate_id"] in registered_candidates):
                    source_candidates.add(row["candidate_id"])
    missing_metric_ids = set(registered_candidates) - all_candidates
    CASE_METRIC_AVAILABILITY = pd.DataFrame([
        {"state": "has a saved display metric", "candidates": len(all_candidates)},
        {"state": "no selected saved metric; not zero", "candidates": len(missing_metric_ids)},
    ])
    saved_retrieval = CASE.retrieval_rows()
    choice_families = {r["family"] for r in CASE.counterfactual_choices()}
record_room_stage("batch_8_saved_evidence", [
    room_check("metric_shards", expected_paintings, set(metric_manifest["shards"])),
    room_check("map_shards", expected_paintings, set(map_manifest["shards"])),
    room_check("registered_candidate_universe", (13879, 13879),
        (metric_manifest["candidate_count"], len(registered_candidates))),
    room_check("saved_metric_population", 11944, len(all_candidates)),
    room_check("exact_source_candidate_coverage", True, source_candidates == all_candidates),
    room_check("unavailable_metric_population_not_zero", 1935, len(missing_metric_ids)),
    room_check("all_exact_candidate_joins", 300, int(CASE_SHARD_AUDIT.exact_identity.sum())),
    room_check("unique_candidate_metric_rows", 300, int(CASE_SHARD_AUDIT.unique_keys.sum())),
    room_check("saved_row_counts", True, bool((CASE_SHARD_AUDIT.rows == CASE_SHARD_AUDIT.expected_rows).all())),
    room_check("opening_candidate", CASE.OPENING, CASE_OPENING["candidate"]["candidate_id"]),
    room_check("exact_sd_seed", 2026, CASE_SD["candidate"]["seed"]),
    room_check("deterministic_uncertainty_is_unavailable_not_zero", None, CASE_NA["images"]["evidence"]["uri"]),
    room_check("retrieval_and_comparison_counts", (100, 10, 14, 7),
        (len(saved_retrieval), len(CASE.saved_panels(PACKAGE, "example_retrieval_panels")),
         len(CASE.saved_panels(PACKAGE, "counterfactual_panels")), len(choice_families))),
    room_check("outbound_network_attempts", [], attempts),
])
print("All 300 compact metric/map shards rehashed; remote raster corpus downloads and full numeric source parity are not claimed.")
display(CASE_SHARD_AUDIT.describe(include="all"))
display(CASE_METRIC_AVAILABILITY)


Checking saved metric availability: outputs/13_classical_metrics/metrics/classical_metrics.csv
Checking saved metric availability: outputs/14_lpips_metrics/metrics/lpips_metrics.csv
Checking saved metric availability: outputs/17_local_consistency_metrics/metrics/local_consistency.csv


,check_id,passed,expected,observed
0,metric_shards,True,"{p002, p109, p142, p172, p092, p107, p269, p05...","{p002, p109, p172, p092, p107, p269, p056, p04..."
1,map_shards,True,"{p002, p109, p142, p172, p092, p107, p269, p05...","{p002, p109, p172, p092, p107, p269, p056, p04..."
2,registered_candidate_universe,True,"(13879, 13879)","(13879, 13879)"
3,saved_metric_population,True,11944,11944
4,exact_source_candidate_coverage,True,True,True
5,unavailable_metric_population_not_zero,True,1935,1935
6,all_exact_candidate_joins,True,300,300
7,unique_candidate_metric_rows,True,300,300
8,saved_row_counts,True,True,True
9,opening_candidate,True,candidate__lama__canonical__p018__mixed_damage...,candidate__lama__canonical__p018__mixed_damage...


All 300 compact metric/map shards rehashed; remote raster corpus downloads and full numeric source parity are not claimed.


,painting_id,rows,exact_identity,unique_keys,maps,expected_rows
count,300,300.000000,300,300,300.000000,300.000000
unique,300,NaN,1,1,NaN,NaN
top,p001,NaN,True,True,NaN,NaN
freq,1,NaN,300,300,NaN,NaN
mean,NaN,119.440000,NaN,NaN,386.613333,119.440000
std,NaN,123.773392,NaN,NaN,495.737035,123.773392
min,NaN,69.000000,NaN,NaN,186.000000,69.000000
25%,NaN,69.000000,NaN,NaN,186.000000,69.000000
50%,NaN,69.000000,NaN,NaN,206.000000,69.000000
75%,NaN,96.000000,NaN,NaN,237.000000,96.000000


,state,candidates
0,has a saved display metric,11944
1,no selected saved metric; not zero,1935


In [37]:
drop_validation_stages(('batch_8_server_smoke', 'batch_8_final_gate', 'batch_9_', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_8_saved_evidence'])

CASE_RENDER_CASES = [
    dict(name="lama_opening", query={"room": "case_explorer"}, scene="ce-stage",
         expect={"candidate.candidate_id": CASE.OPENING, "layer": "difference"}),
    dict(name="lama_uncertainty_na", query={"room": "case_explorer", "ce_layer": "uncertainty"}, scene="ce-stage",
         expect={"candidate.candidate_id": CASE.OPENING, "layer": "uncertainty", "images.evidence.uri": None}),
    dict(name="sd_exact_candidate", query={"room": "case_explorer", "ce_candidate": "sd15__p00__s2026__d0cd65cf894a"}, scene="ce-stage",
         expect={"candidate.candidate_id": "sd15__p00__s2026__d0cd65cf894a", "candidate.seed": 2026}),
    dict(name="colour_saved_layer", query={"room": "case_explorer", "ce_layer": "colour"}, scene="ce-stage",
         expect={"candidate.candidate_id": CASE.OPENING, "layer": "colour"}),
    dict(name="cross_painting_rejected", query={"room": "case_explorer", "ce_painting": "p001", "ce_candidate": CASE.OPENING},
         scene="ce-stage", error="candidate"),
]
record_room_stage("batch_8_server_smoke", render_cases(CASE_RENDER_CASES))


Server render: lama_opening


2026-10-01 10:18:54.445 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:18:56.616 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:18:56.625 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: lama_uncertainty_na


2026-10-01 10:18:56.800 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:18:58.714 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:18:58.723 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: sd_exact_candidate


...

Server render: colour_saved_layer


2026-10-01 10:19:01.018 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:19:02.951 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:19:02.972 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: cross_painting_rejected


2026-10-01 10:19:03.155 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:19:04.101 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


,check_id,passed,expected,observed
0,lama_opening,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
1,lama_uncertainty_na,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
2,sd_exact_candidate,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
3,colour_saved_layer,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
4,cross_painting_rejected,True,"{'network': [], 'exceptions': 0, 'visible_reje...","{'network': [], 'exceptions': 0, 'visible_reje..."


In [38]:
drop_validation_stages(('batch_8_final_gate', 'batch_9_', 'batch_10_'))

finish_room_batch(8, {'batch_8_source_contract', 'batch_8_server_smoke', 'batch_8_saved_evidence'})


,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_8_final_gate,2,2
batch_8_saved_evidence,14,14
batch_8_server_smoke,5,5
batch_8_source_contract,25,25


Batch 8: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.


## Batch 9 — Research Archive — immutable records and explicit limits


Validate the frozen Archive, all compact partitions and N33's exact recorded run/commit. Keep
300 painting, 30 case, one collection, five model and one final report populations separate.
The displayed 536/536 checks belong to saved N33, not to current N35 validation.

The Archive's 2,653 individual publication records and six bundles describe saved metadata;
they are not proof that all remote objects are reachable today. The opening must stay lazy.
Preserve source provenance even where terse wall copy omits working-tree status.

N29 records an artifact-manifest checksum discrepancy. This batch verifies that it is faithfully
reported, NOT that the discrepant artifact is valid. Carry it into Batch 10 for investigation;
do not rewrite historical manifests or waive release-integrity gates to obtain a pass.

Replace ALL historical Batch 9 persistence cells with this six-cell section. No canonical files
are written here. Stop after the final pending-gates cell; Batch 10 is still to be prepared.


In [39]:
drop_validation_stages(('batch_9_', 'batch_10_'))


start_room_batch(9)
record_room_stage("batch_9_source_contract", freeze_checks("research_archive")
    + room_binding_checks("research_archive", 16, 17) + [test_record("archive")]
    + js_checks(["tests/research_archive_controller.test.cjs"]))


Running archive local contract tests; no inference or artifact generation.
test_all_partitions_check_their_saved_hash (n35_checks_test_research_archive.ResearchArchiveTests.test_all_partitions_check_their_saved_hash) ... ok
test_claim_limit_figure_and_check_counts (n35_checks_test_research_archive.ResearchArchiveTests.test_claim_limit_figure_and_check_counts) ... ok
test_collections_and_large_files_are_not_materialized (n35_checks_test_research_archive.ResearchArchiveTests.test_collections_and_large_files_are_not_materialized) ... ok
test_corrupt_selected_file_fails_closed (n35_checks_test_research_archive.ResearchArchiveTests.test_corrupt_selected_file_fails_closed) ... ok
test_exact_figure_bytes_verified (n35_checks_test_research_archive.ResearchArchiveTests.test_exact_figure_bytes_verified) ... ok
test_exact_stage_scope_and_default_record (n35_checks_test_research_archive.ResearchArchiveTests.test_exact_stage_scope_and_default_record) ... ok
test_invalid_selections_never_fall_back (

,check_id,passed,expected,observed
0,research_archive__freeze_count,True,52,52
1,research_archive__docs/dashboard_design_finali...,True,b84d1cf3bd71323da4229db562441f4e90248ef36513c7...,b84d1cf3bd71323da4229db562441f4e90248ef36513c7...
2,research_archive__docs/dashboard_design_finali...,True,a3ccf7c67e37ffa241e545624c9a899106598aabff9ddb...,a3ccf7c67e37ffa241e545624c9a899106598aabff9ddb...
3,research_archive__src/restoration_eval/researc...,True,9055c3416154c278359d5fa9155d6dbf51b1729dcaf3f3...,9055c3416154c278359d5fa9155d6dbf51b1729dcaf3f3...
4,research_archive__src/restoration_eval/researc...,True,cdd5328fbcefd56f391eae162561ad1410b01047141f5c...,cdd5328fbcefd56f391eae162561ad1410b01047141f5c...
5,research_archive__streamlit_assets/evidence/re...,True,910a7f6c5ea99c335e8e3f174994c1802b9d6c8217869e...,910a7f6c5ea99c335e8e3f174994c1802b9d6c8217869e...
6,research_archive__streamlit_assets/evidence/re...,True,4d86da0492dcd9ecea57408d80458c996c5cdc39460164...,4d86da0492dcd9ecea57408d80458c996c5cdc39460164...
7,research_archive__streamlit_assets/evidence/re...,True,fd1ba85c09ef651947dfe1b6d9fe1cfd38687dacf76acb...,fd1ba85c09ef651947dfe1b6d9fe1cfd38687dacf76acb...
8,research_archive__streamlit_assets/evidence/re...,True,f7028ca7424c608f625815fe6017dd489db8a1989a9809...,f7028ca7424c608f625815fe6017dd489db8a1989a9809...
9,research_archive__streamlit_assets/evidence/re...,True,2846b5842288b88d0e5304d1f5a883b765d27d8369fd7d...,2846b5842288b88d0e5304d1f5a883b765d27d8369fd7d...


In [40]:
drop_validation_stages(('batch_9_saved_evidence', 'batch_9_server_smoke', 'batch_9_final_gate', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_9_source_contract'])

from restoration_eval import research_archive as ARCHIVE
with n35.offline() as attempts:
    ARCHIVE_OPENING = ARCHIVE.payload(PACKAGE)
    ARCHIVE_PUBLICATIONS = ARCHIVE.payload(PACKAGE, view="publications")["publications"]
    ARCHIVE_N29 = ARCHIVE.record(PACKAGE, "N29")["summary"]
    verified_n33 = ARCHIVE.verify_manifest(PACKAGE, "N33")
    source_manifest = json.loads((ARCHIVE.INDEX / "manifest.json").read_text(encoding="utf-8"))
    for name in source_manifest["partitions"]:
        ARCHIVE.partition(name)  # Every saved compact partition verifies its own hash.
    candidate_view = ARCHIVE.payload(PACKAGE, view="candidates", painting="p018")
record_room_stage("batch_9_saved_evidence", [
    room_check("stage_set", {f"N{i:02}" for i in range(1, 34)} | {"N12A", "D01", "D02"},
        {r["id"] for r in ARCHIVE_OPENING["stages"]}),
    room_check("opening_saved_run", "run_aef04267c2e44495a4e7a6249426bd4d", ARCHIVE_OPENING["record"]["run"]["run_id"]),
    room_check("report_populations_separate", {"painting": 300, "case": 30, "collection": 1, "model": 5, "final": 1},
        dict(Counter(r["report_type"] for r in ARCHIVE_OPENING["reports"]))),
    room_check("claim_limit_figure_counts", (49, 18, 24), tuple(len(ARCHIVE_OPENING[k]) for k in ("claims", "limitations", "figures"))),
    room_check("saved_n33_checks_not_current_n35_results", {"total": 536, "passed": 536}, ARCHIVE_OPENING["checks"]),
    room_check("lazy_publications", [], ARCHIVE_OPENING["publications"]),
    room_check("individual_publication_records", (2653, {"candidates": 2644, "diagnostics": 9}),
        (len(ARCHIVE_PUBLICATIONS), dict(Counter(r["storage_tier"] for r in ARCHIVE_PUBLICATIONS)))),
    room_check("immutable_publication_revisions", True,
        all(len(r["pinned_revision"]) == 40 and "/resolve/main/" not in r["pinned_url"] for r in ARCHIVE_PUBLICATIONS)),
    room_check("registered_bundle_count", 6, len(ARCHIVE_OPENING["bundles"])),
    room_check("selected_painting_partition", {"p018"}, {r["painting_id"] for r in candidate_view["candidate_rows"]}),
    room_check("n33_run_manifest_integrity", "verified", verified_n33["state"]),
    room_check("known_n29_discrepancy_preserved", "recorded_hash_mismatch", ARCHIVE_N29["artifact_manifest_state"]),
    room_check("outbound_network_attempts", [], attempts),
])
N35_PROVENANCE_ISSUES = pd.DataFrame([{
    "stage": "N29", "issue": ARCHIVE_N29["artifact_manifest_state"],
    "expected": ARCHIVE_N29["artifact_manifest_expected_sha256"],
    "observed": ARCHIVE_N29["artifact_manifest_observed_sha256"],
    "disposition": "Investigate in Batch 10; not repaired, waived or represented as verified here.",
}])
display(N35_PROVENANCE_ISSUES)
print("2,653 publication records and six bundles are saved metadata, NOT fresh remote availability checks.")


,check_id,passed,expected,observed
0,stage_set,True,"{N01, N07, N20, D02, N04, N24, N25, N27, N05, ...","{N01, N07, N20, D02, N04, N24, N25, N27, N05, ..."
1,opening_saved_run,True,run_aef04267c2e44495a4e7a6249426bd4d,run_aef04267c2e44495a4e7a6249426bd4d
2,report_populations_separate,True,"{'painting': 300, 'case': 30, 'collection': 1,...","{'case': 30, 'collection': 1, 'final': 1, 'mod..."
3,claim_limit_figure_counts,True,"(49, 18, 24)","(49, 18, 24)"
4,saved_n33_checks_not_current_n35_results,True,"{'total': 536, 'passed': 536}","{'total': 536, 'passed': 536}"
5,lazy_publications,True,[],[]
6,individual_publication_records,True,"(2653, {'candidates': 2644, 'diagnostics': 9})","(2653, {'candidates': 2644, 'diagnostics': 9})"
7,immutable_publication_revisions,True,True,True
8,registered_bundle_count,True,6,6
9,selected_painting_partition,True,{p018},{p018}


,stage,issue,expected,observed,disposition
0,N29,recorded_hash_mismatch,5df62a9675e9b6ad214475887378d29043312ca792f33a...,9ec4a9f68a49f724b4f56ed3cf259971013378c39961be...,"Investigate in Batch 10; not repaired, waived ..."


2,653 publication records and six bundles are saved metadata, NOT fresh remote availability checks.


In [41]:
drop_validation_stages(('batch_9_server_smoke', 'batch_9_final_gate', 'batch_10_'))

n35.required_stages(VALIDATION.checks, ['batch_9_saved_evidence'])

ARCHIVE_RENDER_CASES = [
    dict(name="n33_opening", query={"room": "research_archive"}, scene="ra-stage",
         expect={"record.summary.id": "N33", "record.run.run_id": "run_aef04267c2e44495a4e7a6249426bd4d", "publications": []}),
    dict(name="d02_saved_record", query={"room": "research_archive", "ar_record": "D02", "ar_view": "record"}, scene="ra-stage",
         expect={"record.summary.id": "D02", "open_view": "record"}),
    dict(name="p018_candidates", query={"room": "research_archive", "ar_view": "candidates", "ar_painting": "p018"}, scene="ra-stage",
         expect={"painting": "p018", "open_view": "candidates"}),
    dict(name="publication_metadata_only", query={"room": "research_archive", "ar_view": "publications_diagnostics"}, scene="ra-stage",
         expect={"open_view": "publications_diagnostics"}),
    dict(name="unknown_stage_rejected", query={"room": "research_archive", "ar_record": "N99"}, scene="ra-stage", error="Unknown stage"),
]
record_room_stage("batch_9_server_smoke", render_cases(ARCHIVE_RENDER_CASES))


Server render: n33_opening


2026-10-01 10:19:21.280 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:19:22.642 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:19:22.649 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: d02_saved_record


2026-10-01 10:19:22.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:19:23.988 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:19:23.996 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: p018_candidates


...

Server render: publication_metadata_only


2026-10-01 10:19:25.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:19:27.040 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
2026-10-01 10:19:27.052 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


Server render: unknown_stage_rejected


2026-10-01 10:19:27.362 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-01 10:19:28.380 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.


,check_id,passed,expected,observed
0,n33_opening,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
1,d02_saved_record,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
2,p018_candidates,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
3,publication_metadata_only,True,"{'network': [], 'errors': [], 'scene': True, '...","{'network': [], 'errors': [], 'scene': True, '..."
4,unknown_stage_rejected,True,"{'network': [], 'exceptions': 0, 'visible_reje...","{'network': [], 'exceptions': 0, 'visible_reje..."


In [42]:
drop_validation_stages(('batch_9_final_gate', 'batch_10_'))

finish_room_batch(9, {'batch_9_source_contract', 'batch_9_server_smoke', 'batch_9_saved_evidence'})

,check_id,passed,expected,observed
0,read_only_outputs_and_dashboard,True,"({}, {'src/restoration_eval/__init__.py': '5f2...","({}, {'src/restoration_eval/__init__.py': '5f2..."
1,canonical_files_absent,True,{},{}


,checks,passed
validation_stage,,
batch_9_final_gate,2,2
batch_9_saved_evidence,13,13
batch_9_server_smoke,5,5
batch_9_source_contract,60,60


Batch 9: local source/data/server checks passed. Canonical outputs: 0.
Previously approved visual layout retained. Fresh browser/Firefox/cloud validation NOT claimed.
N35 completion: NO. These cells do not certify external deployment readiness.


In [43]:
n35.required_stages(VALIDATION.checks, [f"batch_{i}_final_gate" for i in range(2, 10)])
VALIDATION.raise_for_blocking()
assert not n35.snapshot(OUTPUT_ROOT), "Canonical persistence belongs to Batch 10 only."
N35_PENDING_GATES = pd.DataFrame([
    {"gate": "Whole-app required-asset closure", "status": "Inventory required runtime dependencies across every room/filter/report; distinguish optional and historical outputs."},
    {"gate": "Pinned remote coverage and missing assets", "status": "Check Git/HF objects and bundle members; create a deduplicated missing/mismatched list. No uploads authorized by these cells."},
    {"gate": "Deployment-safe loaders", "status": "Resolve local-only originals, producer-scale scans, report fallback and dynamic numeric-parity gaps."},
    {"gate": "Fresh environment", "status": "Clean checkout without the full local outputs tree; Linux/Python/dependency parity and bounded remote cache."},
    {"gate": "Failure/cache/performance", "status": "Cold/warm loads, missing/corrupt files, timeouts, retries, memory/startup/latency/concurrency budgets."},
    {"gate": "Browser QA", "status": "All routes, within-room versus cross-room reloads, Firefox/second monitor, keyboard/focus and three viewports."},
    {"gate": "N29 provenance discrepancy", "status": "Investigate recorded artifact-manifest mismatch; preserve historical evidence, do not silently rewrite it."},
    {"gate": "Publication and hosted verification", "status": "Explicit release step: approved missing artifacts/bundles, Git push, Streamlit deployment, then live-site smoke tests."},
    {"gate": "Batch 10 canonical output gate", "status": "Persist checks, readiness report, run manifest and artifact index only after agreed blocking gates pass; label hosted status honestly."},
])
display(N35_PENDING_GATES)
print("Batches 1–9 local room validation complete in this kernel; all eight tabs plus D02 covered.")
print("N35 is NOT complete. Canonical N35 outputs: 0. Remote publication: NOT RUN. Live deployment: NOT VERIFIED.")
print("STOP HERE. Batch 10 must be prepared next. Do not restore or execute the old pilot persistence cells.")


,gate,status
0,Whole-app required-asset closure,Inventory required runtime dependencies across...
1,Pinned remote coverage and missing assets,Check Git/HF objects and bundle members; creat...
2,Deployment-safe loaders,"Resolve local-only originals, producer-scale s..."
3,Fresh environment,Clean checkout without the full local outputs ...
4,Failure/cache/performance,"Cold/warm loads, missing/corrupt files, timeou..."
5,Browser QA,"All routes, within-room versus cross-room relo..."
6,N29 provenance discrepancy,Investigate recorded artifact-manifest mismatc...
7,Publication and hosted verification,Explicit release step: approved missing artifa...
8,Batch 10 canonical output gate,"Persist checks, readiness report, run manifest..."


Batches 1–9 local room validation complete in this kernel; all eight tabs plus D02 covered.
N35 is NOT complete. Canonical N35 outputs: 0. Remote publication: NOT RUN. Live deployment: NOT VERIFIED.
STOP HERE. Batch 10 must be prepared next. Do not restore or execute the old pilot persistence cells.


## Batch 10 — local integration and deployment readiness

Continue after the successful Batch 9 gate. Replace the old pending-gates cell
48 and everything after it with this section. Do not run the entire notebook.
The optional replacement for cell 8 is only for a future clean-kernel rerun.

This batch validates the nonvisual deployment adapters, exact saved-row
projections, all candidate/report routes, missing-local-file fallbacks,
immutable public samples, cache/failure behavior and local performance.
It performs no inference, scientific recomputation, upload, commit or deployment.
Network cells explicitly fetch small selected public evidence samples.

Original visual baselines are not replaced. Ten precisely fingerprinted backend
deltas are checked separately; CSS, controllers, artwork and layout remain frozen.
N29's historical manifest discrepancy remains visible, with independent byte
verification of all six artifact groups, rather than rewriting its history.

Browser, Firefox and responsive observations cannot be manufactured
by AppTest. Enter real evidence in the manual-gates cell; empty entries stop the
release gate. Local checks do not certify the hosted site. Batch 11 records the
deployed revision, URL, clean Linux runtime and live results after the deployment commit.


In [46]:
import importlib
import json
import subprocess
import sys
from pathlib import Path
sys.path.insert(0, str(PROJECT_ROOT / "tools"))
import n35_room_validation as n35
import n35_batch10 as b10
import n35_deployment_checks as deploy_checks
n35 = importlib.reload(n35)
b10 = importlib.reload(b10)
n35.required_stages(VALIDATION.checks, [f"batch_{i}_final_gate" for i in range(2,10)])
drop_validation_stages(("batch_10_", "batch_11_"))
VALIDATION.raise_for_blocking()
B10_BEFORE = b10.fingerprint()
B10_RECEIPTS = {}

def b10_record(name, result):
    B10_RECEIPTS[name] = result
    record_room_stage("batch_10_" + name, [room_check(name, True, result["passed"],
        description=json.dumps(result, default=str))])

def b10_command(arguments):
    process = subprocess.run([sys.executable, *arguments], cwd=PROJECT_ROOT,
        capture_output=True, text=True, timeout=300)
    print(process.stdout)
    print(process.stderr)
    return {"passed": process.returncode==0, "returncode":process.returncode,
            "stdout":process.stdout, "stderr":process.stderr}

print("Batch 10 initialized. No canonical output written and no deployment performed.")


Batch 10 initialized. No canonical output written and no deployment performed.


In [47]:
# Current source/data regression, not a reuse of pre-backend-change passes.
b10_record("visual_delta", b10.visual_delta_checks())
b10_record("companion", deploy_checks.companion_check())
b10_record("closure", b10.closure_check())
b10_record("transport", b10_command(["-m", "unittest", "discover", "-s", "tests", "-p", "test_evidence_transport.py", "-v"]))
b10_record("persistence_tests", b10_command(["-m", "unittest", "discover", "-s", "tests", "-p", "test_n35_batch10.py", "-v"]))
for group in ("study", "metric", "gallery", "stability", "trust_portrait", "case", "archive"):
    print("Room regression:", group, flush=True)
    result = b10.run_check("suite", group)
    print(result.get("transcript", result))
    b10_record("tests_"+group, result)
b10_record("n29", b10.n29_check())
print("N29: payloads verified; historical manifest checksum discrepancy retained and disclosed.")


,check_id,passed,expected,observed
0,visual_delta,True,True,True


,check_id,passed,expected,observed
0,companion,True,True,True


,check_id,passed,expected,observed
0,closure,True,True,True



test_bundle_and_missing_member (test_evidence_transport.TransportTests.test_bundle_and_missing_member) ... ok
test_changed_local_bytes_do_not_fall_back (test_evidence_transport.TransportTests.test_changed_local_bytes_do_not_fall_back) ... ok
test_cold_warm_and_corrupt_cache (test_evidence_transport.TransportTests.test_cold_warm_and_corrupt_cache) ... ok
test_concurrent_same_object_is_downloaded_once (test_evidence_transport.TransportTests.test_concurrent_same_object_is_downloaded_once) ... ok
test_encoded_image_cache_is_byte_bounded (test_evidence_transport.TransportTests.test_encoded_image_cache_is_byte_bounded) ... ok
test_eviction (test_evidence_transport.TransportTests.test_eviction) ... ok
test_git_newline_reconstruction_requires_exact_hash (test_evidence_transport.TransportTests.test_git_newline_reconstruction_requires_exact_hash) ... ok
test_http_404_does_not_retry (test_evidence_transport.TransportTests.test_http_404_does_not_retry) ... ok
test_invalid_zip (test_evidence_trans

,check_id,passed,expected,observed
0,transport,True,True,True


...

,check_id,passed,expected,observed
0,tests_case,True,True,True


Room regression: archive
Running suite archive
test_all_partitions_check_their_saved_hash (n35_checks_test_research_archive.ResearchArchiveTests.test_all_partitions_check_their_saved_hash) ... ok
test_claim_limit_figure_and_check_counts (n35_checks_test_research_archive.ResearchArchiveTests.test_claim_limit_figure_and_check_counts) ... ok
test_collections_and_large_files_are_not_materialized (n35_checks_test_research_archive.ResearchArchiveTests.test_collections_and_large_files_are_not_materialized) ... ok
test_corrupt_selected_file_fails_closed (n35_checks_test_research_archive.ResearchArchiveTests.test_corrupt_selected_file_fails_closed) ... ok
test_exact_figure_bytes_verified (n35_checks_test_research_archive.ResearchArchiveTests.test_exact_figure_bytes_verified) ... ok
test_exact_stage_scope_and_default_record (n35_checks_test_research_archive.ResearchArchiveTests.test_exact_stage_scope_and_default_record) ... ok
test_invalid_selections_never_fall_back (n35_checks_test_research_arc

,check_id,passed,expected,observed
0,tests_archive,True,True,True


,check_id,passed,expected,observed
0,n29,True,True,True


N29: payloads verified; historical manifest checksum discrepancy retained and disclosed.


In [48]:
# Explicit public reads. This does not download every image or upload anything.
b10_record("remote", b10.run_check("remote"))
b10_record("github", b10.run_check("github"))
print("Exact-byte cold/warm HF samples and immutable Git/Git LFS samples checked.")


Running remote 


,check_id,passed,expected,observed
0,remote,True,True,True


Running github 


,check_id,passed,expected,observed
0,github,True,True,True


Exact-byte cold/warm HF samples and immutable Git/Git LFS samples checked.


In [49]:
# Each render runs in a new process. Untracked scientific files are hidden;
# the new compact deployment companion is explicitly included for staging.
# These are server-side renders, not browser interaction or Linux certification.
for room_id in deploy_checks.ROOMS:
    b10_record("opening_"+room_id, b10.run_check("opening", room_id,
        missing_local=True, remote=True))


Running opening exhibition_foyer


,check_id,passed,expected,observed
0,opening_exhibition_foyer,True,True,True


Running opening study_design


,check_id,passed,expected,observed
0,opening_study_design,True,True,True


Running opening metric_framework


...

,check_id,passed,expected,observed
0,opening_focused_portrait_review,True,True,True


Running opening case_explorer


,check_id,passed,expected,observed
0,opening_case_explorer,True,True,True


Running opening research_archive


,check_id,passed,expected,observed
0,opening_research_archive,True,True,True


In [50]:
# Existing psutil is needed for local sampled RSS; no automatic installation.
# Five warm rerenders per room. A failure is not permission to raise the budget.
for room_id in deploy_checks.ROOMS:
    b10_record("performance_"+room_id, b10.run_check("performance", room_id))
print("Local Python timing/memory checked. Browser navigation p95 remains separate.")


Running performance exhibition_foyer


,check_id,passed,expected,observed
0,performance_exhibition_foyer,True,True,True


Running performance study_design


,check_id,passed,expected,observed
0,performance_study_design,True,True,True


Running performance metric_framework


...

Running performance case_explorer


,check_id,passed,expected,observed
0,performance_case_explorer,True,True,True


Running performance research_archive


,check_id,passed,expected,observed
0,performance_research_archive,True,True,True


Local Python timing/memory checked. Browser navigation p95 remains separate.


## Batch 10 — partial local checkpoint, not release certification

Local user approval: “everything works perfectly, no need to change anything.”
The dashboard remains frozen. Automated receipts are retained, and the revised
closeout helper receives a short targeted test run. No full notebook rerun is needed.

This closes Batch 10 as an evidence checkpoint, not as an all-gates-passed release.
General approval does not establish exact viewport, keyboard, navigation,
download, timing or resource measurements. Existing detailed observations are
preserved; missing ones are saved as blocking, false, pending-not-verified rows.
Any explicitly reported failure still stops checkpoint creation.

Batch 11 must resolve the pending checklist, including all nine routes at
1672×941, 1024×1366 and 390×844, accessibility, Firefox/second-monitor behavior,
document reload behavior, tour, downloads, browser warm p95 ≤1.5 s, peak process
memory ≤512 MiB and foreground remote totals ≤34 MiB, two requests/one bundle,
60 seconds. No configured limits are relaxed. Clean Linux and the actual hosted
revision/URL must also be verified. N35 remains partial and N36 is not cleared.


In [53]:
import importlib
b10 = importlib.reload(b10)
# Exact old/new hashes apply ONLY to the three closeout-helper/test files.
# Application, assets, requirements and configuration cannot be reconciled here.
B10_CHECKPOINT_BEFORE, B10_CLOSEOUT_DELTA = b10.checkpoint_fingerprint(
    B10_BEFORE, {'tools/n35_batch10.py': ['9e38fc65f3a165ee95f28cd14703180ce04e4bcca40bfc027342c926a9054a1c', 'b161b504970c3e5013bb2d29309eaf1bfee2149b73de19e289f7efd49019bf90'], 'tools/prepare_n35_batch10_cells.py': ['46fd5961778c910daa251a56f39116233ba3cb74a6e49d238276da5ffb9e6706', 'c89cb430459a676a2115b273e3f483015d8fd65feb42155cdb8eccee838ebbe0'], 'tests/test_n35_batch10.py': ['5d91214154e566b484207bab2c3fcfc9aca614033d0b108d8788e6d38bfadafc', '17227a72ab8d7d745f81f75ce7cf389019324c67a3c08315393fbfc2a8627680']})
result = b10_command(["-m", "unittest", "discover", "-s", "tests", "-p", "test_n35_batch10.py", "-v"])
result["scope"] = "Current closeout helper tests only; earlier runtime receipts retained"
result["original_validation_fingerprint"] = B10_BEFORE
result["closeout_helper_delta"] = B10_CLOSEOUT_DELTA
b10_record("checkpoint_helper_tests", result)
B10_LOCAL_APPROVAL = "User reported: 'everything works perfectly, no need to change anything'. Qualitative local approval only; no per-check measurement or observation time inferred."
if "B10_BROWSER_OBSERVATIONS" not in globals():
    B10_BROWSER_OBSERVATIONS = {key: {"passed": None, "evidence": "", "observed_at_utc": ""}
                                for key in b10.MANUAL_GATES}
display(pd.DataFrame(b10.manual_checks(B10_BROWSER_OBSERVATIONS)))
print("Missing detailed observations remain pending, not passed. Existing observations preserved.")


Running suite trust_portrait
Notebook SOURCE changed during validation. Save all cells, then rerun without editing.
Running suite trust_portrait

test_automated_blocker_cannot_be_deferred (test_n35_batch10.Batch10Tests.test_automated_blocker_cannot_be_deferred) ... ok
test_checkpoint_cells_parse_and_do_not_invent_passes (test_n35_batch10.Batch10Tests.test_checkpoint_cells_parse_and_do_not_invent_passes) ... ok
test_checkpoint_keeps_unknown_gates_blocking_without_mutating_collector (test_n35_batch10.Batch10Tests.test_checkpoint_keeps_unknown_gates_blocking_without_mutating_collector) ... ok
test_complete_cells_parse (test_n35_batch10.Batch10Tests.test_complete_cells_parse) ... ok
test_default_readiness_still_requires_manual_evidence (test_n35_batch10.Batch10Tests.test_default_readiness_still_requires_manual_evidence) ... ok
test_detailed_manual_pass_preserved (test_n35_batch10.Batch10Tests.test_detailed_manual_pass_preserved) ... ok
test_exact_helper_delta_reconciliation_only (test_n35_

,check_id,passed,expected,observed
0,checkpoint_helper_tests,True,True,True


,check_id,passed,evidence,observed_at_utc
0,desktop_1672x941_all_rooms,False,,
1,tablet_1024x1366_all_rooms,False,,
2,mobile_390x844_all_rooms,False,,
3,keyboard_focus_and_dialogs,False,,
4,firefox_reload_and_second_monitor,False,,
5,same_room_no_document_reload,False,,
6,cross_room_links_and_guided_tour,False,,
7,downloaded_reports_open_self_contained,False,,
8,browser_warm_navigation_p95,False,,
9,remote_action_budget_and_peak_memory,False,,


Missing detailed observations remain pending, not passed. Existing observations preserved.


In [54]:
required = {"visual_delta", "companion", "closure", "transport", "persistence_tests", "n29", "remote", "github", "checkpoint_helper_tests"}
required |= {"tests_"+g for g in ("study", "metric", "gallery", "stability", "trust_portrait", "case", "archive")}
required |= {prefix+r for prefix in ("opening_", "performance_") for r in deploy_checks.ROOMS}
missing = sorted(required-set(B10_RECEIPTS))
if missing: raise RuntimeError("Run missing Batch 10 stages first: " + repr(missing))
if not all(r.get("passed") for r in B10_RECEIPTS.values()):
    raise RuntimeError("A recorded automated failure remains; no checkpoint allowed.")
if b10.fingerprint() != B10_CHECKPOINT_BEFORE:
    raise RuntimeError("Source changed after closeout helper checks; stop and inspect.")
_, B10_PENDING_QUALIFICATIONS = b10.qualification_record(
    VALIDATION, B10_BROWSER_OBSERVATIONS, B10_LOCAL_APPROVAL)
b10_record("checkpoint_gate", {"passed":True,
    "scope":"Automated receipt completeness and partial checkpoint eligibility ONLY",
    "user_approval":B10_LOCAL_APPROVAL,
    "pending_qualifications":B10_PENDING_QUALIFICATIONS,
    "all_qualifications_passed":not B10_PENDING_QUALIFICATIONS,
    "deployment_state":"not_yet_validated", "deployment_url":"", "next_batch":11})
print("Partial checkpoint eligible. Pending qualification IDs:", B10_PENDING_QUALIFICATIONS)


,check_id,passed,expected,observed
0,checkpoint_gate,True,True,True


Partial checkpoint eligible. Pending qualification IDs: ['desktop_1672x941_all_rooms', 'tablet_1024x1366_all_rooms', 'mobile_390x844_all_rooms', 'keyboard_focus_and_dialogs', 'firefox_reload_and_second_monitor', 'same_room_no_document_reload', 'cross_room_links_and_guided_tour', 'downloaded_reports_open_self_contained', 'browser_warm_navigation_p95', 'remote_action_budget_and_peak_memory']


In [55]:
n35.required_stages(VALIDATION.checks, ["batch_10_checkpoint_gate"])
B10_PERSISTED = b10.persist(VALIDATION, B10_RECEIPTS, B10_CHECKPOINT_BEFORE,
    B10_BROWSER_OBSERVATIONS, checkpoint_approval=B10_LOCAL_APPROVAL)
display(B10_PERSISTED)
print("Four canonical N35 checkpoint files saved; any prior run is preserved in the reported backup.")
print("N35 remains PARTIAL. Pending checks were saved as blocking/not passed, not waived.")
print("Next: scoped GitHub/HF publication review → commit → test deployment and pending gates in Batch 11 → N36.")


{'run_id': 'run_c284ad17ddde4b89928cfad7c153d393',
 'canonical_files': ['validation/dashboard_checks.csv',
  'reports/deployment_readiness.md',
  'manifests/run_manifest.json',
  'manifests/artifacts.csv'],
 'prior_run_backup': 'D:\\Masters\\FH\\Thesis\\painting-restoration-eval\\.codex_tmp\\n35_persistence\\previous-n35-20b08ca94b704a9fa7a0d393427d5c71',
 'state': 'local_checkpoint_pending_qualification',
 'pending_qualifications': ['desktop_1672x941_all_rooms',
  'tablet_1024x1366_all_rooms',
  'mobile_390x844_all_rooms',
  'keyboard_focus_and_dialogs',
  'firefox_reload_and_second_monitor',
  'same_room_no_document_reload',
  'cross_room_links_and_guided_tour',
  'downloaded_reports_open_self_contained',
  'browser_warm_navigation_p95',
  'remote_action_budget_and_peak_memory'],
 'n35_complete': False}

Four canonical N35 checkpoint files saved; any prior run is preserved in the reported backup.
N35 remains PARTIAL. Pending checks were saved as blocking/not passed, not waived.
Next: scoped GitHub/HF publication review → commit → test deployment and pending gates in Batch 11 → N36.


## Batch 10 checkpoint handoff

Share the persistence result before preparing the commit. Do not rerun Batches
1–9, restart the kernel, or mark empty observations True just to pass a gate.
Only cells 55–59 are replaced here; keep 55 and 59 as Markdown.

N35 is partial, not certified ready. The saved CSV and manifest explicitly
retain pending qualifications; the public URL and tested revision remain blank.
Batch 11 must close those items and validate the hosted site before N36.
Existing HF bundles are reused unless the publication review finds a real gap.
No dashboard code, visual assets, layout, or scientific output is changed here.


## Batch 11 — live deployment acceptance and N35 closeout

Batch 10 is complete as a saved local checkpoint. The owner tested the live
full-version app, accepted its functionality, requested further interactive
testing stop, and confirmed successful scheduled availability maintenance.
GitHub independently records successful push and workflow-dispatch runs on main.

This batch closes N35 **with disclosed limitations**, not by claiming every
original qualification was measured. Its explicit acceptance exception changes
only the known unmeasured manual qualification rows from blocking to warning;
their `passed` values remain false and their original records remain in details.
Actual failures and automated regressions cannot be waived by this mechanism.
The four local dependency-version warnings also remain. The original numerical
limits in the validation configuration are unchanged.

Live URL: https://fhtw-painting-restoration-main.streamlit.app/

Repository/checker reference: e2b54a695a99ace715adbaee8a7ff21f7a407780.
This reference is not independent attestation of the server's checkout SHA.
Actions verified the Foyer, visitor dialog and Case Explorer evidence in Linux
Chromium; this is not a full clean-Linux reproduction of the application.
The old pilot URL has not been renamed or redirected.

In [56]:
import importlib
import json
import subprocess
import sys
from pathlib import Path
import pandas as pd

B11_ROOT = Path.cwd().resolve()
if B11_ROOT.name == "notebooks":
    B11_ROOT = B11_ROOT.parent
if not (B11_ROOT / "tools/n35_batch11.py").is_file():
    raise RuntimeError("Run from the project root or its notebooks directory.")
sys.path.insert(0, str(B11_ROOT / "tools"))
import n35_batch11 as b11
b11 = importlib.reload(b11)
B11_PRIOR, B11_FRAME, B11_EVIDENCE = b11.load_checkpoint()
B11_BEFORE = b11.source_fingerprint()
B11_PRIOR_SHA = b11.sha256_file(B11_ROOT / b11.b10.OUTPUT / b11.b10.FILES[2])
print("Verified checkpoint:", B11_PRIOR["run_id"], B11_PRIOR["run_status"])
print("Accepted live URL:", B11_EVIDENCE["url"])
display(pd.DataFrame(B11_EVIDENCE["verified_actions_runs"])[
    ["id", "event", "branch", "conclusion", "checker_status", "url"]])


Verified checkpoint: run_c284ad17ddde4b89928cfad7c153d393 partial
Accepted live URL: https://fhtw-painting-restoration-main.streamlit.app/


,id,event,branch,conclusion,checker_status,url
0,36858527530,push,main,success,ready,https://github.com/Rahul-DS25M008/painting_res...
1,36859028102,workflow_dispatch,main,success,ready,https://github.com/Rahul-DS25M008/painting_res...


In [57]:
# Explicit owner-directed acceptance; this is NOT a claim that missing tests passed.
B11_ACCEPT_DISCLOSED_LIMITATIONS = True
B11_REVIEW = b11.accepted_checks(B11_FRAME,
    accept_disclosed_limitations=B11_ACCEPT_DISCLOSED_LIMITATIONS)
print(B11_EVIDENCE["user_acceptance"]["disposition"])
for limitation in B11_EVIDENCE["remaining_limitations"]:
    print("-", limitation)
display(B11_REVIEW.to_dataframe().query("passed == False")[[
    "validation_stage", "check_id", "severity", "observed", "passed"]])
print("False warning rows are intentionally retained; automated blocking failures still stop closeout.")


Owner accepts delivery with the unmeasured qualifications disclosed; not a claim of measured compliance.
- Exact desktop/tablet/mobile viewport and accessibility coverage was not completed independently; owner reports functional acceptance.
- Firefox/second-monitor and complete same-room/cross-room behavior are owner-accepted, not exhaustively independently verified.
- All downloaded reports were not independently opened offline in the live acceptance check.
- Browser warm-navigation p95, live server peak memory and aggregate per-action remote budgets were not measured.
- GitHub used Linux Chromium to visit the hosted app; that does not establish a clean Linux application environment reproduction or exact installed server dependency versions.
- The four Batch 1 local dependency-version warnings remain historical facts.
- Server checkout SHA was not independently read; repository reference and checker revision are recorded separately.
- Availability maintenance is best-effort, not an in

,validation_stage,check_id,severity,observed,passed
61,batch_1_dependency_versions,version__pillow,warning,9.5.0,False
62,batch_1_dependency_versions,version__plotly,warning,6.8.0,False
63,batch_1_dependency_versions,version__pyarrow,warning,24.0.0,False
64,batch_1_dependency_versions,version__streamlit,warning,1.59.0,False
520,batch_11_qualification,desktop_1672x941_all_rooms,warning,owner_accepted_without_measurement,False
521,batch_11_qualification,tablet_1024x1366_all_rooms,warning,owner_accepted_without_measurement,False
522,batch_11_qualification,mobile_390x844_all_rooms,warning,owner_accepted_without_measurement,False
523,batch_11_qualification,keyboard_focus_and_dialogs,warning,owner_accepted_without_measurement,False
524,batch_11_qualification,firefox_reload_and_second_monitor,warning,owner_accepted_without_measurement,False
525,batch_11_qualification,same_room_no_document_reload,warning,owner_accepted_without_measurement,False


False warning rows are intentionally retained; automated blocking failures still stop closeout.


In [58]:
# Small offline closeout tests only: temporary files, no live dashboard sweep.
B11_TESTS = subprocess.run([sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_n35_batch11.py", "-v"],
    cwd=B11_ROOT, capture_output=True, text=True, timeout=120)
print(B11_TESTS.stdout)
print(B11_TESTS.stderr)
if B11_TESTS.returncode != 0:
    raise RuntimeError("Closeout-helper tests failed; no canonical files written.")
if b11.source_fingerprint() != B11_BEFORE:
    raise RuntimeError("Source changed since setup; inspect before continuing.")
B11_REVIEW.raise_for_blocking()
print("Closeout helper passed. Ready for owner-executed persistence with disclosed limitations.")



test_acceptance_preserves_false_and_original_details (test_n35_batch11.CloseoutTests.test_acceptance_preserves_false_and_original_details) ... ok
test_actual_manual_failure_not_waived (test_n35_batch11.CloseoutTests.test_actual_manual_failure_not_waived) ... ok
test_automated_failure_not_waived (test_n35_batch11.CloseoutTests.test_automated_failure_not_waived) ... ok
test_cells_parse_and_do_not_run_live_checks (test_n35_batch11.CloseoutTests.test_cells_parse_and_do_not_run_live_checks) ... ok
test_live_evidence_requires_both_successful_runs (test_n35_batch11.CloseoutTests.test_live_evidence_requires_both_successful_runs) ... ok
test_missing_qualification_fails (test_n35_batch11.CloseoutTests.test_missing_qualification_fails) ... ok
test_no_implicit_acceptance (test_n35_batch11.CloseoutTests.test_no_implicit_acceptance) ... ok
test_persistence_and_rerun_preserve_prior_records (test_n35_batch11.CloseoutTests.test_persistence_and_rerun_preserve_prior_records) ... ok

--------------------

In [59]:
if B11_TESTS.returncode != 0:
    raise RuntimeError("Run the closeout tests successfully first.")
B11_PERSISTED = b11.persist(before=B11_BEFORE, previous_manifest_sha=B11_PRIOR_SHA,
    accept_disclosed_limitations=B11_ACCEPT_DISCLOSED_LIMITATIONS)
display(B11_PERSISTED)
print("N35 completed with disclosed limitations. Four canonical outputs saved; previous run backed up.")
print("Save this notebook, refresh the inventory, then prepare the scoped closeout commit.")
print("N36 may consolidate the accepted record, retaining all warnings and acceptance exceptions.")


{'run_id': 'run_3d52ae7863c44fb1b7f56a99ea0c01fa',
 'status': 'completed',
 'closeout': 'completed_with_disclosed_limitations',
 'url': 'https://fhtw-painting-restoration-main.streamlit.app/',
 'summary': {'check_count': 533,
  'passed_count': 519,
  'failed_count': 14,
  'blocking_failure_count': 0,
  'overall_passed': True,
  'failed_check_ids': ['batch_1_dependency_versions:version__pillow',
   'batch_1_dependency_versions:version__plotly',
   'batch_1_dependency_versions:version__pyarrow',
   'batch_1_dependency_versions:version__streamlit',
   'batch_11_qualification:desktop_1672x941_all_rooms',
   'batch_11_qualification:tablet_1024x1366_all_rooms',
   'batch_11_qualification:mobile_390x844_all_rooms',
   'batch_11_qualification:keyboard_focus_and_dialogs',
   'batch_11_qualification:firefox_reload_and_second_monitor',
   'batch_11_qualification:same_room_no_document_reload',
   'batch_11_qualification:cross_room_links_and_guided_tour',
   'batch_11_qualification:downloaded_repor

N35 completed with disclosed limitations. Four canonical outputs saved; previous run backed up.
Save this notebook, refresh the inventory, then prepare the scoped closeout commit.
N36 may consolidate the accepted record, retaining all warnings and acceptance exceptions.


## N35 handoff — completed with disclosed limitations

The closeout manifest records `run_status: completed`, the accepted live URL,
successful availability runs and the owner's acceptance. Artifacts have
`validation_status: warning`; `all_planned_checks_verified` remains false.
Completion means accepted delivery, not perfect measured compliance. N36
must preserve that distinction, the prior checkpoint and the N29 historical
discrepancy; no scientific evidence is recomputed or retrospectively rewritten.

Save the notebook after cell 64. From the project root, use the existing
project environment to run `python -c "import yaml; print(yaml.__version__)"`
then `python tools/build_project_inventory.py --root . --out-dir outputs/inventory`.
Require both `summary.read_error_file_count` and `summary.read_error_count`
to be zero. Use the incremental default; do not request a full no-reuse scan.
Share cell 64's result and the inventory summary before the scoped commit.

No new HF bundle is created by closeout. Next: commit the N35 record, then
prepare N36's Controlled-300 consolidation. N36 has not been executed or
certified here. Minor cleanup and README work can follow that consolidation.
